# Admission Subphenotypes of Pediatric Acute Encephalopathy: Registered Analysis

This notebook runs the analysis registered on OSF (Preregistration Template for Secondary Data Analysis; analysis plan
"Encephalopathy Subphenotypes Analysis Plan") on the frozen dataset **AE_freeze_v1.4** and nothing else.

**Question.** Do routine blood tests and vital signs at admission separate children with acute encephalopathy into
latent subphenotypes (H1)? Do the subphenotypes differ in IL-6, IL-10 and in death or critical discharge against medical
advice (H2), reappear in two periods (H3), and add prognostic information beyond age and consciousness (H4)?

**How to run.** Put this notebook in one folder with the folder `AE_freeze_v1.4/` (the one that contains `manifest.json`) and the
completed chart-review sheet `chart_review_completed.xlsx` (without it, step 0 is skipped), then run all cells: Run All in
Jupyter, or `jupyter nbconvert --to notebook --execute AE_subphenotypes_analysis.ipynb --output AE_subphenotypes_run.ipynb`.
Requirements: Python 3.13 with the package versions in `requirements.txt`, those of the reported run (StepMix 3.0.0 is registered
and checked below). Resampling steps run in parallel processes (N_WORKERS below, at most 8): the reported run took 3.3 hours on
2 cores; expect about 1 hour with 8 cores. Progress is written to `output_subphenotypes/logs/progress.log` during the run. Keep
output files closed in Excel and PDF viewers until the run ends (on Windows, an open file cannot be replaced).

**Outputs** (`./output_subphenotypes/`): `tables/` (CSV and one Excel workbook), `figures/` (PDF and PNG, with
`figure_legends.md`), `logs/run_log.md` (settings, software, every check, the operational details that the registration leaves
open, and deviations), and `code/ae_lpa_tools.py` (the computational module written by this notebook). No research numbers or
dates leave the notebook: every output is aggregated, and an identifier check runs at the end. Two local folders hold child-level
data and stay on this computer: `cache_frozen_tables/` (typed copies of the frozen tables) and `cache_figure_state/` (the results
that `AE_subphenotypes_redraw_figures.ipynb` uses to redraw the figures in about a minute, without rerunning the analysis).

**Steps** (as registered): 0 measurement validation (chart review); 1 analysis dataset; 2 fit 1-6 classes; 3 choose the number
of classes; 4 describe and label (H1); 5 stability; 6 reproduction by period (H3); 7 validation with BCH three-step estimates
(H2 and secondary validators); 8 beyond severity (H4); 9 simple classifier (exploratory); 10 sensitivity analyses. Two exploratory
analyses added after the reported run of 2026-10-14 follow step 10 and are not in the registration: E1, prognostic information
within the ITES classes; E2, immunomodulatory treatment and its timing by subphenotype.

In [ ]:
# Setup: settings, paths, software check.
import os, re, sys, ast, json, time, math, platform, warnings, itertools, hashlib, textwrap
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

SEED = 20261013                 # one seed for every random step (derived seeds are documented where used)
N_STARTS = 100                  # random starts per class number (registered)
K_MAX = 6                       # 1 to 6 classes (registered)
MIN_CLASS_SHARE = 0.05          # smallest class must hold at least 5% of children (registered)
N_STARTS_FALLBACK = 500         # starts when the best log-likelihood is not reached by at least two starts (registered, Q23)
N_BOOT_STABILITY = 200          # bootstrap refits for stability (registered)
N_BOOT_H2 = 500                 # bootstrap resamples for the BCH confidence intervals (registered)
N_BOOT_H4 = 200                 # bootstrap resamples for the change in C-statistic (registered)
N_BOOT_SENS = 500               # bootstrap resamples for H2 in each sensitivity analysis (same as the main analysis)
N_RANDOM_BOOT = 19              # random starts added to the warm start in each H2 bootstrap refit (operational detail)
N_RANDOM_BOOT_SENS = 4          # the same for the H2 bootstraps of the sensitivity analyses (operational detail)
N_IMPUTATIONS = 20              # multiple imputation for the consensus k-means sensitivity analysis (registered)
N_WORKERS = max(1, min(8, os.cpu_count() or 1))
QUICK = os.environ.get('AE_QUICK') == '1'        # development switch only; the reported run uses QUICK = False
if QUICK:
    N_BOOT_STABILITY, N_BOOT_H2, N_BOOT_H4, N_BOOT_SENS, N_IMPUTATIONS, N_STARTS, N_STARTS_FALLBACK = 4, 20, 20, 10, 2, 10, 10

HERE = Path.cwd()
FREEZE = HERE / 'AE_freeze_v1.4'
if not (FREEZE / 'manifest.json').exists() and (FREEZE / 'AE_freeze_v1.4' / 'manifest.json').exists():
    FREEZE = FREEZE / 'AE_freeze_v1.4'                    # the zip was extracted into a folder of the same name
if not (FREEZE / 'manifest.json').exists():
    raise FileNotFoundError(f'The folder AE_freeze_v1.4 (the one that contains manifest.json) must be next to this notebook, in {HERE}.')
OUT = HERE / 'output_subphenotypes'
TAB, FIG, LOGD, CODE = OUT / 'tables', OUT / 'figures', OUT / 'logs', OUT / 'code'
for p in (OUT, TAB, FIG, LOGD, CODE):
    p.mkdir(parents=True, exist_ok=True)


def locked_files(paths):
    """Files that cannot be opened for writing, such as files open in Excel or a PDF viewer on Windows."""
    out = []
    for f in map(Path, paths):
        if f.is_file():
            try:
                with open(f, 'ab'):
                    pass
            except PermissionError:
                out.append(f.relative_to(HERE).as_posix() if f.is_relative_to(HERE) else str(f))
    return out


_locked = locked_files(OUT.rglob('*'))
if _locked:
    raise RuntimeError('These output files are open in another program; close them and run again: ' + ', '.join(_locked[:10]))
T0 = time.time()
RUNLOG = []          # (section, item, value)
DETAILS = []         # operational details the registration leaves open, fixed here before any model is fitted
DEVIATIONS = []      # departures from the registration, with reasons
CHANGES = []         # settings changed after the development run (reduced-settings test run) and the independent check, with reasons
ADDITIONS = []       # exploratory analyses added after the reported run of 2026-10-14; not in the registration


PROGRESS = LOGD / 'progress.log'          # written as the notebook runs, so a long run can be followed from outside
PROGRESS.write_text('', encoding='utf-8')


def log(section, item, value=''):
    RUNLOG.append((section, item, value))
    line = f'[{section}] {item}: {value}' if value != '' else f'[{section}] {item}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as fh:
        fh.write(f"{time.strftime('%H:%M:%S')} {line}\n")


def detail(text):
    DETAILS.append(text)


def change(text):
    CHANGES.append(text)


def addition(text):
    ADDITIONS.append(text)


def deviation(text):
    DEVIATIONS.append(text)
    print('DEVIATION:', text, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as fh:
        fh.write(f"{time.strftime('%H:%M:%S')} DEVIATION: {text}\n")


def wait_until_writable(paths, minutes=30):
    """Before outputs are replaced: wait until files open in another program are closed (checked every 5 s, for up to `minutes`)."""
    paths, t_start, t_said = list(paths), time.time(), 0.0
    while True:
        locked = locked_files(paths)
        if not locked:
            return
        if time.time() - t_start > 60 * minutes:
            raise RuntimeError('These output files are still open in another program: ' + ', '.join(locked[:10]))
        if time.time() - t_said > 60:
            msg = 'Waiting: close these files so that they can be replaced: ' + ', '.join(locked[:10])
            print(msg, flush=True)
            with open(PROGRESS, 'a', encoding='utf-8') as fh:
                fh.write(f"{time.strftime('%H:%M:%S')} {msg}\n")
            t_said = time.time()
        time.sleep(5)


import scipy, sklearn, statsmodels, matplotlib
import stepmix
if stepmix.__version__ != '3.0.0':
    raise RuntimeError(f'StepMix 3.0.0 is registered; found {stepmix.__version__}.')
from stepmix.stepmix import StepMix
SOFTWARE = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'scipy': scipy.__version__,
            'scikit-learn': sklearn.__version__, 'statsmodels': statsmodels.__version__, 'matplotlib': matplotlib.__version__,
            'stepmix': stepmix.__version__}
log('0 Setup', 'Software', SOFTWARE)
log('0 Setup', 'Settings', {'seed': SEED, 'starts': N_STARTS, 'k_max': K_MAX, 'min_class_share': MIN_CLASS_SHARE, 'boot_stability': N_BOOT_STABILITY,
                            'boot_h2': N_BOOT_H2, 'boot_h4': N_BOOT_H4, 'boot_sensitivity': N_BOOT_SENS, 'imputations': N_IMPUTATIONS,
                            'workers': N_WORKERS, 'quick_development_run': QUICK})

# The frozen dataset: verify every file against its manifest before reading anything.
sys.dont_write_bytecode = True
sys.path.insert(0, str(FREEZE / 'code'))
from load_frozen import load, verify
problems = verify(verbose=False)
_junk = re.compile(r'(^|/)(\.DS_Store|\._[^/]+|Thumbs\.db|desktop\.ini|~\$[^/]+|\.~lock\.[^/]+#)$', re.IGNORECASE)
_ignored = [p.split(': ', 1)[1] for p in problems if p.startswith('not in manifest: ') and _junk.search(p.split(': ', 1)[1])]
problems = [p for p in problems if not (p.startswith('not in manifest: ') and p.split(': ', 1)[1] in _ignored)]
if problems:
    raise RuntimeError('AE_freeze_v1.4 does not match its manifest: ' + '; '.join(problems[:5]))
if _ignored:
    log('0 Setup', 'Files added by the operating system or Office inside AE_freeze_v1.4 (ignored by the integrity check)', _ignored)
MANIFEST = json.loads((FREEZE / 'manifest.json').read_text(encoding='utf-8'))
if MANIFEST.get('freeze_version') != 'v1.4':
    raise RuntimeError('This notebook is registered for AE_freeze_v1.4.')
log('0 Setup', 'Frozen dataset', f"AE_freeze_v1.4 built {MANIFEST['built']}; every file matches manifest.json")

In [ ]:
# Computational module: written next to the outputs and imported, so that parallel worker processes can use it on any platform.
TOOLS_SOURCE = r'''"""Computational tools for the registered subphenotype analysis of AE_freeze_v1.4.

* `fit_diag`: EM for the Gaussian mixture with diagonal covariance and missing values (full-information likelihood). It follows
  StepMix 3.0.0 'gaussian_diag_nan' step for step: per start, uniform random responsibilities from one RandomState stream (stored
  as log(clip(r)) and exponentiated, as StepMix does), an initial M-step of the emission model with uniform class weights, then
  E-step / M-step pairs (class weights and emission parameters updated together), stopping when the average log-likelihood changes
  by less than 1e-10 (at most 1000 iterations); the start with the highest log-likelihood is kept, with the parameters after its
  last M-step. It differs only in arithmetic (matrix products instead of StepMix's element-wise sums), so results agree to
  floating-point rounding; the notebook checks this on the main data. It is used for the resampling refits, where StepMix itself
  would take days.
* BCH three-step estimates (Vermunt 2010; as in StepMix: modal assignment, D = P(assigned | true), weights = modal @ pinv(D)).
* Class alignment across refits (StepMix rule: the permutation minimizing the mean squared difference of posterior probabilities).
* `fit_full`: EM for the Gaussian mixture with full covariance and missing values (sensitivity analysis).
* Consensus k-means after multiple imputation (sensitivity analysis).
Worker functions take only picklable arguments so that they run in separate processes on any platform.
"""
import itertools
import numpy as np
from scipy.special import logsumexp

ABS_TOL, MAX_ITER = 1e-10, 1000


# ------------------------------------------------------------------ diagonal Gaussian mixture with missing values
def prep(X):
    X = np.asarray(X, float)
    M = ~np.isnan(X)
    X0 = np.where(M, X, 0.0)
    return X0, M.astype(float), X0 ** 2


def _m_step(data, resp):
    """Class weights (clipped as in StepMix) and emission parameters from responsibilities; sums over observed values only."""
    X0, Mf, X02 = data
    rs = resp.T @ Mf
    sx = resp.T @ X0
    means = sx / rs
    covs = ((resp.T @ X02) - 2 * means * sx + means ** 2 * rs) / rs
    return np.clip(resp.sum(axis=0) / resp.shape[0], 1e-15, 1 - 1e-15), means, covs


def _log_emission(X0, Mf, X02, means, covs):
    prec = 1.0 / covs
    quad = X02 @ prec.T - 2 * X0 @ (means * prec).T + Mf @ (means ** 2 * prec).T
    return -0.5 * (Mf @ np.log(2 * np.pi * covs).T + quad)


def _e_step(data, params):
    """Average log-likelihood and log responsibilities at the given parameters."""
    weights, means, covs = params
    lp = _log_emission(*data, means, covs) + np.log(weights)[None, :]
    lse = logsumexp(lp, axis=1)
    return float(lse.mean()), lp - lse[:, None]


def em_diag(data, params, abs_tol=ABS_TOL, max_iter=MAX_ITER):
    """EM from the given parameters (StepMix order: E-step, then M-step of weights and emission parameters together).
    Returns (average log-likelihood at the last E-step [StepMix lower_bound], parameters after the last M-step, iterations,
    average log-likelihood at those parameters [StepMix lower_bound_buffer_]). A start whose likelihood or variances become
    non-finite or non-positive is returned as failed (NaN, None)."""
    lower = -np.inf
    with np.errstate(divide='ignore', invalid='ignore', over='ignore', under='ignore'):
        for it in range(1, max_iter + 1):
            prev = lower
            lower, log_resp = _e_step(data, params)
            params = _m_step(data, np.exp(log_resp))
            if not np.isfinite(lower) or not np.all(np.isfinite(params[2])) or np.any(params[2] <= 0):
                return np.nan, None, it, np.nan
            if abs(lower - prev) < abs_tol:
                break
        after = _e_step(data, params)[0]
    return lower, params, it, after


def random_start(data, k, rng):
    """StepMix random initialization: uniform responsibilities normalized by row, stored as log(clip(r, 1e-15, 1 - 1e-15)) and
    exponentiated, M-step of the emission model only, uniform class weights."""
    n = data[0].shape[0]
    resp = rng.uniform(size=(n, k))
    resp /= resp.sum(axis=1)[:, np.newaxis]
    resp = np.exp(np.log(np.clip(resp, 1e-15, 1 - 1e-15)))
    _, means, covs = _m_step(data, resp)
    return np.ones(k) / k, means, covs


def fit_diag(X, k, n_init=100, random_state=None, init_params=None, abs_tol=ABS_TOL, max_iter=MAX_ITER):
    """Best of n_init random starts (StepMix scheme), plus optional warm starts given as parameter tuples (run first).
    Returns avg_ll (best StepMix lower bound), params (after the last M-step of the best start), start_lls (log-likelihood of
    every start at its final parameters, as StepMix's lower_bound_buffer_), iterations and the number of failed starts."""
    rng = random_state if isinstance(random_state, np.random.RandomState) else np.random.RandomState(random_state)
    data = prep(X)
    best, best_par, lls, iters = -np.inf, None, [], []
    starts = [('warm', par) for par in (init_params or [])] + [('random', None)] * n_init
    for kind, par in starts:
        if kind == 'random':
            par = random_start(data, k, rng)
        low, p, it, after = em_diag(data, par, abs_tol, max_iter)
        lls.append(after)
        iters.append(it)
        if p is not None and (low > best or best_par is None):
            best, best_par = low, p
    lls = np.array(lls, float)
    return {'avg_ll': best, 'params': best_par, 'start_lls': lls, 'iterations': np.array(iters), 'n_failed': int(np.isnan(lls).sum())}


def posterior_from(data, par):
    weights, means, covs = par
    X0, Mf, X02 = data
    lp = _log_emission(X0, Mf, X02, means, covs) + np.log(weights)[None, :]
    lse = logsumexp(lp, axis=1)
    return np.exp(lp - lse[:, None]), float(lse.sum())


def posterior(X, par):
    return posterior_from(prep(X), par)


def n_params_diag(k, d):
    return (k - 1) + 2 * k * d


def fit_stats(avg_ll, n, n_par):
    ll = avg_ll * n
    return {'log_likelihood': ll, 'n_parameters': n_par, 'aic': -2 * ll + 2 * n_par, 'bic': -2 * ll + n_par * np.log(n),
            'sabic': -2 * ll + n_par * np.log((n + 2) / 24)}


def relative_entropy(post):
    k = post.shape[1]
    if k == 1:
        return np.nan
    p = np.clip(post, 1e-15, 1)
    return float(1 - (-(p * np.log(p)).sum()) / (post.shape[0] * np.log(k)))


def select(X, k_max=6, n_init=100, seed=0, min_share=0.05, warm=None, fallback=500):
    """Fit 1..k_max classes and apply the registered rule: lowest BIC among solutions whose smallest class (modal
    assignment) holds at least min_share of the children. Each k uses its own RandomState(seed), as StepMix does. A class
    number whose best log-likelihood is reached by fewer than two starts is refitted with `fallback` starts (registered)."""
    X = np.asarray(X, float)
    n, d = X.shape
    res = {}
    for k in range(1, k_max + 1):
        f = fit_diag(X, k, n_init=n_init, random_state=np.random.RandomState(seed), init_params=(warm or {}).get(k))
        reached, starts = int((np.abs(f['start_lls'] - f['avg_ll']) < 1e-6).sum()), n_init
        if k > 1 and reached < 2 and fallback and fallback > n_init:
            f = fit_diag(X, k, n_init=fallback, random_state=np.random.RandomState(seed), init_params=(warm or {}).get(k))
            reached, starts = int((np.abs(f['start_lls'] - f['avg_ll']) < 1e-6).sum()), fallback
        f['reached'], f['starts'] = reached, starts
        post, _ = posterior(X, f['params'])
        sizes = np.bincount(post.argmax(1), minlength=k)
        st = fit_stats(f['avg_ll'], n, n_params_diag(k, d))
        res[k] = dict(f, post=post, sizes=sizes, min_share=sizes.min() / n, **st)
    ok = [k for k in res if res[k]['min_share'] >= min_share]
    k_best = min(ok, key=lambda k: res[k]['bic']) if ok else 1
    return k_best, res


# ------------------------------------------------------------------ alignment and agreement
def best_permutation(ref_post, post):
    """Permutation p of the columns of post minimizing the mean squared difference with ref_post (StepMix rule)."""
    k = ref_post.shape[1]
    best, best_p = np.inf, None
    for p in itertools.permutations(range(k)):
        s = np.mean((ref_post - post[:, list(p)]) ** 2)
        if s < best:
            best, best_p = s, list(p)
    return best_p


def match_by_profile(means_a, means_b):
    """Columns of b matched to classes of a by the Hungarian assignment on the Pearson correlation of class mean profiles."""
    from scipy.optimize import linear_sum_assignment
    ka, kb = means_a.shape[0], means_b.shape[0]
    r = np.corrcoef(np.vstack([means_a, means_b]))[:ka, ka:]
    rows, cols = linear_sum_assignment(-r)
    return dict(zip(cols, rows)), r


def adjusted_rand(a, b):
    from sklearn.metrics import adjusted_rand_score
    return float(adjusted_rand_score(a, b))


# ------------------------------------------------------------------ BCH three-step estimates
def bch_weights(post):
    """Weights of the empirical BCH correction with modal assignment (StepMix: corrections.compute_bch_matrix)."""
    n, k = post.shape
    modal = np.zeros_like(post)
    modal[np.arange(n), post.argmax(1)] = 1
    D = (post.T @ modal / post.mean(axis=0).reshape(-1, 1)) / n
    return modal @ np.linalg.pinv(D)


def bch_means(W, y, binary=False):
    """Class-specific means of a distal outcome; missing values ignored. For a binary outcome the mean is a proportion, clipped to
    [1e-15, 1 - 1e-15] as in StepMix (BCH weights can be negative, so the weighted proportion can fall outside [0, 1]; the clipped
    value is the maximum of the weighted Bernoulli likelihood within [0, 1])."""
    y = np.asarray(y, float)
    obs = ~np.isnan(y)
    w = W[obs]
    with np.errstate(divide='ignore', invalid='ignore'):
        m = (w * y[obs, None]).sum(axis=0) / w.sum(axis=0)
    return np.clip(m, 1e-15, 1 - 1e-15) if binary else m


# ------------------------------------------------------------------ workers for resampling (picklable arguments only)
def stability_worker(args):
    """One bootstrap refit of the full class-number selection; returns the chosen k and the adjusted Rand index between the
    resample model's assignment of the original children and the main assignment."""
    X, ref_labels, seed, k_max, n_init, min_share = args
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, X.shape[0], X.shape[0])
    k_b, res = select(X[idx], k_max=k_max, n_init=n_init, seed=seed + 1, min_share=min_share, fallback=None)
    post, _ = posterior(X, res[k_b]['params'])
    return {'k': int(k_b), 'ari': adjusted_rand(ref_labels, post.argmax(1)), 'bic_by_k': [float(res[k]['bic']) for k in sorted(res)],
            'min_share_by_k': [float(res[k]['min_share']) for k in sorted(res)], 'n_failed': int(sum(res[k]['n_failed'] for k in res))}


def h2_worker(args):
    """One bootstrap replicate of the BCH estimates: refit the measurement model with k classes on the resample (warm start from
    the main solution plus random starts), match its classes to the main model by the correlation of class mean profiles
    (Hungarian assignment; registered rule for label switching), compute BCH estimates on the resample."""
    X, Y, ref_post, main_params, k, seed, n_random, binary = args
    rng = np.random.RandomState(seed)
    n = X.shape[0]
    idx = rng.randint(0, n, n)
    f = fit_diag(X[idx], k, n_init=n_random, random_state=np.random.RandomState(seed + 1), init_params=[main_params])
    mapping, _ = match_by_profile(main_params[1], f['params'][1])          # registered rule: match classes by profile correlation
    perm = [None] * k
    for b_cls, a_cls in mapping.items():
        perm[a_cls] = b_cls
    post_rep, _ = posterior(X[idx], f['params'])
    post_rep = post_rep[:, perm]
    W = bch_weights(post_rep)
    out = {name: bch_means(W, y[idx], binary=name in binary).tolist() for name, y in Y.items()}
    out['_avg_ll'] = f['avg_ll']
    out['_n_failed'] = f['n_failed']
    return out


# ------------------------------------------------------------------ full-covariance Gaussian mixture with missing values
LOG2PI = np.log(2 * np.pi)


def _pattern_groups(X):
    """Rows grouped by missingness pattern: (observed columns, missing columns, rows, observed values)."""
    M = ~np.isnan(X)
    pats, inv = np.unique(M, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    out = []
    for g in range(len(pats)):
        rows = np.where(inv == g)[0]
        oi, mi = np.where(pats[g])[0], np.where(~pats[g])[0]
        out.append((oi, mi, rows, X[np.ix_(rows, oi)]))
    return out


def _log_joint_full(groups, n, w, mu, S):
    """log w_c + log N(x_obs | mu_c,obs, S_c,obs,obs) for every row and class (batched over classes)."""
    k = len(w)
    lp = np.empty((n, k))
    logw = np.log(w)
    ar = np.arange(k)
    for oi, mi, rows, Xo in groups:
        if not len(oi):
            lp[rows] = logw
            continue
        L = np.linalg.cholesky(S[np.ix_(ar, oi, oi)])
        sol = np.linalg.solve(L, (Xo[None, :, :] - mu[:, oi][:, None, :]).transpose(0, 2, 1))
        logdet = 2 * np.log(np.diagonal(L, axis1=1, axis2=2)).sum(axis=1)
        lp[rows] = logw[None, :] - 0.5 * (len(oi) * LOG2PI + logdet[None, :] + (sol ** 2).sum(axis=1).T)
    return lp


def _m_step_full(groups, X0, resp, mu, S, reg):
    """M-step with the conditional expectations of the missing coordinates and their conditional covariances."""
    n, d = X0.shape
    k = resp.shape[1]
    nk = resp.sum(axis=0)
    xs = np.broadcast_to(X0, (k, n, d)).copy()
    Csum = np.zeros((k, d, d))
    ar = np.arange(k)
    for oi, mi, rows, Xo in groups:
        if not len(mi):
            continue
        if len(oi):
            Smo = S[np.ix_(ar, mi, oi)]
            B = np.linalg.solve(S[np.ix_(ar, oi, oi)], Smo.transpose(0, 2, 1)).transpose(0, 2, 1)
            cm = mu[:, mi][:, None, :] + np.einsum('krp,kmp->krm', Xo[None, :, :] - mu[:, oi][:, None, :], B)
            Cm = S[np.ix_(ar, mi, mi)] - B @ Smo.transpose(0, 2, 1)
        else:
            cm = np.broadcast_to(mu[:, mi][:, None, :], (k, len(rows), len(mi)))
            Cm = S[np.ix_(ar, mi, mi)]
        xs[np.ix_(ar, rows, mi)] = cm
        Csum[np.ix_(ar, mi, mi)] += resp[rows].sum(axis=0)[:, None, None] * Cm
    mu_new = np.einsum('nk,knd->kd', resp, xs) / nk[:, None]
    S_new = np.empty((k, d, d))
    for c in range(k):
        Z = xs[c] - mu_new[c]
        S_new[c] = ((resp[:, c][:, None] * Z).T @ Z + Csum[c]) / nk[c] + reg * np.eye(d)
    return nk / n, mu_new, S_new


def fit_full(X, k, n_init=100, random_state=None, max_iter=MAX_ITER, abs_tol=1e-8, reg=1e-6):
    """EM for a Gaussian mixture with full covariance matrices and missing values (missing at random; Ghahramani & Jordan 1994),
    computed by missingness pattern and batched over classes. Random initialization as in fit_diag (uniform responsibilities),
    parameters initialized from available-case moments; a ridge of 1e-6 is added to every covariance matrix."""
    X = np.asarray(X, float)
    n, d = X.shape
    rng = random_state if isinstance(random_state, np.random.RandomState) else np.random.RandomState(random_state)
    M = ~np.isnan(X)
    X0 = np.where(M, X, 0.0)
    groups = _pattern_groups(X)
    best, best_par, lls, its = -np.inf, None, [], []
    for _ in range(n_init):
        resp = rng.uniform(size=(n, k))
        resp /= resp.sum(axis=1)[:, None]
        mu = np.zeros((k, d)); S = np.zeros((k, d, d)); w = resp.mean(0)
        for c in range(k):
            r = resp[:, c][:, None] * M
            mu[c] = (r * X0).sum(0) / r.sum(0)
            Z = np.where(M, X - mu[c], 0.0)
            S[c] = (resp[:, c][:, None] * Z).T @ Z / resp[:, c].sum() + reg * np.eye(d)
        lower, ok, it = -np.inf, True, 0
        for it in range(1, max_iter + 1):
            prev = lower
            try:
                lp = _log_joint_full(groups, n, w, mu, S)
            except np.linalg.LinAlgError:
                ok = False
                break
            lse = logsumexp(lp, axis=1)
            lower = float(lse.mean())
            resp = np.exp(lp - lse[:, None])
            if not np.isfinite(lower):
                ok = False
                break
            if abs(lower - prev) < abs_tol:
                break
            w, mu, S = _m_step_full(groups, X0, resp, mu, S, reg)
        lls.append(lower if ok else -np.inf)
        its.append(it)
        if ok and lower > best:
            best, best_par = lower, (w.copy(), mu.copy(), S.copy())
    return {'avg_ll': best, 'params': best_par, 'start_lls': np.array(lls), 'iterations': np.array(its),
            'n_parameters': (k - 1) + k * d + k * d * (d + 1) // 2}


def posterior_full(X, par):
    X = np.asarray(X, float)
    w, mu, S = par
    lp = _log_joint_full(_pattern_groups(X), X.shape[0], w, mu, S)
    lse = logsumexp(lp, axis=1)
    return np.exp(lp - lse[:, None]), float(lse.sum())


# ------------------------------------------------------------------ consensus k-means after multiple imputation
def consensus_worker(args):
    """Co-clustering counts of k-means over subsamples of one imputed data set (Monti 2003). The same subsamples are used for every k.
    Returns uint8 count matrices: 'both' (pairs sampled together) and one co-clustering matrix per k."""
    Xi, ks, n_sub, frac, seed = args
    from sklearn.cluster import KMeans
    rng = np.random.RandomState(seed)
    n = Xi.shape[0]
    both = np.zeros((n, n), np.uint8)
    co = {k: np.zeros((n, n), np.uint8) for k in ks}
    for b in range(n_sub):
        idx = np.sort(rng.choice(n, int(round(frac * n)), replace=False))
        both[np.ix_(idx, idx)] += 1
        for k in ks:
            lab = KMeans(n_clusters=k, n_init=1, random_state=rng.randint(1 << 30)).fit_predict(Xi[idx])
            co[k][np.ix_(idx, idx)] += (lab[:, None] == lab[None, :]).astype(np.uint8)
    return {'both': both, **co}


def fullcov_worker(args):
    """Fit the full-covariance mixture with k classes (worker for parallel fits over class numbers)."""
    X, k, n_init, seed = args
    f = fit_full(X, k, n_init=n_init, random_state=np.random.RandomState(seed))
    if f['params'] is None:
        return {'k': k, 'avg_ll': -np.inf, 'params': None, 'n_parameters': f['n_parameters'], 'post': None}
    post, _ = posterior_full(X, f['params'])
    return {'k': k, 'avg_ll': f['avg_ll'], 'params': f['params'], 'n_parameters': f['n_parameters'], 'post': post,
            'start_lls': f['start_lls'], 'iterations': f['iterations']}
'''
(CODE / 'ae_lpa_tools.py').write_text(TOOLS_SOURCE, encoding='utf-8')
sys.path.insert(0, str(CODE))
import importlib
import ae_lpa_tools as T
importlib.reload(T)
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
MP = multiprocessing.get_context(os.environ.get('AE_MP', 'spawn'))   # 'spawn' works on every platform


def parallel_map(fn, args_list):
    """Run fn over args_list in N_WORKERS processes (order preserved)."""
    if N_WORKERS == 1 or len(args_list) == 1:
        return [fn(a) for a in args_list]
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=MP) as ex:
        return list(ex.map(fn, args_list, chunksize=1))

## Step 1. Analysis dataset
One row per child (index admission). Indicators: the first value from 24 h before to 24 h after hospital admission, including
results filed under emergency visits linked to the admission (`labs.linked_admission_id`), of 22 blood tests; and the first
charted temperature, heart rate and respiratory rate from admission to 24 h after it. Each laboratory value is divided by its
own reference limit; heart and respiratory rates become z-scores for age (Fleming 2011); then log scale where registered,
winsorizing at the 0.5th and 99.5th percentiles, and standardization to mean 0 and SD 1 in the clustering population.

In [ ]:
# Indicator definitions (registered order; the first member of a correlated pair is the one kept).
INDICATORS = [  # code, label, domain, scale
    ('neutrophils_abs', 'Neutrophil count', 'Inflammation and blood count', 'log'),
    ('wbc', 'White cell count', 'Inflammation and blood count', 'log'),
    ('lymphocytes_pct', 'Lymphocyte percentage', 'Inflammation and blood count', 'linear'),
    ('platelets', 'Platelets', 'Inflammation and blood count', 'log'),
    ('hemoglobin', 'Hemoglobin', 'Inflammation and blood count', 'linear'),
    ('hs_crp', 'High-sensitivity CRP', 'Inflammation and blood count', 'log'),
    ('procalcitonin', 'Procalcitonin', 'Inflammation and blood count', 'log'),
    ('alt', 'ALT', 'Cell injury', 'log'), ('ast', 'AST', 'Cell injury', 'log'), ('ldh', 'LDH', 'Cell injury', 'log'),
    ('ck', 'CK', 'Cell injury', 'log'), ('bilirubin_total', 'Total bilirubin', 'Cell injury', 'log'),
    ('albumin', 'Albumin', 'Cell injury', 'linear'),
    ('creatinine', 'Creatinine', 'Renal and metabolic', 'log'), ('urea', 'Urea', 'Renal and metabolic', 'log'),
    ('sodium', 'Sodium', 'Renal and metabolic', 'linear'), ('ammonia', 'Ammonia', 'Renal and metabolic', 'log'),
    ('lactate', 'Lactate', 'Renal and metabolic', 'log'),
    ('inr', 'INR', 'Coagulation', 'log'), ('aptt', 'APTT', 'Coagulation', 'log'), ('fibrinogen', 'Fibrinogen', 'Coagulation', 'log'),
    ('d_dimer', 'D-dimer', 'Coagulation', 'log'),
    ('temp_c', 'Temperature', 'Charted vital signs', 'linear'), ('hr_z', 'Heart rate (z-score for age)', 'Charted vital signs', 'linear'),
    ('rr_z', 'Respiratory rate (z-score for age)', 'Charted vital signs', 'linear')]
IND = [c for c, *_ in INDICATORS]
LABEL = {c: l for c, l, *_ in INDICATORS}
DOMAIN = {c: d for c, _, d, _ in INDICATORS}
LOG_SCALE = {c for c, _, _, s in INDICATORS if s == 'log'}
LAB_IND = IND[:22]
VIT_IND = IND[22:]
# Reference limit used for each laboratory value: the lower limit where low values are the abnormality of interest, otherwise the
# upper limit. (For a fixed limit the choice does not change the standardized indicator; it matters for age-specific limits.)
LOWER_LIMIT = {'lymphocytes_pct', 'platelets', 'hemoglobin', 'albumin', 'fibrinogen', 'sodium'}
CORR_PAIRS = [('neutrophils_abs', 'wbc'), ('alt', 'ast'), ('creatinine', 'urea')]     # registered: keep the first if rho >= 0.8
SCORE_ITEMS = {'hs_crp': 1, 'procalcitonin': 1, 'alt': 1, 'ast': 1, 'ldh': 1, 'd_dimer': 1, 'albumin': -1, 'platelets': -1}
VALIDATOR_LABS = {'il6': 'IL-6', 'il10': 'IL-10', 'tnf_alpha': 'TNF-alpha', 'ifn_gamma': 'IFN-gamma', 'ferritin': 'Ferritin',
                  'csf_protein': 'CSF protein', 'csf_wbc': 'CSF white cells'}
detail('Reference limit: the lower limit for lymphocyte percentage, platelets, hemoglobin, albumin, fibrinogen and sodium; the upper '
       'limit for the other 16 tests. A missing or non-positive limit is replaced by the most frequent positive limit of that test among '
       'all results of the 740 index admissions.')
detail('Censored results (reported as below or above a reporting limit) enter with the reporting limit as their value.')
detail('First value: earliest report time in the window (no ties occurred); windows include their limits.')

# Fleming S et al. Normal ranges of heart rate and respiratory rate in children from birth to 18 years of age. Lancet 2011;
# 377:1011-8 (doi:10.1016/S0140-6736(10)62226-X), web appendix p 12, Web Tables 4 (respiratory rate) and 5 (heart rate):
# 1st, 10th, 25th, 50th, 75th, 90th and 99th centiles by age band (the 'Birth' row of the heart-rate table is not used).
FLEMING_BANDS_MONTHS = [(0, 3), (3, 6), (6, 9), (9, 12), (12, 18), (18, 24), (24, 36), (36, 48), (48, 72), (72, 96), (96, 144), (144, 180), (180, 216)]
FLEMING_RR = [[25, 34, 40, 43, 52, 57, 66], [24, 33, 38, 41, 49, 55, 64], [23, 31, 36, 39, 47, 52, 61], [22, 30, 35, 37, 45, 50, 58],
              [21, 28, 32, 35, 42, 46, 53], [19, 25, 29, 31, 36, 40, 46], [18, 22, 25, 28, 31, 34, 38], [17, 21, 23, 25, 27, 29, 33],
              [17, 20, 21, 23, 25, 27, 29], [16, 18, 20, 21, 23, 24, 27], [14, 16, 18, 19, 21, 22, 25], [12, 15, 16, 18, 19, 21, 23],
              [11, 13, 15, 16, 18, 19, 22]]
FLEMING_HR = [[107, 123, 133, 143, 154, 164, 181], [104, 120, 129, 140, 150, 159, 175], [98, 114, 123, 134, 143, 152, 168],
              [93, 109, 118, 128, 137, 145, 161], [88, 103, 112, 123, 132, 140, 156], [82, 98, 106, 116, 126, 135, 149],
              [76, 92, 100, 110, 119, 128, 142], [70, 86, 94, 104, 113, 123, 136], [65, 81, 89, 98, 108, 117, 131],
              [59, 74, 82, 91, 101, 111, 123], [52, 67, 75, 84, 93, 103, 115], [47, 62, 69, 78, 87, 96, 108], [43, 58, 65, 73, 83, 92, 104]]
from scipy.stats import norm
FLEMING_Z = norm.ppf([.01, .10, .25, .50, .75, .90, .99])


def age_z(values, age_days, table):
    """z-score for age: piecewise-linear interpolation between the published centiles on the standard-normal scale; beyond the
    1st or 99th centile, linear extrapolation with the slope of the outer interval."""
    out = np.full(len(values), np.nan)
    for i, (v, a) in enumerate(zip(np.asarray(values, float), np.asarray(age_days, float) / 30.4375)):
        if np.isnan(v) or np.isnan(a):
            continue
        b = next((j for j, (lo, hi) in enumerate(FLEMING_BANDS_MONTHS) if lo <= a < hi), len(FLEMING_BANDS_MONTHS) - 1)
        c = np.asarray(table[b], float)
        if v <= c[0]:
            out[i] = FLEMING_Z[0] + (v - c[0]) * (FLEMING_Z[1] - FLEMING_Z[0]) / (c[1] - c[0])
        elif v >= c[-1]:
            out[i] = FLEMING_Z[-1] + (v - c[-1]) * (FLEMING_Z[-1] - FLEMING_Z[-2]) / (c[-1] - c[-2])
        else:
            out[i] = np.interp(v, c, FLEMING_Z)
    return out


detail('Heart and respiratory rates: z-score for age from the centiles of Fleming 2011 (web appendix, Web Tables 4 and 5) by age band, '
       'by piecewise-linear interpolation between centiles on the standard-normal scale and linear extrapolation beyond the 1st and 99th '
       'centiles; heart rate is the monitor heart rate where charted, otherwise the pulse (vital_signs.heart_rate).')

In [ ]:
# Read the frozen tables (typed) and assemble the per-child data.
IDX = load('index_admissions')
ADM = load('admissions')[['visit_id', 'admit_time', 'admit_year', 'age_days', 'age_years', 'sex', 'admission_source']]
SUMM = load('admission_summary')[['visit_id', 'death', 'dama', 'dama_critical_final', 'dama_critical_final_source', 'death_or_dama_critical_final',
                                  'imv_ever', 'vasoactive_ever', 'any_picu_stay', 'los_hours']]
NOTES = load('admission_notes')[['visit_id', 'consciousness', 'consciousness_level', 'pupil_light_reflex', 'exam_temp_c', 'exam_pulse_bpm',
                                 'exam_rr_bpm', 'onset_hours_acute']]
D = IDX.merge(ADM, on='visit_id', how='left').merge(SUMM, on='visit_id', how='left').merge(NOTES, on='visit_id', how='left')
D = D.set_index('visit_id')
assert len(D) == 740 and D.admit_time.notna().all()
D['period'] = np.where(D.admit_year <= 2022, '2019-2022', '2023-2026')

LABS_ALL = load('labs')
LABS_ALL = LABS_ALL[LABS_ALL.linked_admission_id.isin(set(D.index)) & LABS_ALL.value.notna() &
                    LABS_ALL.analyte.isin(LAB_IND + list(VALIDATOR_LABS))].copy()
LABS_ALL['h'] = (LABS_ALL.report_time - LABS_ALL.linked_admission_id.map(D.admit_time)).dt.total_seconds() / 3600
LABS_ALL['limit'] = np.where(LABS_ALL.analyte.isin(LOWER_LIMIT), LABS_ALL.ref_low, LABS_ALL.ref_high)
_mode_lim = LABS_ALL[LABS_ALL.limit > 0].groupby('analyte').limit.agg(lambda s: s.value_counts().idxmax())
_bad = ~(LABS_ALL.limit > 0)
LABS_ALL.loc[_bad, 'limit'] = LABS_ALL.loc[_bad, 'analyte'].map(_mode_lim)
LABS_ALL['ratio'] = LABS_ALL.value / LABS_ALL.limit
LABS_ALL = LABS_ALL.sort_values(['linked_admission_id', 'analyte', 'report_time', 'report_id'])


def lab_matrix(lo_h, hi_h, analytes, own_only=False, how='first', column='ratio'):
    """One value per child and analyte in the window [lo_h, hi_h] around admission: first, or worst (highest for tests read
    against the upper limit, lowest for tests read against the lower limit)."""
    w = LABS_ALL[LABS_ALL.analyte.isin(analytes) & LABS_ALL.h.between(lo_h, hi_h)]
    if own_only:
        w = w[w.visit_id == w.linked_admission_id]
    if how == 'first':
        w = w.drop_duplicates(['linked_admission_id', 'analyte'], keep='first')
    else:
        hi_side = ~w.analyte.isin(LOWER_LIMIT)
        w = pd.concat([w[hi_side].sort_values('ratio', ascending=False), w[~hi_side].sort_values('ratio')]).drop_duplicates(['linked_admission_id', 'analyte'])
    return w.pivot(index='linked_admission_id', columns='analyte', values=column).reindex(index=D.index, columns=analytes)


VS = load('vital_signs')
VS = VS[VS.visit_id.isin(set(D.index))].copy()
VS['h'] = (VS.measure_time - VS.visit_id.map(D.admit_time)).dt.total_seconds() / 3600
VS = VS.sort_values(['visit_id', 'measure_time', 'row_seq'])


def vital_matrix(how='first'):
    v = VS[VS.h.between(0, 24)]
    out = pd.DataFrame(index=D.index)
    for c, src in [('temp_c', 'temp_c'), ('heart_rate', 'heart_rate'), ('resp_rate', 'resp_rate')]:
        s = v.dropna(subset=[src])
        s = s.drop_duplicates('visit_id') if how == 'first' else s.sort_values(src, ascending=False).drop_duplicates('visit_id')
        out[c] = s.set_index('visit_id')[src]
    out['hr_z'] = age_z(out.heart_rate.values, D.age_days.values, FLEMING_HR)
    out['rr_z'] = age_z(out.resp_rate.values, D.age_days.values, FLEMING_RR)
    return out


def raw_indicators(lo_h=-24, hi_h=24, own_only=False, how='first', exam_vitals=False):
    R = lab_matrix(lo_h, hi_h, LAB_IND, own_only, how)
    if exam_vitals:
        v = pd.DataFrame({'temp_c': D.exam_temp_c, 'hr_z': age_z(D.exam_pulse_bpm.values, D.age_days.values, FLEMING_HR),
                          'rr_z': age_z(D.exam_rr_bpm.values, D.age_days.values, FLEMING_RR)}, index=D.index)
    else:
        v = vital_matrix(how)
    return pd.concat([R, v[['temp_c', 'hr_z', 'rr_z']]], axis=1)[IND]


def transform(raw, rows, winsorize=True, columns=None):
    """Log scale where registered, winsorize at the 0.5th and 99.5th percentiles, standardize (all within rows)."""
    X = raw.loc[rows, columns or list(raw.columns)].astype(float).copy()
    for c in X.columns:
        if c in LOG_SCALE:
            X[c] = np.log(X[c])
    if winsorize:
        X = X.clip(X.quantile(0.005), X.quantile(0.995), axis=1)
    return (X - X.mean()) / X.std()


RAW = raw_indicators()
D['n_labs'] = RAW[LAB_IND].notna().sum(axis=1)
D['few_tests'] = D.n_labs < 11
D['clustered'] = ~D.neonate.astype(bool) & ~D.few_tests
VITALS_RAW = vital_matrix()
D['heart_rate'], D['resp_rate'] = VITALS_RAW.heart_rate, VITALS_RAW.resp_rate
LABVAL = lab_matrix(-24, 24, LAB_IND, column='value')          # first values in original units (descriptive tables)
VALID = lab_matrix(-24, 48, list(VALIDATOR_LABS), column='value')
_zero = {}
for c in VALIDATOR_LABS:
    D[c] = VALID[c]
    v = VALID[c]
    half_min = v[v > 0].min() / 2
    if c != 'csf_wbc':                                                      # CSF white cells: log(count + 1), zero is a valid count
        _zero[VALIDATOR_LABS[c]] = int((v <= 0).sum())
    D['log_' + c] = np.log(v.where(v.isna() | (v > 0), half_min))           # a result of 0 (below the assay range) -> half the smallest positive value
    D['log_' + c + '_zero_missing'] = np.log(v.where(v > 0))
change('Cytokine and other validator results reported as exactly 0 (not marked as censored) enter the log scale as half the smallest '
       'positive result of that test, instead of being set to missing; added after the independent check found two IL-10 results of 0. '
       'The H2 IL-10 estimate with these results set to missing is logged as a check.')
log('1 Data', 'Index admissions', len(D))
log('1 Data', 'Validator results of exactly 0 (first value in the window, 740 children)', {k: v for k, v in _zero.items() if v} or 'none')
log('1 Data', 'Left out of clustering: neonates / fewer than 11 of the 22 blood tests / both', f"{int(D.neonate.sum())} / {int(D.few_tests.sum())} / {int((D.neonate & D.few_tests).sum())}")
log('1 Data', 'Clustering population', f"{int(D.clustered.sum())} ({D[D.clustered].period.value_counts().sort_index().to_dict()})")

# Availability (share of the 740 measured in the window) and the 60% rule.
AVAIL = pd.DataFrame({'indicator': [LABEL[c] for c in IND], 'domain': [DOMAIN[c] for c in IND], 'scale': ['log' if c in LOG_SCALE else 'linear' for c in IND],
                      'measured_740': [int(RAW[c].notna().sum()) for c in IND],
                      'pct_740': [round(100 * RAW[c].notna().mean(), 1) for c in IND],
                      'measured_clustered': [int(RAW.loc[D.clustered, c].notna().sum()) for c in IND],
                      'pct_clustered': [round(100 * RAW.loc[D.clustered, c].notna().mean(), 1) for c in IND]}, index=IND)
AVAIL['passes_60pct_rule'] = AVAIL.pct_740 >= 60
CAND = [c for c in IND if AVAIL.loc[c, 'passes_60pct_rule']]
log('1 Data', 'Indicators passing the 60% availability rule', f'{len(CAND)} of {len(IND)}')

# Correlated pairs (Spearman, clustering population, pairwise complete).
XC_ALL = transform(RAW, D.index[D.clustered], columns=CAND)
RHO = XC_ALL.corr(method='spearman')
PAIRS = []
for a, b in CORR_PAIRS:
    r = RHO.loc[a, b] if a in RHO and b in RHO else np.nan
    PAIRS.append({'pair': f'{LABEL[a]} / {LABEL[b]}', 'spearman_rho': round(float(r), 3), 'dropped': LABEL[b] if r >= 0.8 else ''})
DROPPED = [b for (a, b), p in zip(CORR_PAIRS, PAIRS) if p['dropped']]
_other = [(LABEL[a], LABEL[b], round(float(RHO.loc[a, b]), 3)) for i, a in enumerate(CAND) for b in CAND[i + 1:]
          if abs(RHO.loc[a, b]) >= 0.8 and (a, b) not in CORR_PAIRS]
FINAL = [c for c in CAND if c not in DROPPED]
AVAIL['in_model'] = [c in FINAL for c in IND]
deviation('Correlated pairs: the member listed first in the analysis plan\'s indicator table is kept (plan decision P6, and the order of '
          'the registration\'s own indicator list): neutrophil count over white cell count, ALT over AST, creatinine over urea. The '
          'registration\'s sentence in Q12 names white cells before neutrophils, which read literally keeps the white cell count; an '
          'additional, unregistered sensitivity analysis keeps the white cell count instead of the neutrophil count.')
log('1 Data', 'Registered correlated pairs', PAIRS)
log('1 Data', 'Other pairs with |rho| >= 0.8 (not registered for dropping)', _other or 'none')
log('1 Data', 'Clustering indicators', f'{len(FINAL)}: ' + ', '.join(LABEL[c] for c in FINAL))
CLU = D.index[D.clustered]
XC = transform(RAW, CLU, columns=FINAL)
detail('Standardization uses the sample SD (n - 1); with the population SD every log-likelihood shifts by a constant and the solutions are '
       'identical.')
_flag = lab_matrix(-24, 24, LAB_IND, column='outside_physiological_limits').loc[CLU]
log('1 Data', 'First laboratory values flagged by the freeze as outside physiological limits (clustered children)',
    int(_flag.apply(lambda col: col.map(lambda x: bool(x) if pd.notna(x) else False)).values.sum()))
detail('Q15: the freeze sets implausible vital signs to missing; laboratory results outside physiological limits keep their value and carry a '
       'flag (labs.outside_physiological_limits). The number of flagged first values used for clustering is logged.')
X = XC.values
N, P = X.shape
log('1 Data', 'Missing indicator values in the clustering matrix', f'{round(100 * float(np.isnan(X).mean()), 1)}%')
# Standardized values of every candidate indicator (the score uses AST even if it were dropped from the model)
XS_ALL = transform(RAW, CLU, columns=CAND)

## Step 0. Measurement validation: chart review of consciousness and the pupillary light reflex
Sixty index admissions drawn by the frozen release (fixed seed); a clinician recorded both variables without seeing the
extracted values. Agreement: Cohen's kappa (95% CI from 2,000 bootstrap resamples) and exact binomial CIs for the share agreeing.

In [ ]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import beta as _beta
CR_FILE = HERE / 'chart_review_completed.xlsx'
CHART = {}
if CR_FILE.exists():
    cr = pd.read_excel(CR_FILE, sheet_name='review', dtype=str)
    rel = pd.read_excel(FREEZE / 'adjudication_INTERNAL' / 'chart_review_INTERNAL.xlsx', sheet_name='review', dtype=str)
    same = cr[['review_id', 'visit_id']].equals(rel[['review_id', 'visit_id']])
    for c in ['general_examination', 'specialty_examination']:
        same = same and cr[c].fillna('').equals(rel[c].fillna(''))
    if not same:
        raise RuntimeError('The completed chart-review sheet does not match the sheet released with AE_freeze_v1.4.')
    m = cr.merge(load('admission_notes')[['visit_id', 'consciousness', 'pupil_light_reflex']], on='visit_id', how='left')
    rng = np.random.RandomState(SEED)
    rows = []
    for name, rcol, xcol in [('Consciousness', 'reviewer_consciousness', 'consciousness'), ('Pupillary light reflex', 'reviewer_pupil_light_reflex', 'pupil_light_reflex')]:
        a = m[rcol].fillna('not_documented').values
        b = m[xcol].fillna('not_documented').values
        k = int((a == b).sum()); n = len(a)
        kap = cohen_kappa_score(a, b)
        boots = []
        for _ in range(2000):
            i = rng.randint(0, n, n)
            boots.append(1.0 if len(set(a[i]) | set(b[i])) < 2 or (a[i] == b[i]).all() else cohen_kappa_score(a[i], b[i]))
        lo = _beta.ppf(0.025, k, n - k + 1) if k > 0 else 0.0
        hi = _beta.ppf(0.975, k + 1, n - k) if k < n else 1.0
        rows.append({'variable': name, 'children': n, 'agree': k, 'agreement_pct': round(100 * k / n, 1), 'agreement_ci': f'{100 * lo:.1f}-{100 * hi:.1f}',
                     'kappa': round(kap, 3), 'kappa_ci': f'{np.percentile(boots, 2.5):.2f}-{np.percentile(boots, 97.5):.2f}'})
        CHART[name] = pd.crosstab(pd.Series(a, name='reviewer'), pd.Series(b, name='extracted'))
    CHART_TAB = pd.DataFrame(rows)
    log('0 Measurement', 'Chart review (kappa)', CHART_TAB[['variable', 'agree', 'kappa', 'kappa_ci']].to_dict('records'))
else:
    CHART_TAB = pd.DataFrame()
    log('0 Measurement', 'Chart review', 'chart_review_completed.xlsx not found; agreement not computed')
CHART_TAB

## Steps 2 and 3. Latent profile analysis with 1 to 6 classes and the choice of the number of classes
StepMix 3.0.0, Gaussian emission with diagonal covariance and missing values by full-information likelihood
(`gaussian_diag_nan`), 100 random starts per class number, fixed seed. Rule: the lowest BIC among solutions whose smallest class
(modal assignment) holds at least 5% of the children. If the best log-likelihood of a class number is not reached by at least two
starts, that class number is refitted with 500 starts.

In [ ]:
detail('Starts reaching the best log-likelihood: average log-likelihood within 1e-6 of the best start (StepMix lower_bound_buffer_).')
detail('Class size for the 5% rule: number of children whose most likely class it is (modal assignment).')
detail('SABIC uses the sample-size adjustment of Sclove (n* = (n + 2) / 24); StepMix 3.0.0 reports a different formula.')
FITS, MODELS = [], {}
for k in range(1, K_MAX + 1):
    t = time.time()
    starts = N_STARTS
    while True:
        mdl = StepMix(n_components=k, measurement='gaussian_diag_nan', n_init=starts, random_state=SEED, progress_bar=0, verbose=0)
        mdl.fit(X)
        buf = np.array(mdl.lower_bound_buffer_)
        reached = int((np.abs(buf - mdl.lower_bound_) < 1e-6).sum())
        if reached >= 2 or k == 1 or starts >= N_STARTS_FALLBACK:
            break
        starts = N_STARTS_FALLBACK
    post = mdl.predict_proba(X)
    sizes = np.bincount(post.argmax(1), minlength=k)
    st = T.fit_stats(mdl.lower_bound_, N, T.n_params_diag(k, P))
    if abs(st['bic'] - mdl.bic(X)) > 1e-6 * abs(st['bic']):
        raise RuntimeError('Parameter count differs from StepMix.')
    mpp = [float(post[post.argmax(1) == c, c].mean()) if sizes[c] else np.nan for c in range(k)]
    FITS.append({'classes': k, 'log_likelihood': round(st['log_likelihood'], 2), 'parameters': st['n_parameters'], 'aic': round(st['aic'], 1),
                 'bic': round(st['bic'], 1), 'sabic': round(st['sabic'], 1), 'relative_entropy': round(T.relative_entropy(post), 3) if k > 1 else np.nan,
                 'smallest_class_n': int(sizes.min()), 'smallest_class_pct': round(100 * sizes.min() / N, 1),
                 'min_mean_posterior': round(min(mpp), 3), 'starts': starts, 'starts_reaching_best': reached,
                 'meets_size_rule': bool(sizes.min() / N >= MIN_CLASS_SHARE)})
    MODELS[k] = mdl
    log('2 Fit', f'{k} classes', f"BIC {st['bic']:.1f}; class sizes {sizes.tolist()}; best LL reached by {reached} of {starts} starts; {time.time() - t:.0f} s")
    if k > 1 and reached < 2:
        deviation(f'{k} classes: the best log-likelihood was reached by fewer than two of {starts} starts.')
FITS = pd.DataFrame(FITS)
eligible = FITS[FITS.meets_size_rule & (FITS.classes >= 1)]
K_STAR = int(eligible.sort_values('bic').classes.iloc[0]) if len(eligible) else 1
log('3 Choice', 'Number of classes (lowest BIC among solutions meeting the 5% rule)', K_STAR)
if K_STAR == K_MAX and bool(np.all(np.diff(FITS.bic.values) < 0)):
    log('3 Choice', 'Note', f'BIC still decreases at the largest registered number of classes ({K_MAX}); the registered range sets the number of classes')
FITS

### Check of the fast EM used for refits
Resampling (stability, H2 bootstrap) and the refits of subsets (H3, sensitivity analyses) use the EM of `ae_lpa_tools`, which follows
StepMix's algorithm step for step but computes the sums by matrix products (StepMix would take days for the resampling). Three
checks on the main data: (a) the same EM map: after 1, 2, 5 and 20 iterations of the same start, the parameters agree with StepMix;
(b) the same likelihood: at StepMix's fitted parameters, the average log-likelihood and the posterior probabilities agree; (c) the
same starts: with the same seed, every start ends at the same log-likelihood. (a) and (b) must hold to 1e-8, or the notebook
stops. (c) is reported: a start can, in principle, end elsewhere when rounding differences of order 1e-16 are amplified along its path.

In [ ]:
_map = []
for k in range(2, K_MAX + 1):
    for iters in (1, 2, 5, 20):
        sm = StepMix(n_components=k, measurement='gaussian_diag_nan', n_init=1, max_iter=iters, random_state=SEED, progress_bar=0, verbose=0)
        sm.fit(X)
        ps = sm.get_parameters()
        _d = T.prep(X)
        _, pf, _, _ = T.em_diag(_d, T.random_start(_d, k, np.random.RandomState(SEED)), max_iter=iters)
        _map.append(max(float(np.max(np.abs(ps['weights'] - pf[0]))), float(np.max(np.abs(ps['measurement']['means'] - pf[1]))),
                        float(np.max(np.abs(ps['measurement']['covariances'] - pf[2])))))
_lik = []
for k in range(1, K_MAX + 1):
    ps = MODELS[k].get_parameters()
    pf, ll_sum = T.posterior(X, (ps['weights'], ps['measurement']['means'], ps['measurement']['covariances']))
    _lik.append(max(abs(ll_sum / N - MODELS[k].score(X)), float(np.max(np.abs(pf - MODELS[k].predict_proba(X))))))
_starts, _best = [], []
for k in range(1, K_MAX + 1):
    f = T.fit_diag(X, k, n_init=MODELS[k].n_init, random_state=SEED)
    dif = np.abs(f['start_lls'] - np.array(MODELS[k].lower_bound_buffer_))
    _starts.append((int((dif < 1e-8).sum()), len(dif), float(np.nanmax(dif))))
    _best.append(abs(f['avg_ll'] - MODELS[k].lower_bound_))
FAST_CHECK = {'em_map_max_difference': max(_map), 'likelihood_max_difference': max(_lik),
              'starts_agreeing': f'{sum(s[0] for s in _starts)} of {sum(s[1] for s in _starts)}',
              'start_ll_max_difference': max(s[2] for s in _starts), 'best_ll_max_difference': max(_best)}
log('2 Fit', 'Fast EM vs StepMix (a) EM map, (b) likelihood, (c) starts',
    {kk: (f'{v:.1e}' if isinstance(v, float) else v) for kk, v in FAST_CHECK.items()})
if not (max(_map) < 1e-8 and max(_lik) < 1e-8):
    raise RuntimeError('The fast EM does not reproduce StepMix; refits would not be valid.')
for k, (a, b, m) in zip(range(1, K_MAX + 1), _starts):
    if a < b:
        log('2 Fit', f'{k} classes: starts ending at a different log-likelihood than in StepMix', f'{b - a} of {b} (largest difference {m:.1e})')

## Step 4. Describe and label (H1)
Class labels fixed in advance: the inflammation and cell-injury score is the mean of the standardized CRP, procalcitonin, ALT,
AST, LDH and D-dimer and of the negated standardized albumin and platelets. The class with the highest mean score is called
hyperinflammatory, the class with the lowest least abnormal. H1 is supported if at least two classes are selected and the
hyperinflammatory class's mean score is at least 0.5 SD above the cohort mean. Other classes are named after their profiles.

In [ ]:
detail('Score: per child, the mean of the available score items (at least 4 of 8); class mean score: posterior-probability-weighted mean '
       'of the child scores; cohort mean and SD over the clustered children with a score.')
detail('Display order of classes: by mean score, S1 (highest, hyperinflammatory) to Sk (lowest, least abnormal).')
MAIN = MODELS[K_STAR]
POST0 = MAIN.predict_proba(X)
_items = pd.DataFrame({c: s * XS_ALL[c] for c, s in SCORE_ITEMS.items()})
SCORE = _items.mean(axis=1).where(_items.notna().sum(axis=1) >= 4)
_sc = SCORE.values
_ok = ~np.isnan(_sc)
cls_score0 = np.array([np.sum(POST0[_ok, c] * _sc[_ok]) / np.sum(POST0[_ok, c]) for c in range(K_STAR)])
ORDER = list(np.argsort(-cls_score0))                     # model class index for S1..Sk
POST = POST0[:, ORDER]
MODAL = POST.argmax(1)
CLS = [f'S{i + 1}' for i in range(K_STAR)]
CLASS_SCORE = cls_score0[ORDER]
SC_MEAN, SC_SD = float(np.nanmean(_sc)), float(np.nanstd(_sc, ddof=1))
H1_EFFECT = (CLASS_SCORE[0] - SC_MEAN) / SC_SD if K_STAR > 1 else np.nan
H1 = bool(K_STAR >= 2 and H1_EFFECT >= 0.5)
HYPER, LEAST = 0, K_STAR - 1
MAIN_PAR = MAIN.get_parameters()['measurement']
MAIN_MEANS = np.asarray(MAIN_PAR['means'])[ORDER]
MAIN_VARS = np.asarray(MAIN_PAR['covariances'])[ORDER]
MAIN_WEIGHTS = np.asarray(MAIN.get_parameters()['weights'])[ORDER]
MAIN_TUPLE = (MAIN_WEIGHTS, MAIN_MEANS, MAIN_VARS)                     # fast-EM parameter tuple in display order
_p_chk, _ = T.posterior(X, MAIN_TUPLE)
assert np.allclose(_p_chk, POST, atol=1e-6)
# Check of the score definition: the registration's wording ("the mean of ... CRP, procalcitonin, ALT, AST, LDH and D-dimer, minus albumin
# and platelets") read literally as the mean of the six minus albumin minus platelets.
_six = XS_ALL[['hs_crp', 'procalcitonin', 'alt', 'ast', 'ldh', 'd_dimer']]
_alt = (_six.mean(axis=1) - XS_ALL['albumin'] - XS_ALL['platelets']).values
_ok2 = ~np.isnan(_alt)
_cs2 = np.array([np.sum(POST0[_ok2, c] * _alt[_ok2]) / np.sum(POST0[_ok2, c]) for c in range(K_STAR)])
_ord2 = list(np.argsort(-_cs2))
log('4 Describe', 'Score read as mean of six minus albumin minus platelets: same hyperinflammatory / least abnormal class; H1 effect (SD)',
    f"{_ord2[0] == ORDER[0]} / {_ord2[-1] == ORDER[-1]}; {(_cs2[_ord2[0]] - np.nanmean(_alt)) / np.nanstd(_alt, ddof=1):.2f}")
detail('Score: albumin and platelets enter the mean with a negative sign (8 items); the literal alternative reading is logged as a check.')
SIZES = np.bincount(MODAL, minlength=K_STAR)
log('4 Describe', 'Class sizes (modal assignment)', dict(zip(CLS, SIZES.tolist())))
log('4 Describe', 'Class mean scores (cohort mean, SD)', f"{dict(zip(CLS, np.round(CLASS_SCORE, 3).tolist()))} ({SC_MEAN:.3f}, {SC_SD:.3f})")
log('4 Describe', 'H1', f"hyperinflammatory class {CLS[HYPER]} score {H1_EFFECT:.2f} SD above the cohort mean -> {'supported' if H1 else 'not supported'}")

# Class summary and profiles
CLASS_TAB = pd.DataFrame({'class': CLS, 'n_modal': SIZES, 'pct_modal': np.round(100 * SIZES / N, 1),
                          'estimated_share_pct': np.round(100 * MAIN_WEIGHTS, 1),
                          'mean_posterior_assigned': [round(float(POST[MODAL == c, c].mean()), 3) for c in range(K_STAR)],
                          'pct_posterior_ge_0_8': [round(100 * float((POST[MODAL == c, c] >= 0.8).mean()), 1) for c in range(K_STAR)],
                          'mean_score': np.round(CLASS_SCORE, 3)})
CLASS_TAB['label'] = [('hyperinflammatory' if c == HYPER else ('least abnormal' if c == LEAST else '')) for c in range(K_STAR)]
PROFILE = pd.DataFrame(MAIN_MEANS.T, index=[LABEL[c] for c in FINAL], columns=CLS)
PROFILE.insert(0, 'domain', [DOMAIN[c] for c in FINAL])
_obs = XC.assign(_c=[CLS[c] for c in MODAL]).groupby('_c').mean().T.reindex(columns=CLS)
_obs.index = [LABEL[c] for c in _obs.index]
PROFILE_OBS = _obs


def med_iqr(s):
    s = pd.Series(s).dropna()
    return f'{s.median():.3g} ({s.quantile(.25):.3g}-{s.quantile(.75):.3g})' if len(s) else ''


_raw = pd.concat([LABVAL, VITALS_RAW[['temp_c', 'heart_rate', 'resp_rate']]], axis=1).loc[CLU]
_raw['class'] = [CLS[c] for c in MODAL]
RAW_BY_CLASS = pd.DataFrame({cl: {c: med_iqr(_raw.loc[_raw['class'] == cl, c]) for c in LAB_IND + ['temp_c', 'heart_rate', 'resp_rate']} for cl in CLS})
RAW_BY_CLASS.insert(0, 'all', [med_iqr(_raw[c]) for c in LAB_IND + ['temp_c', 'heart_rate', 'resp_rate']])
RAW_BY_CLASS.index = [LABEL.get(c, {'heart_rate': 'Heart rate (beats/min)', 'resp_rate': 'Respiratory rate (breaths/min)'}.get(c, c)) for c in RAW_BY_CLASS.index]
CLASS_TAB

## Step 5. Stability
200 bootstrap resamples of the clustered children; in each, the full choice of the number of classes is repeated (1-6 classes,
100 random starts each). Reported: how often the main number of classes is chosen, and the adjusted Rand index between the
resample model's assignment of the original children and the main assignment.

In [ ]:
detail('Stability refits use the fast EM (StepMix algorithm, checked above) with 100 random starts per class number and no 500-start '
       'fallback; resample b uses RandomState(SEED + 7919 b) for the resample and SEED + 7919 b + 1 for the starts.')
t = time.time()
STAB = parallel_map(T.stability_worker, [(X, MODAL, SEED + 7919 * b, K_MAX, N_STARTS, MIN_CLASS_SHARE) for b in range(1, N_BOOT_STABILITY + 1)])
STAB_K = np.array([r['k'] for r in STAB])
STAB_ARI = np.array([r['ari'] for r in STAB])
STAB_TAB = pd.DataFrame({'classes_chosen': range(1, K_MAX + 1), 'resamples': [int((STAB_K == k).sum()) for k in range(1, K_MAX + 1)]})
STAB_TAB['pct'] = (100 * STAB_TAB.resamples / len(STAB)).round(1)
log('5 Stability', f'Resamples choosing {K_STAR} classes', f"{int((STAB_K == K_STAR).sum())} of {len(STAB)} ({100 * (STAB_K == K_STAR).mean():.1f}%)")
log('5 Stability', 'Adjusted Rand index vs the main assignment, median (IQR)',
    f'{np.median(STAB_ARI):.2f} ({np.percentile(STAB_ARI, 25):.2f}-{np.percentile(STAB_ARI, 75):.2f}); {time.time() - t:.0f} s')
STAB_TAB

## Step 6. Reproduction by period (H3)
Independent fits in 2019-2022 and 2023-2026 (same procedure). H3 is met if both periods choose the same number of classes and the
two period models agree on the 2023-2026 children with Cohen's kappa of at least 0.6, after matching classes by the correlation of
their mean profiles.

In [ ]:
detail('Period fits of the main analysis (H3) use StepMix with the registered fit and choice rules (100 starts, 500 when fewer than two '
       'reach the best log-likelihood); the period fits within sensitivity analyses use the fast EM with the same rules.')
detail('Period fits use the transformed matrix of the whole clustering population (log, winsorizing and standardization over all 695); '
       'the diagonal Gaussian model is invariant to the per-indicator scaling, so this only fixes the winsorizing limits.')
detail('H3 kappa: the 2019-2022 model assigns the 2023-2026 children (most likely class); its classes are matched to those of the '
       '2023-2026 model by the Hungarian assignment on the Pearson correlation of class mean profiles; Cohen\'s kappa, unweighted.')
from sklearn.metrics import cohen_kappa_score as kappa_score


def stepmix_select(Xm, seed=SEED):
    """The registered fit and choice with StepMix itself (as in steps 2 and 3): 1-6 classes, 100 starts, 500 when the best
    log-likelihood is reached by fewer than two starts; lowest BIC among solutions meeting the 5% rule."""
    n, d = Xm.shape
    res = {}
    for k in range(1, K_MAX + 1):
        starts = N_STARTS
        while True:
            mdl = StepMix(n_components=k, measurement='gaussian_diag_nan', n_init=starts, random_state=seed, progress_bar=0, verbose=0)
            mdl.fit(Xm)
            reached = int((np.abs(np.array(mdl.lower_bound_buffer_) - mdl.lower_bound_) < 1e-6).sum())
            if reached >= 2 or k == 1 or starts >= N_STARTS_FALLBACK:
                break
            starts = N_STARTS_FALLBACK
        par = mdl.get_parameters()
        params = (np.asarray(par['weights']), np.asarray(par['measurement']['means']), np.asarray(par['measurement']['covariances']))
        post = mdl.predict_proba(Xm)
        sizes = np.bincount(post.argmax(1), minlength=k)
        res[k] = dict(params=params, post=post, sizes=sizes, min_share=sizes.min() / n, reached=reached, starts=starts,
                      **T.fit_stats(mdl.lower_bound_, n, T.n_params_diag(k, d)))
    ok = [k for k in res if res[k]['min_share'] >= MIN_CLASS_SHARE]
    return (min(ok, key=lambda k: res[k]['bic']) if ok else 1), res


def period_fits(Xm, early_mask, seed=SEED, n_init=None, engine='stepmix'):
    if engine == 'stepmix':
        k_e, res_e = stepmix_select(Xm[early_mask], seed)
        k_l, res_l = stepmix_select(Xm[~early_mask], seed)
    else:
        k_e, res_e = T.select(Xm[early_mask], K_MAX, n_init or N_STARTS, seed, MIN_CLASS_SHARE, fallback=N_STARTS_FALLBACK)
        k_l, res_l = T.select(Xm[~early_mask], K_MAX, n_init or N_STARTS, seed, MIN_CLASS_SHARE, fallback=N_STARTS_FALLBACK)
    out = {'k_early': k_e, 'k_late': k_l, 'same_k': k_e == k_l, 'kappa': np.nan, 'met': False}
    if k_e == k_l and k_e > 1:
        pe, pl = res_e[k_e]['params'], res_l[k_l]['params']
        mapping, r = T.match_by_profile(pe[1], pl[1])            # late class -> early class
        pred_e = T.posterior(Xm[~early_mask], pe)[0].argmax(1)
        lab_l = np.array([mapping[c] for c in T.posterior(Xm[~early_mask], pl)[0].argmax(1)])
        out['kappa'] = float(kappa_score(pred_e, lab_l))
        out['matched_profile_r'] = [round(float(r[mapping[c], c]), 3) for c in range(k_l)]
        out['met'] = out['kappa'] >= 0.6
    out['n_early'], out['n_late'] = int(early_mask.sum()), int((~early_mask).sum())
    out['bic_early'] = {k: round(float(res_e[k]['bic']), 1) for k in res_e}
    out['bic_late'] = {k: round(float(res_l[k]['bic']), 1) for k in res_l}
    out['min_share_early'] = {k: round(float(res_e[k]['min_share']), 3) for k in res_e}
    out['min_share_late'] = {k: round(float(res_l[k]['min_share']), 3) for k in res_l}
    out['starts_early'] = {k: f"{res_e[k]['reached']} of {res_e[k].get('starts', n_init or N_STARTS)}" for k in res_e}
    out['starts_late'] = {k: f"{res_l[k]['reached']} of {res_l[k].get('starts', n_init or N_STARTS)}" for k in res_l}
    out['res_early'], out['res_late'] = res_e, res_l
    return out


EARLY = (D.loc[CLU, 'period'] == '2019-2022').values
t = time.time()
H3 = period_fits(X, EARLY)
for _nm, _res in [('2019-2022', H3['res_early']), ('2023-2026', H3['res_late'])]:
    _bad = [k for k in _res if k > 1 and _res[k]['reached'] < 2]
    if _bad:
        deviation(f'H3 period fit {_nm}: for {", ".join(map(str, _bad))} classes the best log-likelihood was reached by fewer than two of '
                  f'{N_STARTS_FALLBACK} starts (registered rule not met); chosen: {H3["k_early" if _nm == "2019-2022" else "k_late"]} classes.')
log('6 Reproduce', 'Classes chosen: 2019-2022 / 2023-2026', f"{H3['k_early']} / {H3['k_late']} (n = {H3['n_early']} / {H3['n_late']})")
log('6 Reproduce', 'BIC by number of classes, 2019-2022 / 2023-2026', f"{H3['bic_early']} / {H3['bic_late']}")
log('6 Reproduce', 'Smallest class share, 2019-2022 / 2023-2026', f"{H3['min_share_early']} / {H3['min_share_late']}")
log('6 Reproduce', 'Starts reaching the best log-likelihood, 2019-2022 / 2023-2026', f"{H3['starts_early']} / {H3['starts_late']}")
log('6 Reproduce', 'H3', f"kappa {H3['kappa']:.2f} -> {'met' if H3['met'] else 'not met'}; {time.time() - t:.0f} s")
# Agreement of each period model with the main model on that period's children (descriptive)
for name, mask, res, k in [('2019-2022', EARLY, H3['res_early'], H3['k_early']), ('2023-2026', ~EARLY, H3['res_late'], H3['k_late'])]:
    if k == K_STAR:
        mp, _ = T.match_by_profile(MAIN_MEANS, res[k]['params'][1])
        lab = np.array([mp[c] for c in T.posterior(X[mask], res[k]['params'])[0].argmax(1)])
        log('6 Reproduce', f'Agreement of the {name} model with the main model on its children (kappa)', round(float(kappa_score(MODAL[mask], lab)), 3))

## Step 7. Validation (H2 and secondary validators)
Bias-corrected three-step (BCH) estimates carry the classification uncertainty. Primary (H2): hyperinflammatory versus least
abnormal class, ratio of geometric means of IL-6 and IL-10 (first value from 24 h before to 48 h after admission) and risk ratio
of death or critical discharge against medical advice (adjudicated label); 95% CIs from 500 bootstrap resamples; the three tests
are corrected by the Holm method at a family-wise two-sided alpha of 0.05. Secondary validators: estimates with 95% CIs, no
significance claims.

In [ ]:
detail('H2 bootstrap: each resample refits the measurement model with the chosen number of classes (warm start from the main solution plus '
       '19 random starts, best log-likelihood), matches its classes to the main model by the correlation of class mean profiles (Hungarian '
       'assignment; registered rule for label switching), and computes BCH estimates on the resample. CIs: percentiles 2.5 and 97.5.')
change('H2 bootstrap: classes are matched to the main model by profile correlation, the registered rule; the development run used '
       'StepMix\'s rule (permutation of posterior probabilities). Changed after the independent check.')
detail('Binary distal outcomes: the BCH-weighted proportion of a class can fall below 0 because BCH weights can be negative; as in StepMix '
       'it is clipped to [1e-15, 1 - 1e-15] (shown as 0.0%). A risk ratio whose percentile bound exceeds 10^6 (a replicate estimating no '
       'events in a class) is shown as unbounded.')
detail('P values for the three H2 contrasts only (all other contrasts are estimates with CIs): two-sided percentile-bootstrap P values, '
       '2 x the smaller of (1 + replicates with log ratio <= 0) / (B + 1) and (1 + replicates with log ratio >= 0) / (B + 1), so that '
       'P < 0.05 exactly when the 95% percentile CI excludes 1.')
change('H2 P values: percentile-bootstrap P values replace the Wald tests with the bootstrap standard error that the development run used '
       '(the registration does not fix the method); a Wald standard error is undefined or inflated when a replicate estimates no events in '
       'a class, which the development run showed for a class without events.')
detail('Distal outcomes on the log scale: IL-6, IL-10, TNF-alpha, ferritin, CSF protein (log of the value), CSF white cells (log of count + 1) and '
       'length of stay (log hours); reported as geometric means (count + 1 back-transformed for CSF white cells).')
detail('Consciousness for H4 and the validators: four levels as registered, alert; altered (drowsy, confused, delirium or irritable); stupor; '
       'coma (light, moderate, unspecified or deep); sedated or unrecorded = missing.')
change('Consciousness: irritable is counted as altered, as registered; the development run counted it as alert (3 children). Corrected '
       'after the independent check.')
CONS4 = {'alert': 'alert', 'irritable': 'altered', 'drowsy': 'altered', 'confused': 'altered', 'delirium': 'altered', 'stupor': 'stupor',
         'light_coma': 'coma', 'moderate_coma': 'coma', 'coma_unspecified': 'coma', 'deep_coma': 'coma'}
D['cons4'] = D.consciousness.map(CONS4)
DC = D.loc[CLU]


def b01(s):
    return pd.Series(s, dtype='object').map(lambda v: np.nan if v is None or v is pd.NA or (isinstance(v, float) and np.isnan(v)) else float(bool(v))).astype(float).values


def outcome_arrays(Dc, poor_col='death_or_dama_critical_final'):
    Y = {'IL-6': Dc.log_il6.values, 'IL-10': Dc.log_il10.values, 'Death or critical discharge': b01(Dc[poor_col]),
         'TNF-alpha': Dc.log_tnf_alpha.values, 'Ferritin': Dc.log_ferritin.values, 'CSF protein': Dc.log_csf_protein.values,
         'CSF white cells': np.log(Dc.csf_wbc.values + 1), 'Death': b01(Dc.death), 'Critical discharge against advice': b01(Dc.dama_critical_final),
         'Invasive ventilation': b01(Dc.imv_ever), 'Vasoactive infusion': b01(Dc.vasoactive_ever), 'PICU stay': b01(Dc.any_picu_stay),
         'Length of stay': np.log(Dc.los_hours.values), 'Status epilepticus': b01(Dc.status_epilepticus),
         'Abnormal pupillary light reflex': np.where(Dc.pupil_light_reflex.isna(), np.nan, Dc.pupil_light_reflex.isin(['sluggish', 'absent']).astype(float)),
         'First EEG: diffuse slowing or worse': np.where(Dc.eeg_first_grade.isna(), np.nan, (Dc.eeg_first_grade.astype(float) >= 2).astype(float)),
         'First EEG: low voltage or suppressed': np.where(Dc.eeg_first_grade.isna(), np.nan, (Dc.eeg_first_grade.astype(float) >= 3).astype(float))}
    for lev in ['alert', 'altered', 'stupor', 'coma']:
        Y[f'Consciousness: {lev}'] = np.where(Dc.cons4.isna(), np.nan, (Dc.cons4 == lev).astype(float))
    for tier in ['Definite ITES', 'ITES-related', 'Possible ITES', 'Sepsis-associated encephalopathy', 'Unclassifiable', 'Non-ITES']:
        Y[f'ITES tier: {tier}'] = (Dc.ites_tier == tier).astype(float).values
    return {k: np.asarray(v, float) for k, v in Y.items()}


CONT = {'IL-6', 'IL-10', 'TNF-alpha', 'Ferritin', 'CSF protein', 'CSF white cells', 'Length of stay'}
PRIMARY = ['IL-6', 'IL-10', 'Death or critical discharge']
Y_MAIN = outcome_arrays(DC)
BINARY = [k for k in Y_MAIN if k not in CONT]
assert not np.isnan(Y_MAIN['Death or critical discharge']).any(), 'The primary outcome must be known for every clustered child.'

# Point estimates (own BCH implementation) and the check against StepMix's three-step BCH estimator for the primary outcomes.
W_MAIN = T.bch_weights(POST)
EST = {k: T.bch_means(W_MAIN, v, binary=k in BINARY) for k, v in Y_MAIN.items()}
_Yp = np.column_stack([Y_MAIN[k] for k in PRIMARY])
_sm3 = StepMix(n_components=K_STAR, measurement='gaussian_diag_nan', n_steps=3, correction='BCH', n_init=MAIN.n_init, random_state=SEED,
               structural={'il6': {'model': 'gaussian_diag_nan', 'n_columns': 1}, 'il10': {'model': 'gaussian_diag_nan', 'n_columns': 1},
                           'poor': {'model': 'binary', 'n_columns': 1}}, progress_bar=0, verbose=0).fit(X, _Yp)
_perm = T.best_permutation(POST, _sm3.predict_proba(X))
_sp = _sm3.get_parameters()['structural']
_sm_est = {'IL-6': np.ravel(_sp['il6']['means'])[_perm], 'IL-10': np.ravel(_sp['il10']['means'])[_perm], 'Death or critical discharge': np.ravel(_sp['poor']['pis'])[_perm]}
BCH_EQUAL = all(np.allclose(_sm_est[k], EST[k], atol=1e-6) for k in PRIMARY)
log('7 Validate', 'Own BCH estimates equal StepMix three-step BCH (primary outcomes)', BCH_EQUAL)
if not BCH_EQUAL:
    raise RuntimeError('BCH estimates differ from StepMix.')
log('7 Validate', 'BCH classification matrix D: condition number (BCH fails only if D cannot be inverted)',
    f"{np.linalg.cond((POST.T @ np.eye(K_STAR)[MODAL] / POST.mean(axis=0).reshape(-1, 1)) / N):.2f}")
detail('Q23 (BCH fails -> ML correction): BCH is considered failed only if the classification matrix D cannot be inverted. A class proportion '
       'below 0 is clipped at 0 (StepMix) and does not count as failure. As a supplementary check, the primary estimates are also computed '
       'with StepMix\'s ML correction (point estimates).')
_ml = StepMix(n_components=K_STAR, measurement='gaussian_diag_nan', n_steps=3, correction='ML', n_init=MAIN.n_init, random_state=SEED,
              structural={'il6': {'model': 'gaussian_diag_nan', 'n_columns': 1}, 'il10': {'model': 'gaussian_diag_nan', 'n_columns': 1},
                          'poor': {'model': 'binary', 'n_columns': 1}}, progress_bar=0, verbose=0).fit(X, _Yp)
_permml = T.best_permutation(POST, _ml.predict_proba(X))
_spm = _ml.get_parameters()['structural']
ML_EST = {'IL-6': np.ravel(_spm['il6']['means'])[_permml], 'IL-10': np.ravel(_spm['il10']['means'])[_permml],
          'Death or critical discharge': np.ravel(_spm['poor']['pis'])[_permml]}
log('7 Validate', 'Supplementary: ML-corrected three-step estimates, hyperinflammatory vs least abnormal',
    {'IL-6 ratio of geometric means': round(float(np.exp(ML_EST['IL-6'][HYPER] - ML_EST['IL-6'][LEAST])), 2),
     'IL-10 ratio of geometric means': round(float(np.exp(ML_EST['IL-10'][HYPER] - ML_EST['IL-10'][LEAST])), 2),
     'risk ratio': round(float(ML_EST['Death or critical discharge'][HYPER] / ML_EST['Death or critical discharge'][LEAST]), 2),
     'risks by class (%)': np.round(100 * ML_EST['Death or critical discharge'], 1).tolist()})
_e10 = T.bch_means(W_MAIN, DC['log_il10_zero_missing'].values)
log('7 Validate', 'Check: IL-10 ratio of geometric means with results of 0 set to missing',
    f"{np.exp(_e10[HYPER] - _e10[LEAST]):.2f} (main: {np.exp(EST['IL-10'][HYPER] - EST['IL-10'][LEAST]):.2f})")

t = time.time()
BOOT = parallel_map(T.h2_worker, [(X, Y_MAIN, POST, MAIN_TUPLE, K_STAR, SEED + 104729 * b, N_RANDOM_BOOT, BINARY) for b in range(1, N_BOOT_H2 + 1)])
log('7 Validate', 'Bootstrap replicates', f"{len(BOOT)} in {time.time() - t:.0f} s; failed starts in all refits: {sum(b['_n_failed'] for b in BOOT)}")


def boot_p(db):
    """Two-sided percentile-bootstrap P value for a log ratio (null value 0), with the +1 correction."""
    db = np.asarray(db, float)
    db = db[~np.isnan(db)]
    if not len(db):
        return np.nan
    return float(min(1.0, 2 * min((np.sum(db <= 0) + 1) / (len(db) + 1), (np.sum(db >= 0) + 1) / (len(db) + 1))))


def ratio_ci(db):
    lo, hi = np.exp(np.percentile(db, [2.5, 97.5])) if len(db) else (np.nan, np.nan)
    return (float(lo) if lo >= 1e-6 else 0.0), (float(hi) if hi <= 1e6 else np.inf)


def summarize_validation(est, boot, Y, modal, cls_names, hyper, least):
    rows, contr = [], []
    B = {k: np.array([b[k] for b in boot]) for k in est}
    for k, e in est.items():
        cont = k in CONT
        tr = (lambda v: np.exp(v) - 1) if k == 'CSF white cells' else (np.exp if cont else (lambda v: 0.0 if v < 1e-9 else 100 * v))
        lo, hi = np.nanpercentile(B[k], 2.5, axis=0), np.nanpercentile(B[k], 97.5, axis=0)
        n_obs = [int((~np.isnan(Y[k][modal == c])).sum()) for c in range(len(cls_names))]
        rows.append({'validator': k, 'scale': 'geometric mean' if cont else '%', **{f'{c}': f'{tr(e[i]):.3g} ({tr(lo[i]):.3g}-{tr(hi[i]):.3g})' for i, c in enumerate(cls_names)},
                     **{f'n_{c}': n_obs[i] for i, c in enumerate(cls_names)}})
        if len(cls_names) > 1:
            if cont:
                d = e[hyper] - e[least]; db = B[k][:, hyper] - B[k][:, least]
                eff, name = np.exp(d), 'ratio of geometric means'
            else:
                with np.errstate(divide='ignore', invalid='ignore'):
                    d = np.log(e[hyper] / e[least]); db = np.log(B[k][:, hyper] / B[k][:, least])
                eff, name = np.exp(d), 'risk ratio'
            db = db[~np.isnan(db)]
            lo_, hi_ = ratio_ci(db)
            contr.append({'validator': k, 'contrast': f'{cls_names[hyper]} vs {cls_names[least]}', 'measure': name, 'estimate': float(eff) if eff <= 1e6 else np.inf,
                          'ci_low': lo_, 'ci_high': hi_, 'p_value': boot_p(db) if k in PRIMARY else np.nan,
                          'bootstrap_replicates_used': int(len(db))})
    return pd.DataFrame(rows), pd.DataFrame(contr)


VAL_TAB, CONTRAST = summarize_validation(EST, BOOT, Y_MAIN, MODAL, CLS, HYPER, LEAST)
if K_STAR > 1:
    prim = CONTRAST[CONTRAST.validator.isin(PRIMARY)].copy()
    order = np.argsort(prim.p_value.values)
    holm = np.empty(len(prim)); running = 0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(prim) - rank) * prim.p_value.values[i]))
        holm[i] = running
    prim['p_holm'] = holm
    prim['significant_holm_0.05'] = prim.p_holm < 0.05
    H2_TAB = prim
    log('7 Validate', 'H2 (Holm-adjusted)', prim[['validator', 'measure', 'estimate', 'ci_low', 'ci_high', 'p_value', 'p_holm']].round(4).to_dict('records'))
else:
    H2_TAB = pd.DataFrame()
    log('7 Validate', 'H2', 'not tested: one class')

# Descriptive: ITES crosswalk, treatments and IFN-gamma by modal class.
_mc = pd.Series([CLS[c] for c in MODAL], index=CLU, name='class')
CROSS_ITES = pd.crosstab(DC.ites_class, _mc).reindex(columns=CLS)
CROSS_ITES['all'] = CROSS_ITES.sum(axis=1)
CROSS_TIER = pd.crosstab(DC.ites_tier, _mc).reindex(columns=CLS)
TREAT = pd.DataFrame({c: [f"{int(DC.loc[_mc == c, t].sum())} ({100 * DC.loc[_mc == c, t].mean():.1f}%)" for t in ['methylprednisolone', 'ivig', 'tocilizumab']]
                      for c in CLS}, index=['Methylprednisolone', 'IVIG', 'Tocilizumab'])
IFN = pd.DataFrame({c: [med_iqr(DC.loc[_mc == c, 'ifn_gamma']), int(DC.loc[_mc == c, 'ifn_gamma'].notna().sum())] for c in CLS}, index=['IFN-gamma median (IQR), pg/mL', 'measured'])
H2_TAB

## Step 8. Beyond severity (H4)
Logistic regression for death or critical discharge with age (restricted cubic spline, 3 knots) and consciousness at admission in
four levels; then subphenotype (most likely class) is added. Likelihood-ratio test (two-sided alpha 0.05) and the change in
C-statistic with a 95% CI from 200 bootstrap resamples (both models refitted in each resample).

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
detail('Age spline: restricted cubic spline with knots at the 10th, 50th and 90th percentiles of age in the analysed children (Harrell); '
       'reference categories: alert consciousness and the least abnormal class.')


def rcs3(x, knots):
    t1, t2, t3 = knots
    p = lambda u: np.clip(u, 0, None) ** 3
    return (p(x - t1) - p(x - t2) * (t3 - t1) / (t3 - t2) + p(x - t3) * (t2 - t1) / (t3 - t2)) / (t3 - t1) ** 2


def h4_design(Dc, labels, least, knots=None):
    age = Dc.age_years.astype(float).values
    knots = knots if knots is not None else np.percentile(age, [10, 50, 90])
    base = pd.DataFrame({'const': 1.0, 'age': age, 'age_rcs': rcs3(age, knots)}, index=Dc.index)
    for lev in ['altered', 'stupor', 'coma']:
        base[f'cons_{lev}'] = (Dc.cons4 == lev).astype(float)
    full = base.copy()
    for c in sorted(set(labels)):
        if c != least:
            full[f'class_{c}'] = (labels == c).astype(float)
    return base, full, knots


def logit_fit(y, Xd):
    """Maximum-likelihood logistic regression (Newton); returns the fit and whether it converged. A class without events (or with only
    events) has no finite odds ratio; the likelihood still reaches its maximum, so the likelihood-ratio test and C-statistic are defined."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m = sm.Logit(y, Xd).fit(disp=0, maxiter=200)
    return m, bool(m.mle_retvals.get('converged', False))


def h4(Dc, labels, least, poor_col='death_or_dama_critical_final', n_boot=0, seed=SEED):
    keep = Dc.cons4.notna().values
    Dk, lab = Dc[keep], np.asarray(labels)[keep]
    y = b01(Dk[poor_col])
    base, full, knots = h4_design(Dk, lab, least)
    try:
        m0, conv0 = logit_fit(y, base)
        m1, conv1 = logit_fit(y, full)
    except Exception as e:
        return {'error': str(e), 'n': int(keep.sum())}
    lr = 2 * (m1.llf - m0.llf)
    df_ = full.shape[1] - base.shape[1]
    from scipy.stats import chi2
    out = {'n': int(keep.sum()), 'events': int(y.sum()), 'lr_stat': float(lr), 'df': int(df_), 'p_value': float(chi2.sf(lr, df_)) if df_ > 0 else np.nan,
           'c_base': float(roc_auc_score(y, m0.predict(base))), 'c_full': float(roc_auc_score(y, m1.predict(full)))}
    out['delta_c'] = out['c_full'] - out['c_base']
    out['converged'] = conv0 and conv1
    ev = {str(c): (int(y[lab == c].sum()), int((lab == c).sum())) for c in sorted(set(lab))}
    out['events_by_class'] = {c: f'{e} of {n}' for c, (e, n) in ev.items()}
    ci = m1.conf_int()
    out['odds_ratios'] = {c: (f'{np.exp(m1.params[c]):.2f} ({np.exp(ci.loc[c, 0]):.2f}-{np.exp(ci.loc[c, 1]):.2f})'
                              if 0 < ev[c[6:]][0] < ev[c[6:]][1] else 'not estimable (no events or only events in this class)')
                          for c in full.columns if c.startswith('class_')}
    if n_boot:
        rng = np.random.RandomState(seed)
        deltas, nonconv = [], 0
        for _ in range(n_boot):
            i = rng.randint(0, len(y), len(y))
            try:
                b0, c0 = logit_fit(y[i], base.iloc[i])
                b1, c1 = logit_fit(y[i], full.iloc[i])
                nonconv += int(not (c0 and c1))
                deltas.append(roc_auc_score(y[i], b1.predict(full.iloc[i])) - roc_auc_score(y[i], b0.predict(base.iloc[i])))
            except Exception:
                continue
        out['delta_c_ci'] = (float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)))
        out['boot_used'] = len(deltas)
        out['boot_not_converged'] = nonconv
    return out


def h4_parametric_p(Dc, labels, least, poor_col='death_or_dama_critical_final', n_sim=1000, seed=SEED):
    """Parametric-bootstrap P value for the H4 likelihood-ratio statistic: outcomes simulated from the fitted age-and-consciousness
    model, both models refitted, P = (1 + simulated LR >= observed LR) / (1 + simulations). Valid when a class has no events."""
    keep = Dc.cons4.notna().values
    Dk, lab = Dc[keep], np.asarray(labels)[keep]
    y = b01(Dk[poor_col])
    base, full, _ = h4_design(Dk, lab, least)
    m0, _ = logit_fit(y, base)
    m1, _ = logit_fit(y, full)
    lr_obs = 2 * (m1.llf - m0.llf)
    p0 = np.asarray(m0.predict(base))
    rng = np.random.RandomState(seed)
    lrs = []
    for _ in range(n_sim):
        ys = (rng.uniform(size=len(p0)) < p0).astype(float)
        try:
            a, _ = logit_fit(ys, base)
            b, _ = logit_fit(ys, full)
            lrs.append(2 * (b.llf - a.llf))
        except Exception:
            continue
    lrs = np.array(lrs)
    return {'lr': float(lr_obs), 'p_parametric_bootstrap': float((1 + np.sum(lrs >= lr_obs)) / (1 + len(lrs))), 'simulations': int(len(lrs)),
            'simulated_lr_95th_percentile': float(np.percentile(lrs, 95)) if len(lrs) else np.nan}


LAB_NAMES = np.array([CLS[c] for c in MODAL])
H4 = h4(DC, LAB_NAMES, CLS[LEAST], n_boot=N_BOOT_H4) if K_STAR > 1 else {}
if H4 and 'error' not in H4:
    log('8 Beyond severity', 'H4', f"n {H4['n']} ({H4['events']} events); LR {H4['lr_stat']:.2f} on {H4['df']} df, P = {H4['p_value']:.4f}; "
        f"C {H4['c_base']:.3f} -> {H4['c_full']:.3f}, change {H4['delta_c']:.3f} (95% CI {H4['delta_c_ci'][0]:.3f} to {H4['delta_c_ci'][1]:.3f}); "
        f"models converged: {H4['converged']}; events by class {H4['events_by_class']}; bootstrap fits not converged {H4['boot_not_converged']} of {H4['boot_used']}")
elif H4:
    deviation(f"H4 models could not be fitted: {H4['error']}")
if H4 and 'error' not in H4 and not H4['converged']:
    H4_PB = h4_parametric_p(DC, LAB_NAMES, CLS[LEAST])
    H4['p_parametric_bootstrap'] = H4_PB['p_parametric_bootstrap']
    log('8 Beyond severity', 'Supplementary: parametric-bootstrap P for the LR statistic (a class without events makes its odds ratio infinite)', H4_PB)
change('H4: when a class has no events (quasi-complete separation, seen in the development run), a parametric-bootstrap P value for the '
       'likelihood-ratio statistic (1,000 outcome vectors simulated from the age-and-consciousness model) is reported next to the registered '
       'chi-square P value, and the odds ratio of that class is reported as not estimable.')

## Step 9. Simple classifier (exploratory)
L1-penalized multinomial logistic regression with at most 5 indicators, trained on the main model's labels of the 2019-2022
children and tested on the 2023-2026 children (accuracy and one-versus-rest AUC per class).

In [ ]:
from sklearn.linear_model import LogisticRegression
detail('Classifier: standardized clustering indicators, missing values replaced by the training medians; the penalty is the weakest on a '
       'grid of 60 values (C from 10^-3 to 10^1) whose model uses at most 5 indicators (any non-zero coefficient).')
change('Classifier: besides the registered L1 model, an unpenalized multinomial model refitted on the same indicators is reported as a '
       'second exploratory variant, because the L1 model at this penalty shrinks the predicted probabilities toward the class shares '
       '(seen in the development run).')
_SK = tuple(int(x) for x in sklearn.__version__.split('.')[:2])


def logreg_l1(C_):
    """L1-penalized multinomial logistic regression (scikit-learn 1.8 replaced penalty='l1' by l1_ratio=1)."""
    kw = {'l1_ratio': 1.0} if _SK >= (1, 8) else {'penalty': 'l1'}
    return LogisticRegression(C=C_, solver='saga', max_iter=20000, random_state=SEED, **kw)


def logreg_unpenalized():
    kw = {'C': np.inf} if _SK >= (1, 8) else {'penalty': None}
    return LogisticRegression(solver='lbfgs', max_iter=20000, **kw)


CLF = {}
if K_STAR > 1:
    Xtr, Xte = XC[EARLY].copy(), XC[~EARLY].copy()
    med = Xtr.median()
    Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
    ytr, yte = MODAL[EARLY], MODAL[~EARLY]
    best = None
    for C_ in np.logspace(-3, 1, 60):
        clf = logreg_l1(C_).fit(Xtr, ytr)
        used = [FINAL[j] for j in np.where(np.abs(clf.coef_).max(axis=0) > 1e-8)[0]]
        if len(used) <= 5:
            best = (C_, clf, used)
    def clf_eval(model, cols):
        pred = model.predict(Xte[cols])
        prob = model.predict_proba(Xte[cols])
        aucs = {CLS[c]: (round(float(roc_auc_score((yte == c).astype(int), prob[:, list(model.classes_).index(c)])), 3)
                         if c in model.classes_ and 0 < (yte == c).sum() < len(yte) else np.nan) for c in range(K_STAR)}
        return {'accuracy': round(float((pred == yte).mean()), 3), 'auc_by_class': aucs, 'classes_predicted_in_test': sorted({CLS[c] for c in pred})}

    if best:
        C_, clf, used = best
        CLF = {'C': float(C_), 'indicators': [LABEL[c] for c in used], 'n_train': int(len(ytr)), 'n_test': int(len(yte)),
               'l1_model': clf_eval(clf, list(Xtr.columns))}
        refit = logreg_unpenalized().fit(Xtr[used], ytr)
        CLF['refitted_unpenalized'] = clf_eval(refit, used)
        log('9 Classifier', 'Simple classifier', CLF)

## Step 10. Sensitivity analyses
Each repeats the main analysis with one change (registered list). Agreement with the main solution is reported as Cohen's kappa
when the same number of classes is chosen (classes matched by the correlation of their mean profiles), otherwise as the closest
main class of each sensitivity class. Primary H2 estimates carry 95% CIs from 500 bootstrap resamples, as in the main analysis.

In [ ]:
detail('Sensitivity analyses recompute the whole preprocessing (log scale, winsorizing, standardization) within the children they analyse.')
detail('H2 bootstraps of the sensitivity analyses: warm start from the analysis solution plus 4 random starts per refit (19 in the main analysis).')
D['poor_unsure_not_critical'] = D.death.fillna(False).astype(bool) | (D.dama_critical_final.fillna(False).astype(bool)
                                                                     & ~D.dama_critical_final_source.eq('rule (clinician unsure)'))
DC['poor_unsure_not_critical'] = D.loc[CLU, 'poor_unsure_not_critical']
deviation('Refits in resampling do not apply the registered rule that the best log-likelihood be reached by at least two starts (500 '
          'starts otherwise): stability refits use 100 random starts; H2 bootstrap refits use a warm start from the main solution plus 19 '
          'random starts (4 in sensitivity analyses). Applying the rule in every refit would multiply the run time several-fold. Warm '
          'starts may make the bootstrap CIs somewhat narrower.')
deviation('Software: refits in resampling (stability, H2 bootstraps) and all latent profile fits within the sensitivity analyses use the EM of ae_lpa_tools, '
          'a NumPy implementation of StepMix 3.0.0\'s algorithm for gaussian_diag_nan, instead of StepMix itself (StepMix would take days). '
          'On the main data it reproduces StepMix\'s parameters after 1, 2, 5 and 20 iterations, its likelihood, and the final '
          'log-likelihood of every start (checks logged in step 2). The main fits, the period fits for H3 and all BCH checks use StepMix.')
deviation('Sensitivity analyses repeat the fit, the choice of the number of classes, the description (H1), the period refits (H3), the H2 '
          'contrasts with bootstrap CIs and H4, but not the 200-resample stability bootstrap, which is run for the main analysis only (the '
          'registration lists stability as a check of its own; the analysis plan says each sensitivity analysis repeats steps 1 to 6). One '
          'stability bootstrap takes about 75 minutes on 2 cores. Secondary validators are not repeated.')
_n_unsure = int((D.loc[CLU, 'dama_critical_final'].fillna(False).astype(bool) & D.loc[CLU, 'dama_critical_final_source'].eq('rule (clinician unsure)')
                 & ~D.loc[CLU, 'death'].fillna(False).astype(bool)).sum())
log('10 Sensitivity', 'Critical discharges resting on the rule because the reviewer was unsure (clustered children)', _n_unsure)


def score_order(post, rows, raw, winsorize=True):
    """Order of classes by the registered score (highest first), the ordered class scores and the H1 effect (SD units)."""
    XSv = transform(raw, rows, winsorize, CAND)
    items = pd.DataFrame({c: s * XSv[c] for c, s in SCORE_ITEMS.items()})
    sc = items.mean(axis=1).where(items.notna().sum(axis=1) >= 4).values
    ok = ~np.isnan(sc)
    k = post.shape[1]
    cs = np.array([np.sum(post[ok, c] * sc[ok]) / np.sum(post[ok, c]) for c in range(k)])
    order = list(np.argsort(-cs))
    eff = (cs[order[0]] - np.nanmean(sc)) / np.nanstd(sc, ddof=1) if k > 1 else np.nan
    return order, cs[order], eff


def agreement_with_main(rows, modal_v, means_v):
    """Kappa with the main assignment on the common children when the number of classes is the same (classes matched by profile
    correlation); otherwise the closest main class of each sensitivity class."""
    rows = pd.Index(rows)
    common = CLU.isin(rows)
    idx_v = pd.Series(np.arange(len(rows)), index=rows)[CLU[common]].values
    mapping, r = T.match_by_profile(MAIN_MEANS, means_v)
    kv = means_v.shape[0]
    # profile_r[i][j]: correlation of main class i (S1..Sk) with class j of this analysis; matched_main[j]: main class matched to class j
    if kv == K_STAR:
        lab = np.array([mapping[c] for c in modal_v[idx_v]])
        return {'kappa_vs_main': round(float(kappa_score(MODAL[common], lab)), 3), 'common_children': int(common.sum()),
                'matched_profile_r': [round(float(r[mapping[c], c]), 3) for c in range(kv)],
                'matched_main': [int(mapping[c]) for c in range(kv)], 'profile_r': np.round(r, 3).tolist()}
    return {'kappa_vs_main': np.nan, 'common_children': int(common.sum()),
            'closest_main_class': {f'class {j + 1}': f'{CLS[int(np.argmax(r[:, j]))]} (r = {r[:, j].max():.2f})' for j in range(kv)},
            'matched_main': [int(np.argmax(r[:, j])) for j in range(kv)], 'profile_r': np.round(r, 3).tolist()}


def h2_point_and_boot(Xv, post, params, Dv, poor_col, n_boot, seed):
    """Primary H2 contrasts (BCH) with bootstrap CIs, classes already in score order (first = hyperinflammatory)."""
    Y = outcome_arrays(Dv, poor_col)
    Y = {k: Y[k] for k in PRIMARY}
    k = post.shape[1]
    if k < 2:
        return pd.DataFrame()
    est = {kk: T.bch_means(T.bch_weights(post), v, binary=kk in BINARY) for kk, v in Y.items()}
    boot = parallel_map(T.h2_worker, [(Xv, Y, post, params, k, seed + 104729 * b, N_RANDOM_BOOT_SENS, BINARY) for b in range(1, n_boot + 1)])
    _, con = summarize_validation(est, boot, Y, post.argmax(1), [f'S{i + 1}' for i in range(k)], 0, k - 1)
    return con


def lpa_variant(name, raw, rows, columns=None, winsorize=True, poor_col='death_or_dama_critical_final', do_h3=True, n_boot=None):
    t = time.time()
    rows = pd.Index(rows)
    Xv = transform(raw, rows, winsorize, columns or FINAL).values
    k, res = T.select(Xv, K_MAX, N_STARTS, SEED, MIN_CLASS_SHARE, fallback=N_STARTS_FALLBACK)
    order, cs, eff = score_order(res[k]['post'], rows, raw, winsorize)
    w, mu, var = res[k]['params']
    params = (w[order], mu[order], var[order])
    post = res[k]['post'][:, order]
    modal = post.argmax(1)
    out = {'analysis': name, 'children': len(rows), 'indicators': Xv.shape[1], 'classes': k,
           'class_sizes': np.bincount(modal, minlength=k).tolist(), 'h1_effect_sd': round(float(eff), 2) if k > 1 else np.nan,
           'h1_supported': bool(k >= 2 and eff >= 0.5), 'bic_by_k': {kk: round(float(res[kk]['bic']), 1) for kk in res},
           'min_share_by_k': {kk: round(float(res[kk]['min_share']), 3) for kk in res}}
    out.update(agreement_with_main(rows, modal, params[1]))
    out['h2'] = h2_point_and_boot(Xv, post, params, D.loc[rows], poor_col, N_BOOT_SENS if n_boot is None else n_boot, SEED)
    if do_h3:
        hh = period_fits(Xv, (D.loc[rows, 'period'] == '2019-2022').values, engine='fast')
        out.update({'h3_k_early': hh['k_early'], 'h3_k_late': hh['k_late'], 'h3_kappa': hh['kappa'], 'h3_met': hh['met']})
    if k > 1:
        hh4 = h4(D.loc[rows], np.array([f'S{c + 1}' for c in modal]), f'S{k}', poor_col=poor_col)
        out.update({'h4_p': hh4.get('p_value', np.nan), 'h4_delta_c': hh4.get('delta_c', np.nan)})
    out['seconds'] = round(time.time() - t)
    log('10 Sensitivity', name, {kk: v for kk, v in out.items() if kk not in ('h2', 'bic_by_k', 'min_share_by_k', 'profile_r', 'matched_main')})
    return out


SENS = []
SENS.append(lpa_variant('Worst value in the window', raw_indicators(how='worst'), CLU))
SENS.append(lpa_variant('Window 12 h before to 24 h after admission, own admission only', raw_indicators(lo_h=-12, hi_h=24, own_only=True), CLU))
SENS.append(lpa_variant('Without winsorizing', RAW, CLU, winsorize=False))
SENS.append(lpa_variant('Leaving out children with a non-ITES class', RAW, CLU[(D.loc[CLU, 'ites_tier'] != 'Non-ITES').values]))
SENS.append(lpa_variant('Leaving out children transferred from another hospital', RAW, CLU[(D.loc[CLU, 'admission_source'] != 'transfer_other_hospital').values]))
SENS.append(lpa_variant('Vital signs from the admission examination', raw_indicators(exam_vitals=True), CLU))
_on = D.loc[CLU, 'onset_hours_acute']
SENS.append(lpa_variant('Onset of the acute illness 24 h or less before admission', RAW, CLU[(_on <= 24).values], do_h3=False))
SENS.append(lpa_variant('Onset of the acute illness more than 24 h before admission', RAW, CLU[(_on > 24).values], do_h3=False))
SENS.append(lpa_variant('Additional (not registered): white cell count kept instead of neutrophil count', RAW, CLU,
                        columns=['wbc' if c == 'neutrophils_abs' else c for c in FINAL]))
detail('Worst value: the highest value for tests read against the upper limit and for temperature, heart rate and respiratory rate; the lowest '
       'for tests read against the lower limit.')
detail('Onset strata: the period refits (H3) are not repeated within strata, which would split them into groups of about 90-250 children.')

# Outcome label with the reviewer-unsure critical discharges counted as not critical (clustering unchanged).
_con = h2_point_and_boot(X, POST, MAIN_TUPLE, DC, 'poor_unsure_not_critical', N_BOOT_SENS, SEED)
_h4u = h4(DC, LAB_NAMES, CLS[LEAST], poor_col='poor_unsure_not_critical') if K_STAR > 1 else {}
SENS.append({'analysis': 'Reviewer-unsure critical discharges counted as not critical', 'children': N, 'indicators': P, 'classes': K_STAR,
             'class_sizes': SIZES.tolist(), 'kappa_vs_main': 1.0, 'h1_effect_sd': round(float(H1_EFFECT), 2), 'h1_supported': H1,
             'h2': _con, 'h4_p': _h4u.get('p_value', np.nan), 'h4_delta_c': _h4u.get('delta_c', np.nan)})
_rr = _con[_con.validator == 'Death or critical discharge'] if len(_con) else pd.DataFrame()
log('10 Sensitivity', 'Reviewer-unsure discharges as not critical',
    {'events': int(D.loc[CLU, 'poor_unsure_not_critical'].sum()),
     'h2_risk_ratio': (f"{_rr.estimate.iloc[0]:.2f} ({_rr.ci_low.iloc[0]:.2f}-{_rr.ci_high.iloc[0]:.2f})" if len(_rr) else 'not computed'),
     'h4_p': _h4u.get('p_value'), 'h4_delta_c': _h4u.get('delta_c')})

### Sensitivity analyses with a different model
Gaussian mixture with full (correlated) covariance matrices, and consensus k-means after multiple imputation (Seymour 2019).

In [ ]:
N_STARTS_FULL = 20 if not QUICK else 2
FULL_CHUNK = 5                     # starts per parallel task (tasks of the slowest class numbers run first)
deviation(f'Full-covariance model: {N_STARTS_FULL} random starts per class number instead of 100, and no bootstrap CIs for H2 '
          '(one start takes about 5 s with 2-4 classes and about 50 s with 5-6 classes, which reach the limit of 1,000 EM iterations; '
          '100 starts and 500 bootstrap refits would take days). Missing values are handled by full-information likelihood as in the '
          'main model (EM with conditional expectations of the missing values).')
detail('Full-covariance model: EM stops when the average log-likelihood changes by less than 1e-8 or after 1,000 iterations; a ridge of 1e-6 '
       'is added to each covariance matrix; starts of class number k run in chunks of 5 with RandomState(SEED + 7 j) for chunk j.')


def fullcov_select(Xv, seed=SEED):
    tasks = []
    for k in range(K_MAX, 0, -1):
        n_k = 1 if k == 1 else N_STARTS_FULL
        for j, s0 in enumerate(range(0, n_k, FULL_CHUNK)):
            tasks.append((Xv, k, min(FULL_CHUNK, n_k - s0), seed + 7 * j))
    fits = parallel_map(T.fullcov_worker, tasks)
    n = Xv.shape[0]
    res, iters = {}, {}
    for f in fits:
        iters.setdefault(f['k'], []).extend(list(f.get('iterations', [])))
        if f['params'] is None:
            continue
        if f['k'] not in res or f['avg_ll'] > res[f['k']]['avg_ll']:
            sizes = np.bincount(f['post'].argmax(1), minlength=f['k'])
            st = T.fit_stats(f['avg_ll'], n, f['n_parameters'])
            res[f['k']] = dict(f, sizes=sizes, min_share=sizes.min() / n, **st)
    for k in res:
        res[k]['starts_at_iteration_limit'] = int(np.sum(np.array(iters[k]) >= 1000))
        res[k]['starts'] = len(iters[k])
    ok = [k for k in res if res[k]['min_share'] >= MIN_CLASS_SHARE]
    return (min(ok, key=lambda k: res[k]['bic']) if ok else 1), res


t = time.time()
k_f, res_f = fullcov_select(X)
order_f, cs_f, eff_f = score_order(res_f[k_f]['post'], CLU, RAW)
post_f = res_f[k_f]['post'][:, order_f]
means_f = res_f[k_f]['params'][1][order_f]
out = {'analysis': 'Gaussian model with full covariance', 'children': N, 'indicators': P, 'classes': k_f,
       'class_sizes': np.bincount(post_f.argmax(1), minlength=k_f).tolist(), 'h1_effect_sd': round(float(eff_f), 2) if k_f > 1 else np.nan,
       'h1_supported': bool(k_f >= 2 and eff_f >= 0.5), 'bic_by_k': {k: round(float(res_f[k]['bic']), 1) for k in sorted(res_f)},
       'starts_at_iteration_limit_by_k': {k: f"{res_f[k]['starts_at_iteration_limit']} of {res_f[k]['starts']}" for k in sorted(res_f)}}
out.update(agreement_with_main(CLU, post_f.argmax(1), means_f))
if k_f > 1:
    Yp = {k: outcome_arrays(DC)[k] for k in PRIMARY}
    est_f = {k: T.bch_means(T.bch_weights(post_f), v, binary=k in BINARY) for k, v in Yp.items()}
    out['h2'] = pd.DataFrame([{'validator': k, 'contrast': f'S1 vs S{k_f}',
                               'measure': 'ratio of geometric means' if k in CONT else 'risk ratio',
                               'estimate': float(np.exp(e[0] - e[-1])) if k in CONT else float(e[0] / e[-1])} for k, e in est_f.items()])
    hh4 = h4(DC, np.array([f'S{c + 1}' for c in post_f.argmax(1)]), f'S{k_f}')
    out.update({'h4_p': hh4.get('p_value', np.nan), 'h4_delta_c': hh4.get('delta_c', np.nan)})
    ke, re_ = fullcov_select(X[EARLY]); kl, rl_ = fullcov_select(X[~EARLY])
    out.update({'h3_k_early': ke, 'h3_k_late': kl})
    if ke == kl and ke > 1:
        mp, _ = T.match_by_profile(re_[ke]['params'][1], rl_[kl]['params'][1])
        pe = T.posterior_full(X[~EARLY], re_[ke]['params'])[0].argmax(1)
        pl = np.array([mp[c] for c in rl_[kl]['post'].argmax(1)])
        out['h3_kappa'] = float(kappa_score(pe, pl)); out['h3_met'] = out['h3_kappa'] >= 0.6
out['seconds'] = round(time.time() - t)
SENS.append(out)
log('10 Sensitivity', out['analysis'], {kk: v for kk, v in out.items() if kk not in ('h2', 'bic_by_k', 'profile_r', 'matched_main')})

In [ ]:
# Consensus k-means after multiple imputation (Seymour CW et al. JAMA 2019): 20 imputed data sets (chained equations with draws from
# the posterior), in each 50 k-means runs on 80% subsamples for k = 2-6; consensus matrices pooled over imputations; k chosen by the
# lowest proportion of ambiguous clustering (PAC, consensus values between 0.1 and 0.9); clusters from average-linkage hierarchical
# clustering of 1 - consensus.
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.cluster import AgglomerativeClustering
detail('Consensus k-means: IterativeImputer (sample_posterior=True, 10 iterations, random_state SEED + i) on the standardized indicators; 50 '
       'subsamples of 80% per imputation and k (1,000 per k in total), k-means with one start per subsample; k chosen by the lowest PAC '
       '(Senbabaoglu 2014) among 2-6; one-class solutions are not defined in consensus clustering.')
N_SUB = 50 if not QUICK else 5


def consensus_kmeans(Xv, seed=SEED):
    imps = [IterativeImputer(sample_posterior=True, max_iter=10, random_state=seed + i).fit_transform(Xv) for i in range(N_IMPUTATIONS)]
    parts = parallel_map(T.consensus_worker, [(imps[i], list(range(2, K_MAX + 1)), N_SUB, 0.8, seed + 31 * i) for i in range(N_IMPUTATIONS)])
    n = Xv.shape[0]
    pac, cons = {}, {}
    off = ~np.eye(n, dtype=bool)
    both = sum(p['both'].astype(np.int32) for p in parts)
    for k in range(2, K_MAX + 1):
        co = sum(p[k].astype(np.int32) for p in parts)
        Cm = np.where(both > 0, co / np.maximum(both, 1), np.nan)
        v = Cm[off]
        v = v[~np.isnan(v)]
        pac[k] = float(((v > 0.1) & (v < 0.9)).mean())
        cons[k] = Cm
    k_best = min(pac, key=pac.get)
    dist = 1 - np.nan_to_num(cons[k_best], nan=0.0)
    np.fill_diagonal(dist, 0)
    lab = AgglomerativeClustering(n_clusters=k_best, metric='precomputed', linkage='average').fit_predict(dist)
    Xi = np.mean(imps, axis=0)
    return k_best, lab, pac, Xi


t = time.time()
k_c, lab_c, pac_c, Xi_c = consensus_kmeans(X)
post_c = np.eye(k_c)[lab_c]                       # hard assignments as one-hot "posteriors"
order_c, cs_c, eff_c = score_order(post_c, CLU, RAW)
post_c = post_c[:, order_c]
lab_c = post_c.argmax(1)
means_c = np.array([np.nanmean(np.where(lab_c[:, None] == c, X, np.nan), axis=0) for c in range(k_c)])
out = {'analysis': 'Consensus k-means after multiple imputation', 'children': N, 'indicators': P, 'classes': k_c,
       'class_sizes': np.bincount(lab_c, minlength=k_c).tolist(), 'h1_effect_sd': round(float(eff_c), 2), 'h1_supported': bool(eff_c >= 0.5),
       'pac_by_k': {k: round(v, 3) for k, v in pac_c.items()}}
out.update(agreement_with_main(CLU, lab_c, means_c))
Yp = {k: outcome_arrays(DC)[k] for k in PRIMARY}
rng = np.random.RandomState(SEED)
rows_h2 = []
for k, y in Yp.items():
    def contrast(idx):
        a, b = y[idx][lab_c[idx] == 0], y[idx][lab_c[idx] == k_c - 1]
        a, b = a[~np.isnan(a)], b[~np.isnan(b)]
        return (np.mean(a) - np.mean(b)) if k in CONT else np.log(np.mean(a) / np.mean(b))
    with np.errstate(divide='ignore', invalid='ignore'):
        d0 = contrast(np.arange(N))
    bs = []
    for _ in range(N_BOOT_SENS):
        i = rng.randint(0, N, N)
        with np.errstate(divide='ignore', invalid='ignore'):
            bs.append(np.clip(contrast(i), -1e3, 1e3))         # a resample with no events in a cluster gives an unbounded ratio
    bs = np.array(bs, float)
    lo_, hi_ = ratio_ci(bs[~np.isnan(bs)])
    rows_h2.append({'validator': k, 'contrast': f'S1 vs S{k_c}', 'measure': 'ratio of geometric means' if k in CONT else 'risk ratio',
                    'estimate': float(np.exp(d0)) if np.isfinite(d0) and np.exp(d0) <= 1e6 else np.inf, 'ci_low': lo_, 'ci_high': hi_,
                    'p_value': boot_p(bs)})
out['h2'] = pd.DataFrame(rows_h2)
hh4 = h4(DC, np.array([f'S{c + 1}' for c in lab_c]), f'S{k_c}')
out.update({'h4_p': hh4.get('p_value', np.nan), 'h4_delta_c': hh4.get('delta_c', np.nan)})
ke, labe, _, Xie = consensus_kmeans(X[EARLY]); kl, labl, _, Xil = consensus_kmeans(X[~EARLY])
out.update({'h3_k_early': ke, 'h3_k_late': kl})
if ke == kl:
    ce = np.array([Xie[labe == c].mean(0) for c in range(ke)]); cl = np.array([Xil[labl == c].mean(0) for c in range(kl)])
    mp, _ = T.match_by_profile(ce, cl)
    pred_e = np.argmin(((Xil[:, None, :] - ce[None]) ** 2).sum(-1), axis=1)       # nearest 2019-2022 centroid
    out['h3_kappa'] = float(kappa_score(pred_e, np.array([mp[c] for c in labl]))); out['h3_met'] = out['h3_kappa'] >= 0.6
out['seconds'] = round(time.time() - t)
SENS.append(out)
log('10 Sensitivity', out['analysis'], {kk: v for kk, v in out.items() if kk not in ('h2', 'profile_r', 'matched_main')})

## Exploratory analyses added after the reported run (not in the registration)
Added on 2026-10-14, after the registered analysis above was complete, for the manuscript. Every output of this section says "not
registered". **E1** asks whether the subphenotypes carry prognostic information within the infection-triggered encephalopathy
syndrome (ITES) classes. **E2** describes immunomodulatory treatment and its timing by subphenotype; it estimates no treatment
effect, because treatment followed severity (confounding by indication). Neither analysis changes any registered result.

In [ ]:
# E1. Prognostic information of the subphenotypes within the ITES classes (not registered).
ITES_GROUPS = ['ANE', 'SAE, AFCE or ASEM', 'Other or possible ITES', 'Unclassifiable', 'Non-ITES']
_GMAP = {'ANE': ITES_GROUPS[0], 'SAE': ITES_GROUPS[1], 'AFCE': ITES_GROUPS[1], 'ASEM': ITES_GROUPS[1],
         'AESD': ITES_GROUPS[2], 'MERS': ITES_GROUPS[2], 'FIRES': ITES_GROUPS[2], 'HHE': ITES_GROUPS[2], 'Possible ITES': ITES_GROUPS[2],
         'Unclassifiable': ITES_GROUPS[3], 'Non-ITES: alternative diagnosis': ITES_GROUPS[4],
         'Non-ITES: encephalitis/meningitis': ITES_GROUPS[4], 'Non-ITES: neonatal encephalopathy': ITES_GROUPS[4]}
ITES_GRP = DC.ites_class.astype(str).map(_GMAP)
if ITES_GRP.isna().any():
    raise RuntimeError(f'ITES class without a group: {sorted(set(DC.ites_class.astype(str)[ITES_GRP.isna()]))}')
ITES_GRP = ITES_GRP.values
Y_POOR = b01(DC['death_or_dama_critical_final'])
addition('E1: prognostic information of the subphenotypes within the ITES classes. The ITES classes were grouped into five categories, each '
         'with events: ANE; SAE, AFCE or ASEM; other or possible ITES (AESD, MERS, FIRES, HHE and possible ITES); unclassifiable; non-ITES. '
         '(a) Death or critical discharge by subphenotype (most likely class) within each ITES class. (b) Risk ratio of the hyperinflammatory '
         'subphenotype versus all others within each category (Katz CI) and pooled across categories (Mantel-Haenszel, Greenland-Robins '
         'variance). (c) The H4 model (age as a restricted cubic spline with 3 knots; consciousness in four levels) plus ITES category, '
         'with and without subphenotype: likelihood-ratio test (chi-square, and parametric bootstrap with 1,000 simulations as for H4, '
         'because one subphenotype has no events) and change in C-statistic with a 95% bootstrap CI (both models refitted).')

_t = pd.DataFrame({'ites': DC.ites_class.astype(str).values, 'cls': LAB_NAMES, 'y': Y_POOR})
_cell = _t.groupby(['ites', 'cls']).y.agg(['sum', 'count'])
E1_EVENTS = (_cell['sum'].astype(int).astype(str) + '/' + _cell['count'].astype(str)).unstack().reindex(columns=CLS).fillna('')
E1_EVENTS['all'] = _t.groupby('ites').y.agg(lambda v: f'{int(v.sum())}/{len(v)}')
E1_EVENTS.insert(0, 'ites_category', [_GMAP[i] for i in E1_EVENTS.index])
E1_EVENTS = E1_EVENTS.sort_values('ites_category', key=lambda s: s.map(ITES_GROUPS.index), kind='stable')
E1_EVENTS.index.name = 'ites_class'


def rr_katz(a, n1, b, n0):
    """Risk ratio with a 95% CI on the log scale (Katz); not estimable when a group has no events or only events."""
    if min(a, b) == 0 or (a == n1 and b == n0):
        return np.nan, np.nan, np.nan
    rr = (a / n1) / (b / n0)
    se = math.sqrt(1 / a - 1 / n1 + 1 / b - 1 / n0)
    return rr, rr * math.exp(-1.96 * se), rr * math.exp(1.96 * se)


_rows, _num, _den, _var = [], 0.0, 0.0, 0.0
_hyp = LAB_NAMES == CLS[HYPER]
for g in ITES_GROUPS:
    m = ITES_GRP == g
    a, n1 = int(Y_POOR[m & _hyp].sum()), int((m & _hyp).sum())
    b, n0 = int(Y_POOR[m & ~_hyp].sum()), int((m & ~_hyp).sum())
    rr, lo, hi = rr_katz(a, n1, b, n0)
    _rows.append({'ites_category': g, f'{CLS[HYPER]}: events/children': f'{a}/{n1}' + (f' ({100 * a / n1:.0f}%)' if n1 else ''),
                  'other subphenotypes: events/children': f'{b}/{n0}' + (f' ({100 * b / n0:.0f}%)' if n0 else ''),
                  'risk_ratio': round(rr, 2) if np.isfinite(rr) else 'not estimable', 'ci_low': round(lo, 2) if np.isfinite(lo) else '',
                  'ci_high': round(hi, 2) if np.isfinite(hi) else ''})
    N_ = n1 + n0
    if N_:
        _num += a * n0 / N_
        _den += b * n1 / N_
        _var += (n1 * n0 * (a + b) - a * b * N_) / N_ ** 2
RR_MH = _num / _den
_se = math.sqrt(_var / (_num * _den))
RR_MH_CI = (RR_MH * math.exp(-1.96 * _se), RR_MH * math.exp(1.96 * _se))
_rows.append({'ites_category': 'Pooled across categories (Mantel-Haenszel)',
              f'{CLS[HYPER]}: events/children': f'{int(Y_POOR[_hyp].sum())}/{int(_hyp.sum())}',
              'other subphenotypes: events/children': f'{int(Y_POOR[~_hyp].sum())}/{int((~_hyp).sum())}',
              'risk_ratio': round(RR_MH, 2), 'ci_low': round(RR_MH_CI[0], 2), 'ci_high': round(RR_MH_CI[1], 2)})
E1_HYPER = pd.DataFrame(_rows)


def beyond_ites(Dc, labels, groups, least, n_boot, n_sim=1000, seed=SEED):
    """The H4 model plus ITES category, with and without subphenotype: likelihood-ratio test (chi-square and parametric bootstrap) and
    change in C-statistic with a percentile bootstrap CI."""
    from scipy.stats import chi2
    keep = Dc.cons4.notna().values
    Dk, lab, grp = Dc[keep], np.asarray(labels)[keep], np.asarray(groups)[keep]
    y = b01(Dk['death_or_dama_critical_final'])
    base, full0, _ = h4_design(Dk, lab, least)
    for g in ITES_GROUPS:
        if g != 'Other or possible ITES':                   # reference category: the largest, with the lowest risk
            base[f'ites_{g}'] = (grp == g).astype(float)
    full = pd.concat([base, full0[[c for c in full0.columns if c.startswith('class_')]]], axis=1)
    m0, c0 = logit_fit(y, base)
    m1, c1 = logit_fit(y, full)
    lr = float(2 * (m1.llf - m0.llf))
    df_ = full.shape[1] - base.shape[1]
    out = {'n': int(keep.sum()), 'events': int(y.sum()), 'lr_stat': lr, 'df': int(df_), 'p_chi_square': float(chi2.sf(lr, df_)),
           'c_base': float(roc_auc_score(y, m0.predict(base))), 'c_full': float(roc_auc_score(y, m1.predict(full))),
           'models_converged': bool(c0 and c1)}
    out['delta_c'] = out['c_full'] - out['c_base']
    rng = np.random.RandomState(seed)
    deltas = []
    for _ in range(n_boot):
        i = rng.randint(0, len(y), len(y))
        try:
            b0, _c = logit_fit(y[i], base.iloc[i])
            b1, _c = logit_fit(y[i], full.iloc[i])
            deltas.append(roc_auc_score(y[i], b1.predict(full.iloc[i])) - roc_auc_score(y[i], b0.predict(base.iloc[i])))
        except Exception:
            continue
    out['delta_c_ci'] = (float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)))
    out['boot_used'] = len(deltas)
    p0 = np.asarray(m0.predict(base))
    rng = np.random.RandomState(seed + 1)
    lrs = []
    for _ in range(n_sim):
        ys = (rng.uniform(size=len(p0)) < p0).astype(float)
        try:
            a_, _c = logit_fit(ys, base)
            b_, _c = logit_fit(ys, full)
            lrs.append(2 * (b_.llf - a_.llf))
        except Exception:
            continue
    lrs = np.array(lrs)
    out['p_parametric_bootstrap'] = float((1 + np.sum(lrs >= lr)) / (1 + len(lrs)))
    out['simulations'] = int(len(lrs))
    return out


E1_MODEL = beyond_ites(DC, LAB_NAMES, ITES_GRP, CLS[LEAST], n_boot=N_BOOT_H4)
log('12 Exploratory (not registered)', 'E1: events by ITES category, hyperinflammatory vs other subphenotypes',
    E1_HYPER[['ites_category', f'{CLS[HYPER]}: events/children', 'other subphenotypes: events/children', 'risk_ratio', 'ci_low', 'ci_high']]
    .to_dict('records'))
log('12 Exploratory (not registered)', 'E1: subphenotype added to age, consciousness and ITES category',
    f"n {E1_MODEL['n']} ({E1_MODEL['events']} events); LR {E1_MODEL['lr_stat']:.2f} on {E1_MODEL['df']} df, chi-square P = "
    f"{E1_MODEL['p_chi_square']:.2g}, parametric-bootstrap P = {E1_MODEL['p_parametric_bootstrap']:.4f} ({E1_MODEL['simulations']} simulations); "
    f"C {E1_MODEL['c_base']:.3f} -> {E1_MODEL['c_full']:.3f}, change {E1_MODEL['delta_c']:.3f} (95% CI {E1_MODEL['delta_c_ci'][0]:.3f} to "
    f"{E1_MODEL['delta_c_ci'][1]:.3f}; {E1_MODEL['boot_used']} resamples); models converged: {E1_MODEL['models_converged']}")
E1_MODEL_TAB = pd.DataFrame([{k: (f'{v[0]:.3f} to {v[1]:.3f}' if isinstance(v, tuple) else v) for k, v in E1_MODEL.items()}])
E1_EVENTS

In [ ]:
# E2. Immunomodulatory treatment and its timing by subphenotype (not registered; descriptive).
DRO = load('drug_orders')
DRO = DRO[DRO.visit_id.isin(DC.index)]


def hours_to_first(mask):
    """Hours from admission to the first order of a drug in the index admission (NaN when never ordered)."""
    t = DRO[mask].groupby('visit_id').order_time.min()
    return ((t.reindex(DC.index) - DC.admit_time).dt.total_seconds() / 3600).values


H_MP = hours_to_first(DRO.drug_name_cn.str.contains('甲泼尼龙|甲强龙|甲基强的松龙', na=False))
H_IG = hours_to_first(DRO.drug_classes.fillna('').str.contains('immunoglobulin'))
H_TCZ = hours_to_first(DRO.drug_classes.fillna('').str.contains('il6_receptor_blocker'))
for _nm, _h, _flag in [('methylprednisolone', H_MP, 'methylprednisolone'), ('immunoglobulin', H_IG, 'ivig'), ('tocilizumab', H_TCZ, 'tocilizumab')]:
    _agree = int((~np.isnan(_h) == DC[_flag].fillna(False).astype(bool).values).sum())
    log('12 Exploratory (not registered)', f'E2: children with a first {_nm} order time / agreeing with the frozen treatment flag',
        f'{int((~np.isnan(_h)).sum())} / {_agree} of {len(DC)}')
addition('E2: immunomodulatory treatment and its timing by subphenotype (descriptive). Hours from admission to the first order of '
         'methylprednisolone (drug name as for the frozen flag), immunoglobulin and tocilizumab (drug classes of the frozen drug_orders table) '
         'in the index admission; "within 24 h" means a first order at most 24 h after admission (orders before admission, such as at a '
         'referring hospital, are not recorded). Among children still in hospital 24 h after admission (landmark), death or critical '
         'discharge, coma at admission and shock in the discharge diagnoses are shown by methylprednisolone within 24 h, within each '
         'subphenotype. No treatment effect is estimated: treatment followed severity (confounding by indication).')


def n_pct(m, d=None):
    m = np.asarray(m, bool)
    d = len(m) if d is None else d
    return f'{int(m.sum())} ({100 * m.sum() / d:.0f}%)' if d else ''


_rows = []
for i, c in enumerate(CLS):
    m = MODAL == i
    r = {'subphenotype': c, 'children': int(m.sum())}
    for nm, h in [('Methylprednisolone', H_MP), ('Immunoglobulin', H_IG), ('Tocilizumab', H_TCZ)]:
        hh = h[m]
        r[f'{nm}, any'] = n_pct(~np.isnan(hh))
        r[f'{nm}, within 24 h of admission'] = n_pct(hh <= 24)
        r[f'{nm}, hours from admission to first order, median (IQR)'] = med_iqr(hh[~np.isnan(hh)])
    _rows.append(r)
E2_TIMING = pd.DataFrame(_rows).set_index('subphenotype').T

LAND = DC.los_hours.values > 24
_coma = DC.cons4.eq('coma').values
_cons_known = DC.cons4.notna().values
_shock = DC.shock_dx.fillna(False).astype(bool).values
_rows = []
for i, c in enumerate(CLS):
    for early in (True, False):
        m = (MODAL == i) & LAND & ((H_MP <= 24) == early)
        _rows.append({'subphenotype': c, 'methylprednisolone within 24 h': 'yes' if early else 'no', 'children': int(m.sum()),
                      'death or critical discharge': n_pct(Y_POOR[m] == 1),
                      'coma at admission (of recorded)': n_pct(_coma[m & _cons_known], int((m & _cons_known).sum())),
                      'shock in discharge diagnoses': n_pct(_shock[m])})
E2_LANDMARK = pd.DataFrame(_rows)
log('12 Exploratory (not registered)', 'E2: landmark at 24 h (children discharged or dead within 24 h of admission / their events)',
    f'{int((~LAND).sum())} / {int(Y_POOR[~LAND].sum())}')
log('12 Exploratory (not registered)', 'E2: methylprednisolone within 24 h of admission by subphenotype',
    {c: E2_TIMING.loc['Methylprednisolone, within 24 h of admission', c] for c in CLS})
E2_TIMING

## Outputs: tables

In [ ]:
# Table 1: characteristics by subphenotype (most likely class); clustered children.
def pct(mask, denom_mask=None):
    m = pd.Series(mask).fillna(False).astype(bool)
    d = len(m) if denom_mask is None else int(pd.Series(denom_mask).sum())
    return f'{int(m.sum())} ({100 * m.sum() / d:.1f}%)' if d else ''


def t1_column(Dg):
    r = {'Children': f'{len(Dg)}', 'Age, years, median (IQR)': med_iqr(Dg.age_years),
         'Male sex': pct(Dg.sex.eq('male')), 'Admitted 2023-2026': pct(Dg.period.eq('2023-2026')),
         'Admitted through the emergency department': pct(Dg.admission_source.eq('emergency')),
         'Transferred from another hospital': pct(Dg.admission_source.eq('transfer_other_hospital')),
         'Onset of acute illness <= 24 h before admission': pct(Dg.onset_hours_acute.le(24), Dg.onset_hours_acute.notna()),
         'Onset recorded': pct(Dg.onset_hours_acute.notna())}
    for lev in ['alert', 'altered', 'stupor', 'coma']:
        r[f'Consciousness at admission: {lev}'] = pct(Dg.cons4.eq(lev), Dg.cons4.notna())
    r['Consciousness: sedated or not recorded'] = pct(Dg.cons4.isna())
    r['Status epilepticus'] = pct(Dg.status_epilepticus)
    for tier in ['Definite ITES', 'ITES-related', 'Possible ITES', 'Sepsis-associated encephalopathy', 'Unclassifiable', 'Non-ITES']:
        r[f'ITES tier: {tier}'] = pct(Dg.ites_tier.eq(tier))
    for c, l in [('methylprednisolone', 'Methylprednisolone'), ('ivig', 'IVIG'), ('tocilizumab', 'Tocilizumab')]:
        r[l] = pct(Dg[c])
    return r


TABLE1 = pd.DataFrame({'All clustered': t1_column(DC), **{c: t1_column(DC[MODAL == i]) for i, c in enumerate(CLS)}})

# Sensitivity summary
srows = []
for s in SENS:
    h2 = s.get('h2', pd.DataFrame())
    get = lambda v: h2[h2.validator == v].iloc[0] if len(h2) and (h2.validator == v).any() else None
    row = {'analysis': s['analysis'], 'children': s.get('children'), 'classes': s.get('classes'), 'class_sizes': s.get('class_sizes'),
           'kappa_vs_main': s.get('kappa_vs_main'), 'closest_main_class': s.get('closest_main_class', ''), 'h1_effect_sd': s.get('h1_effect_sd'),
           'h1_supported': s.get('h1_supported')}
    for v in PRIMARY:
        g = get(v)
        if g is not None and 'estimate' in g and pd.notna(g['estimate']):
            ci = f" ({g['ci_low']:.2f}-{g['ci_high']:.2f})" if 'ci_low' in g and pd.notna(g.get('ci_low', np.nan)) else ''
            row[f'H2 {v}'] = f"{g['estimate']:.2f}{ci}"
    row.update({'h3_k_early_late': f"{s.get('h3_k_early', '')}/{s.get('h3_k_late', '')}" if 'h3_k_early' in s else 'not repeated',
                'h3_kappa': s.get('h3_kappa', np.nan), 'h4_p': s.get('h4_p', np.nan), 'h4_delta_c': s.get('h4_delta_c', np.nan)})
    srows.append(row)
SENS_TAB = pd.DataFrame(srows)

MAIN_SUMMARY = pd.DataFrame([
    {'item': 'Children clustered', 'value': N},
    {'item': 'Clustering indicators', 'value': P},
    {'item': 'Classes chosen', 'value': K_STAR},
    {'item': 'H1: hyperinflammatory class score above the cohort mean (SD units)', 'value': round(float(H1_EFFECT), 2) if K_STAR > 1 else ''},
    {'item': 'H1 supported', 'value': H1},
    {'item': f'Stability: resamples choosing {K_STAR} classes', 'value': f"{int((STAB_K == K_STAR).sum())} of {len(STAB)}"},
    {'item': 'Stability: adjusted Rand index, median (IQR)', 'value': f'{np.median(STAB_ARI):.2f} ({np.percentile(STAB_ARI, 25):.2f}-{np.percentile(STAB_ARI, 75):.2f})'},
    {'item': 'H3: classes chosen 2019-2022 / 2023-2026', 'value': f"{H3['k_early']} / {H3['k_late']}"},
    {'item': 'H3: kappa on 2023-2026 children', 'value': round(H3['kappa'], 3) if np.isfinite(H3['kappa']) else 'not computed (different numbers of classes)'},
    {'item': 'H3 met', 'value': H3['met']},
    {'item': 'H4: likelihood-ratio P', 'value': H4.get('p_value', '')},
    {'item': 'H4: parametric-bootstrap P (only when a class has no events)', 'value': H4.get('p_parametric_bootstrap', '')},
    {'item': 'H4: change in C-statistic (95% CI)', 'value': f"{H4['delta_c']:.3f} ({H4['delta_c_ci'][0]:.3f} to {H4['delta_c_ci'][1]:.3f})" if 'delta_c_ci' in H4 else ''}])

# Agreement of each class with the main solution across the sensitivity analyses that refit the classes (profile correlation r)
_agr = []
for s in SENS:
    if 'profile_r' not in s:
        continue
    r_ = np.array(s['profile_r'])
    row = {'analysis': s['analysis'], 'classes': s.get('classes'), 'kappa_vs_main': s.get('kappa_vs_main')}
    for i, c in enumerate(CLS):
        js = [j for j, m in enumerate(s['matched_main']) if m == i]
        row[f'r_{c}'] = round(float(max(r_[i, j] for j in js)), 3) if js else np.nan
    _agr.append(row)
SENS_AGREE = pd.DataFrame(_agr)

TABLES = {'main_summary': MAIN_SUMMARY, 'availability': AVAIL.reset_index(drop=True), 'correlated_pairs': pd.DataFrame(PAIRS), 'fit_statistics': FITS,
          'class_summary': CLASS_TAB, 'table1_by_class': TABLE1.reset_index().rename(columns={'index': 'characteristic'}),
          'profiles_model_means': PROFILE.reset_index().rename(columns={'index': 'indicator'}),
          'profiles_observed_means': PROFILE_OBS.reset_index().rename(columns={'index': 'indicator'}),
          'indicators_original_units': RAW_BY_CLASS.reset_index().rename(columns={'index': 'indicator'}),
          'validators_by_class': VAL_TAB, 'validator_contrasts': CONTRAST, 'h2_primary': H2_TAB,
          'ites_class_crosswalk': CROSS_ITES.reset_index(), 'treatments_by_class': TREAT.reset_index().rename(columns={'index': 'treatment'}),
          'ifn_gamma_by_class': IFN.reset_index().rename(columns={'index': 'item'}), 'stability': STAB_TAB,
          'h4': pd.DataFrame([{k: (str(v) if isinstance(v, (dict, tuple)) else v) for k, v in H4.items()}]) if H4 else pd.DataFrame(),
          'classifier': pd.DataFrame([{k: (str(v) if isinstance(v, (dict, list)) else v) for k, v in CLF.items()}]) if CLF else pd.DataFrame(),
          'sensitivity_summary': SENS_TAB, 'sensitivity_class_agreement': SENS_AGREE, 'chart_review_agreement': CHART_TAB,
          'exploratory_E1_ites_events': E1_EVENTS.reset_index(), 'exploratory_E1_hyper_vs_other': E1_HYPER,
          'exploratory_E1_beyond_ites': E1_MODEL_TAB, 'exploratory_E2_treatment_timing': E2_TIMING.reset_index().rename(columns={'index': 'item'}),
          'exploratory_E2_landmark': E2_LANDMARK}
wait_until_writable(TAB.glob('*'))
for name, df in TABLES.items():
    df.to_csv(TAB / f'{name}.csv', index=False, encoding='utf-8-sig')
with pd.ExcelWriter(TAB / 'subphenotype_results.xlsx', engine='openpyxl') as xw:
    for name, df in TABLES.items():
        df.to_excel(xw, sheet_name=name[:31], index=False)
log('11 Outputs', 'Tables written', len(TABLES))

In [ ]:
# Saved results for redrawing the figures. AE_subphenotypes_redraw_figures.ipynb loads them and runs the figure cells below, so the
# figures can be redrawn in about a minute without rerunning the analysis. The file holds child-level data: it stays next to this
# notebook, outside the output folder, and is not shared. The hash of the analysis cells above lets the redraw check that it uses
# results of the same analysis code.
import pickle, types
STATE_MARKER = '# Saved results for redrawing the figures.'
END_MARKER = '# Run log, then the identifier check over every output file.'


def analysis_code_digest():
    """Hash of the analysis cells of this notebook (the code cells before this one), ignoring comments and layout."""
    nbs = sorted(HERE.glob('AE_subphenotypes_analysis*.ipynb'), key=lambda p: (p.name != 'AE_subphenotypes_analysis.ipynb', -p.stat().st_mtime))
    for nb in nbs:
        try:
            src = [''.join(c['source']) if isinstance(c['source'], list) else c['source']
                   for c in json.loads(nb.read_text(encoding='utf-8'))['cells'] if c['cell_type'] == 'code']
            end = next(i for i, s in enumerate(src) if s.lstrip().startswith(STATE_MARKER))
            return hashlib.sha256('\n'.join(ast.dump(ast.parse(s)) for s in src[:end]).encode('utf-8')).hexdigest()[:16]
        except Exception:
            continue
    return None


STATE_DIR = HERE / 'cache_figure_state'
STATE_DIR.mkdir(exist_ok=True)
_not_saved = {'In', 'Out', 'exit', 'quit', 'get_ipython', 'HERE', 'FREEZE', 'OUT', 'TAB', 'FIG', 'LOGD', 'CODE', 'PROGRESS', 'STATE_DIR', 'MP'}
_state = {}
for _k, _v in list(globals().items()):
    if (_k.startswith('_') or _k in _not_saved or isinstance(_v, (types.ModuleType, type))
            or (callable(_v) and not hasattr(_v, 'shape'))):
        continue
    try:
        pickle.dumps(_v, protocol=pickle.HIGHEST_PROTOCOL)
    except Exception:
        continue
    _state[_k] = _v
with open(STATE_DIR / 'figure_state_INTERNAL.pkl', 'wb') as _fh:
    pickle.dump({'saved': time.strftime('%Y-%m-%d %H:%M:%S'), 'analysis_code': analysis_code_digest(), 'software': SOFTWARE,
                 'objects': _state}, _fh, protocol=pickle.HIGHEST_PROTOCOL)
log('11 Outputs', 'Results saved for redrawing the figures (local folder cache_figure_state, not shared)', f'{len(_state)} objects')

## Figures
Main-text Figures 1-6 and supplementary Figures S1-S6, drawn from the results above (nothing is typed in by hand). One color
system throughout, built on the three primary colors: the subphenotypes run from red (S1, hyperinflammatory) through orange,
gold, teal and sky blue to deep blue (S6, least abnormal), in the order of the registered inflammation score; heat maps use
blue-white-red for signed values and white-red or white-blue for proportions and agreement. Figure legends, with every statistic,
are written to `figures/figure_legends.md`. Style: Arial (Liberation Sans where Arial is not installed), bold axis titles, no
figure titles, panel letters aligned with the y-axis title, white background, no gridlines, every reference line in the legend.
Size: all figures are drawn 10 inches wide, and the main figures at most 10.3 inches high. The journal prints main figures at the
full page width of 170 mm (scale 0.67), where the smallest text (10 to 10.5 points here) prints at about 7 points.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, PathPatch, Rectangle, Polygon
from matplotlib.path import Path as MPath
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, to_rgb, to_hex
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
from sklearn.metrics import roc_curve

wait_until_writable(FIG.glob('*'))
for _f in FIG.glob('*'):                     # figures are regenerated in full on every run
    if _f.is_file():
        _f.unlink()
FIGURE_CODE_VERSION = '2026-10-15'      # date of the last change to the figure cells; the redraw notebook reports it
log('11 Outputs', 'Figure code version', FIGURE_CODE_VERSION)
_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((f for f in ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans'] if f in _fonts), 'sans-serif')
log('11 Outputs', 'Figure font', FONT if FONT == 'Arial' else f'{FONT} (Arial is not installed on this computer)')
plt.rcParams.update({'font.family': FONT, 'font.size': 11, 'axes.labelsize': 12, 'axes.labelweight': 'bold', 'xtick.labelsize': 11,
                     'ytick.labelsize': 11, 'legend.fontsize': 10.5, 'legend.handlelength': 1.6, 'legend.borderaxespad': 0.3, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': False, 'axes.linewidth': 0.9, 'xtick.major.width': 0.9, 'ytick.major.width': 0.9,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white', 'legend.frameon': False,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'mathtext.fontset': 'custom', 'mathtext.rm': FONT,
                     'mathtext.it': f'{FONT}:italic', 'mathtext.bf': f'{FONT}:bold', 'mathtext.cal': FONT, 'mathtext.sf': FONT})
RYB = ['#C62828', '#EF6C00', '#E3A600', '#2A9D8F', '#4FA3D9', '#1F3F8F']        # red ... gold ... blue
CC = RYB if K_STAR == 6 else [to_hex(LinearSegmentedColormap.from_list('ryb', RYB)(x)) for x in np.linspace(0, 1, K_STAR)]
RED, ORANGE, GOLD, TEAL, SKY, BLUE = RYB
GREY = '#8A8A8A'
CMAP_RED = LinearSegmentedColormap.from_list('white_red', ['#FFFFFF', '#F6C1B8', '#E0533F', '#9E1B1B'])
CMAP_BLUE = LinearSegmentedColormap.from_list('white_blue', ['#FFFFFF', '#BFD7EE', '#4FA3D9', '#1F3F8F'])
LEGENDS = {}
NUM_WORDS = {2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}


def _txt_color(rgb):
    r, g, b = to_rgb(rgb)
    return 'white' if 0.299 * r + 0.587 * g + 0.114 * b < 0.55 else 'black'


def minus(s):
    return s.replace('-', '−')


def sci(v, digits=1):
    """A small number in scientific notation for legends, e.g. 4.8 × 10⁻¹⁴."""
    m, e = f'{v:.{digits}e}'.split('e')
    sup = str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹')
    return f'{m} × 10{str(int(e)).translate(sup)}'


def panel_letters(fig, pairs):
    """Bold letters above the upper-left corner of each panel, aligned with the left edge of its y-axis title (or tick labels).
    An entry may carry a third element: an artist above the axes (such as a legend) that the letter is placed above."""
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    for ax, letter, *above in pairs:
        bb = ax.yaxis.get_tightbbox(rend) if ax.axison else ax.get_tightbbox(rend)
        x0 = inv.transform(((bb if bb is not None else ax.get_window_extent(rend)).x0, 0))[0]
        top = max(ax.get_window_extent(rend).y1, ax.xaxis.get_tightbbox(rend).y1 if ax.axison and ax.xaxis.get_tightbbox(rend) is not None
                  else 0, 0 if ax.axison else ax.get_tightbbox(rend).y1, *[a.get_window_extent(rend).y1 for a in above])
        y1 = inv.transform((0, top))[1]
        fig.text(x0, y1 + 0.08 / fig.get_figheight(), letter, fontsize=16, fontweight='bold', ha='left', va='bottom')


def area(n):
    """Marker area (points squared) of a circle standing for n children."""
    return 5 * n + 12


def size_key(band, counts=(10, 50, 100), fs=10.5):
    """Key to circle areas (see area) drawn in a band axes, centered, the largest circle touching the top."""
    fig_ = band.figure
    band.set_xlim(0, 1); band.set_ylim(0, 1); band.axis('off')
    fig_.canvas.draw()
    rend = fig_.canvas.get_renderer()
    bb = band.get_window_extent(rend)
    w_in, h_in = bb.width / fig_.dpi, bb.height / fig_.dpi
    items, x = [], 0.0
    for n in counts:
        s = area(n)
        d_in = (math.sqrt(s) + 1.5) / 72
        t = band.text(0, 0.5, f'{n} children', fontsize=fs, va='center', ha='left')
        tw = t.get_window_extent(rend).width / fig_.dpi
        items.append((s, d_in, t, x))
        x += d_in + 0.07 + tw + 0.4
    x0 = (w_in - (x - 0.4)) / 2
    yc = 1 - items[-1][1] / 2 / h_in
    for s, d_in, t, xx in items:
        band.scatter((x0 + xx + d_in / 2) / w_in, yc, s=s, color='white', edgecolor=GREY, linewidth=1.3, clip_on=False)
        t.set_position(((x0 + xx + d_in + 0.07) / w_in, yc))


def save(fig, name, legend):
    for ext in ('pdf', 'png'):
        fig.savefig(FIG / f'{name}.{ext}', dpi=300, bbox_inches='tight')
    LEGENDS[name] = legend
    plt.show()
    plt.close(fig)


def class_axis(ax, counts=None, label='Subphenotype'):
    ax.set_xticks(range(K_STAR))
    ax.set_xticklabels([f'{c}\n({counts[i]})' if counts is not None else c for i, c in enumerate(CLS)])
    ax.set_xlim(-0.6, K_STAR - 0.4)
    if label:
        ax.set_xlabel(label)


BMAT = {k: np.array([b[k] for b in BOOT], float) for k in EST}


def display_values(name):
    """BCH point estimate and 95% bootstrap percentile CI per class, on the display scale."""
    e, B = np.asarray(EST[name], float), BMAT[name]
    lo, hi = np.nanpercentile(B, 2.5, axis=0), np.nanpercentile(B, 97.5, axis=0)
    if name == 'CSF white cells':
        f = lambda v: np.exp(v) - 1
    elif name == 'Length of stay':
        f = lambda v: np.exp(v) / 24
    elif name in CONT:
        f = np.exp
    else:
        f = lambda v: np.where(np.asarray(v) < 1e-9, 0.0, 100 * np.asarray(v))
    return f(e), f(lo), f(hi)


def n_measured(name):
    return [int((~np.isnan(Y_MAIN[name][MODAL == c])).sum()) for c in range(K_STAR)]


def g3(v):
    if v >= 1000:
        return f'{float(f"{v:.3g}"):,.0f}'
    return f'{v:.3g}' if v >= 1 else f'{v:.2g}'


def log_axis(ax, axis='y'):
    """Logarithmic axis with plain-number tick labels (1, 10, 100 or 1, 2, 5, 10 ...), unlabeled minor ticks."""
    a = ax.yaxis if axis == 'y' else ax.xaxis
    lo, hi = (ax.get_ylim() if axis == 'y' else ax.get_xlim())
    dec = np.log10(hi / lo) if lo > 0 else 3
    subs = (1.0,) if dec > 2.6 else ((1.0, 3.0) if dec > 1.3 else (1.0, 2.0, 5.0))
    a.set_major_locator(LogLocator(base=10, subs=subs))
    a.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v >= 1000 else f'{v:g}'))
    a.set_minor_formatter(NullFormatter())


def dot_panel(ax, name, ylabel, log=True, show_n=True):
    """BCH estimates with 95% CIs by subphenotype; the value is written above the upper CI limit."""
    est, lo, hi = display_values(name)
    for c in range(K_STAR):
        ax.errorbar(c, est[c], yerr=[[max(0, est[c] - lo[c])], [max(0, hi[c] - est[c])]], fmt='o', color=CC[c], ecolor=CC[c],
                    elinewidth=1.8, capsize=3.5, ms=7.5, mec='white', mew=0.8, zorder=3)
    if log:
        ax.set_yscale('log')
        ax.set_ylim(np.nanmin(lo) / 1.7, np.nanmax(hi) * 3.4)
    else:
        rng_ = np.nanmax(hi) - np.nanmin(lo)
        ax.set_ylim(np.nanmin(lo) - 0.05 * rng_, np.nanmax(hi) + 0.2 * rng_)
    for c in range(K_STAR):
        ax.annotate(g3(est[c]), (c, hi[c]), xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10.5)
    if log:
        log_axis(ax)
    class_axis(ax, n_measured(name) if show_n else None)
    ax.set_ylabel(ylabel)


def bar_panel(ax, name, ylabel, show_n=True):
    est, lo, hi = display_values(name)
    ax.bar(range(K_STAR), est, width=0.66, color=CC, alpha=0.88, zorder=2)
    ax.errorbar(range(K_STAR), est, yerr=[np.maximum(0, est - lo), np.maximum(0, hi - est)], fmt='none', ecolor='#333333', elinewidth=1.1,
                capsize=3, zorder=3)
    for c in range(K_STAR):
        ax.annotate(f'{est[c]:.0f}', (c, hi[c]), xytext=(0, 2.5), textcoords='offset points', ha='center', va='bottom', fontsize=10.5)
    ax.set_ylim(0, min(108, max(hi) * 1.2 + 6))
    class_axis(ax, n_measured(name) if show_n else None)
    ax.set_ylabel(ylabel)


SHORT = {'Heart rate (z-score for age)': 'Heart rate, z-score', 'Respiratory rate (z-score for age)': 'Respiratory rate, z-score',
         'High-sensitivity CRP': 'CRP', 'Lymphocyte percentage': 'Lymphocytes, %', 'Neutrophil count': 'Neutrophils'}
UNITS = {'neutrophils_abs': r'Neutrophils, $10^9$/L', 'wbc': r'White cells, $10^9$/L', 'lymphocytes_pct': 'Lymphocytes, %',
         'platelets': r'Platelets, $10^9$/L', 'hemoglobin': 'Hemoglobin, g/L', 'hs_crp': 'CRP, mg/L', 'procalcitonin': 'Procalcitonin, ng/mL',
         'alt': 'ALT, U/L', 'ast': 'AST, U/L', 'ldh': 'LDH, U/L', 'ck': 'CK, U/L', 'bilirubin_total': 'Bilirubin, µmol/L',
         'albumin': 'Albumin, g/L', 'creatinine': 'Creatinine, µmol/L', 'urea': 'Urea, mmol/L', 'sodium': 'Sodium, mmol/L',
         'ammonia': 'Ammonia, µmol/L', 'lactate': 'Lactate, mmol/L', 'inr': 'INR', 'aptt': 'APTT, s', 'fibrinogen': 'Fibrinogen, g/L',
         'd_dimer': 'D-dimer, mg/L FEU', 'temp_c': 'Temperature, °C', 'heart_rate': 'Heart rate, beats/min',
         'resp_rate': 'Respiratory rate, breaths/min'}
SHORT_SENS = {'Worst value in the window': 'Worst value in the window', 'Window 12 h before to 24 h after admission, own admission only':
              'Window −12 to +24 h, own admission', 'Without winsorizing': 'Without winsorizing',
              'Leaving out children with a non-ITES class': 'Without non-ITES children',
              'Leaving out children transferred from another hospital': 'Without transferred children',
              'Vital signs from the admission examination': 'Vital signs from the examination',
              'Onset of the acute illness 24 h or less before admission': 'Onset ≤ 24 h before admission',
              'Onset of the acute illness more than 24 h before admission': 'Onset > 24 h before admission',
              'Additional (not registered): white cell count kept instead of neutrophil count': 'White cells instead of neutrophils*',
              'Gaussian model with full covariance': 'Full covariance', 'Consensus k-means after multiple imputation': 'Consensus k-means'}
RAWV = pd.concat([LABVAL, VITALS_RAW[['temp_c', 'heart_rate', 'resp_rate']]], axis=1).loc[CLU]


def box_panel(ax, col, show_labels=True, rotate=False):
    vals = RAWV[col].values.astype(float)
    groups = [vals[(MODAL == c) & ~np.isnan(vals)] for c in range(K_STAR)]
    bp = ax.boxplot(groups, positions=range(K_STAR), widths=0.64, patch_artist=True, showfliers=False, whis=(5, 95),
                    medianprops=dict(color='black', lw=1.5), whiskerprops=dict(color='#444444'), capprops=dict(color='#444444'))
    for c, b in enumerate(bp['boxes']):
        b.set_facecolor(CC[c]); b.set_alpha(0.85); b.set_edgecolor('#444444')
    logy = col in LOG_SCALE and col != 'inr'
    if logy:
        ax.set_yscale('log')
    if show_labels:
        span = np.nanpercentile(vals, 99) - np.nanpercentile(vals, 1)
        for c, g in enumerate(groups):
            if len(g):
                top = np.percentile(g, 95)
                ax.text(c, top * 1.1 if logy else top + 0.02 * span, g3(float(np.median(g))), ha='center', va='bottom',
                        fontsize=10, rotation=90 if rotate else 0)
    if logy:
        lo_, hi_ = ax.get_ylim(); ax.set_ylim(lo_, hi_ * (1.9 if rotate else 1.35)); log_axis(ax)
    else:
        lo_, hi_ = ax.get_ylim(); ax.set_ylim(lo_, hi_ + (0.12 if rotate else 0.05) * (hi_ - lo_))
    class_axis(ax, None, label=None)
    ax.set_ylabel(UNITS.get(col, col))


H2P = H2_TAB.set_index('validator') if len(H2_TAB) else pd.DataFrame()


def h2_text(v):
    r = H2P.loc[v]
    hi = 'not bounded' if not np.isfinite(r.ci_high) else f'{r.ci_high:.1f}'
    return f"{r.estimate:.1f} (95% CI {r.ci_low:.1f} to {hi}; P = {r.p_value:.3f}, Holm-adjusted P = {r.p_holm:.3f})"

In [ ]:
# Figure 1. Study design and derivation of the subphenotypes.
n_neo = int(D.neonate.sum()); n_few = int((D.few_tests & ~D.neonate.astype(bool)).sum())
n_e, n_l = int((D.loc[CLU, 'period'] == '2019-2022').sum()), int((D.loc[CLU, 'period'] == '2023-2026').sum())
n_lab = sum(c in LAB_IND for c in FINAL); n_vit = P - n_lab
yrs = f"{int(D.admit_year.min())}–{int(D.admit_year.max())}"
W1, H1 = 10.0, 8.7
fig = plt.figure(figsize=(W1, H1))


def ax_in1(x, y, w, h):
    return fig.add_axes([x / W1, y / H1, w / W1, h / H1])


XA0, YA0, WA1, HA1 = 0.25, 4.3, 9.5, 4.1                                  # flow chart, inches
axA = ax_in1(XA0, YA0, WA1, HA1)
axB = ax_in1(1.05, 0.8, 3.15, 2.85)
axC = ax_in1(6.2, 0.8, 3.55, 2.85)
axA.set_xlim(0, 1); axA.set_ylim(0, 1); axA.axis('off')
PT_X, PT_Y = 1 / 72 / WA1, 1 / 72 / HA1                                  # one point in axes units
LW_BOX, LW_LINE, LINE_C, PAD = 1.3, 1.4, '#555555', 0.004
FS1 = 10.5


def fy(inch):
    return inch / HA1


def fbox(x0, y0, x1, y1, text, fc, ec, fs=FS1, color='black', weight='normal'):
    """Rounded box whose outline runs along x0..x1, y0..y1. Returns the outer edges of the drawn outline (left, bottom, right, top)."""
    axA.add_patch(FancyBboxPatch((x0 + PAD, y0 + PAD * WA1 / HA1), x1 - x0 - 2 * PAD, y1 - y0 - 2 * PAD * WA1 / HA1,
                                 boxstyle=f'round,pad={PAD},rounding_size=0.008', mutation_aspect=WA1 / HA1,
                                 fc=fc, ec=ec, lw=LW_BOX, zorder=2, clip_on=False))
    axA.text((x0 + x1) / 2, (y0 + y1) / 2, text, ha='center', va='center', fontsize=fs, color=color, fontweight=weight, linespacing=1.3,
             zorder=3, clip_on=False)
    hx, hy = LW_BOX / 2 * PT_X, LW_BOX / 2 * PT_Y
    return x0 - hx, y0 - hy, x1 + hx, y1 + hy


def fline(xs, ys, cap='butt'):
    axA.plot(xs, ys, color=LINE_C, lw=LW_LINE, solid_capstyle=cap, zorder=1, clip_on=False)


def farrow(x0, y0, x1, y1, head_len=7.0, head_w=5.6):
    """Arrow whose tip lies exactly on (x1, y1), the outer edge of the target box's outline: it touches the box without entering it."""
    ux, uy = (x1 - x0) / PT_X, (y1 - y0) / PT_Y
    L = math.hypot(ux, uy); ux, uy = ux / L, uy / L                      # direction, in points
    bx, by = x1 - ux * head_len * PT_X, y1 - uy * head_len * PT_Y        # base of the head
    fline([x0, x1 - ux * 0.8 * head_len * PT_X], [y0, y1 - uy * 0.8 * head_len * PT_Y])
    px, py = -uy * head_w / 2 * PT_X, ux * head_w / 2 * PT_Y
    axA.add_patch(Polygon([(x1, y1), (bx + px, by + py), (bx - px, by - py)], closed=True, fc=LINE_C, ec='none', zorder=1, clip_on=False))


# Rows, bottom to top, in inches: analysis steps (3 lines), bus, subphenotypes (2 lines), bus, latent profile analysis (3 lines),
# arrow, clustered children (2 lines), branch to the children not clustered, all children (2 lines).
R5, R4, R3, R2, R1 = (0.0, 0.68), (1.06, 1.54), (1.94, 2.62), (2.88, 3.36), (3.62, 4.1)
XM, XW = 0.30, 0.60                                                       # main column: center and width (axes units)
r1 = fbox(0.0, fy(R1[0]), XW, fy(R1[1]), f'Children with acute encephalopathy, first acute-phase admission, {yrs}\nn = {len(D)}',
          '#E3ECF7', BLUE)
r2 = fbox(0.0, fy(R2[0]), XW, fy(R2[1]), f'Clustered: n = {N}\n2019–2022: {n_e}  ·  2023–2026: {n_l}', '#E3ECF7', BLUE)
y_br = (r1[1] + r2[3]) / 2
r_ex = fbox(0.66, y_br - fy(0.36), 1.0, y_br + fy(0.36),
            f'Not clustered\nNeonates (< 28 days): {n_neo}\nFewer than 11 of 22 blood tests: {n_few}', '#F2F2F2', GREY)
r3 = fbox(0.0, fy(R3[0]), XW, fy(R3[1]), f'Latent profile analysis of {P} indicators at admission\n{n_lab} blood tests and {n_vit} vital signs\n'
          f'1–{K_MAX} classes; lowest BIC with every class ≥ {MIN_CLASS_SHARE:.0%}', '#FBF1D3', GOLD)
farrow(XM, r1[1], XM, r2[3]); farrow(XM, y_br, r_ex[0], y_br); farrow(XM, r2[1], XM, r3[3])
GAP = 0.016                                                               # space between neighboring boxes in a row
cw = (1.0 - (K_STAR - 1) * GAP) / K_STAR
rs = [fbox(c * (cw + GAP), fy(R4[0]), c * (cw + GAP) + cw, fy(R4[1]), f'{CLS[c]}\n{SIZES[c]} ({100 * SIZES[c] / N:.1f}%)', CC[c], CC[c],
           fs=11, color=_txt_color(CC[c]), weight='bold') for c in range(K_STAR)]
xs_ = [(b[0] + b[2]) / 2 for b in rs]
y_bus1 = (r3[1] + rs[0][3]) / 2
fline([XM, XM], [r3[1], y_bus1]); fline([xs_[0], xs_[-1]], [y_bus1, y_bus1], cap='projecting')
for x_, b in zip(xs_, rs):
    farrow(x_, y_bus1, x_, b[3])
steps = [('Validation (H2)', 'cytokines and\noutcomes'), ('Reproduction (H3)', 'two periods\nfitted apart'),
         ('Beyond severity (H4)', 'added to age and\nconsciousness'),
         ('Robustness', f'{N_BOOT_STABILITY} bootstraps,\n{len(SENS)} sensitivity analyses')]
sw = (1.0 - (len(steps) - 1) * GAP) / len(steps)
ra = [fbox(i * (sw + GAP), fy(R5[0]), i * (sw + GAP) + sw, fy(R5[1]), f'{h}\n{t}', '#F8E1DE', RED) for i, (h, t) in enumerate(steps)]
xa_ = [(b[0] + b[2]) / 2 for b in ra]
y_bus2 = (rs[0][1] + ra[0][3]) / 2
for x_, b in zip(xs_, rs):
    fline([x_, x_], [b[1], y_bus2])
fline([min(xs_ + xa_), max(xs_ + xa_)], [y_bus2, y_bus2], cap='projecting')
for x_, b in zip(xa_, ra):
    farrow(x_, y_bus2, x_, b[3])

ks = FITS.classes.values
axB2 = axB.twinx()
axB2.bar(ks, FITS.smallest_class_pct.values, width=0.55, color=GOLD, alpha=0.55, zorder=1, label='Smallest class, %')
for k_, v_ in zip(ks, FITS.smallest_class_pct.values):
    axB2.text(k_, v_ + 3, f'{v_:.0f}' if v_ >= 99.95 else f'{v_:.1f}', ha='center', va='bottom', fontsize=10, color='#7A5B00')
axB2.axhline(100 * MIN_CLASS_SHARE, color=RED, ls='--', lw=1.3, label=f'{MIN_CLASS_SHARE:.0%} minimum class size')
axB2.set_ylim(0, 230); axB2.set_yticks([0, 50, 100]); axB2.set_ylabel('Smallest class, %'); axB2.spines['right'].set_visible(True)
axB2.yaxis.set_label_coords(1.17, 0.22)
axB.plot(ks, FITS.bic.values, '-o', color=BLUE, lw=2, ms=6, zorder=3, label='BIC')
axB.plot([K_STAR], [FITS.bic.values[K_STAR - 1]], 'o', ms=13, mfc='none', mec=RED, mew=2, zorder=4, label=f'Chosen: {K_STAR} classes')
rng_ = FITS.bic.max() - FITS.bic.min()
for k_, v_ in zip(ks, FITS.bic.values):
    inner = ks.min() < k_ < ks.max()                                      # labels of inner points sit up and to the right of the line
    axB.annotate(f'{v_ / 1000:.1f}', (k_, v_), xytext=(5, 6) if inner else (0, 8), textcoords='offset points',
                 ha='left' if inner else 'center', va='bottom', fontsize=10, color=BLUE)
axB.set_zorder(axB2.get_zorder() + 1); axB.patch.set_visible(False)
axB.set_xticks(ks); axB.set_xlabel('Number of classes'); axB.set_ylabel('BIC, thousands')
axB.set_ylim(FITS.bic.min() - 0.95 * rng_, FITS.bic.max() + 0.75 * rng_)
axB.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v / 1000:.0f}'))
h1_, l1_ = axB.get_legend_handles_labels(); h2_, l2_ = axB2.get_legend_handles_labels()
axB.legend(h1_ + h2_, l1_ + l2_, loc='upper right', fontsize=10, ncol=1)

vio = axC.violinplot([POST[MODAL == c, c] for c in range(K_STAR)], positions=range(K_STAR), widths=0.82, showextrema=False)
for c, b in enumerate(vio['bodies']):
    b.set_facecolor(CC[c]); b.set_edgecolor(CC[c]); b.set_alpha(0.8)
mpp = [float(POST[MODAL == c, c].mean()) for c in range(K_STAR)]
axC.scatter(range(K_STAR), mpp, s=30, color='white', edgecolor='black', zorder=3, label='Mean')
for c in range(K_STAR):
    axC.text(c, 1.015, f'{mpp[c]:.2f}', ha='center', va='bottom', fontsize=10.5)
axC.set_ylim(max(0, float(np.min(POST.max(1))) - 0.05), 1.09)
class_axis(axC, SIZES)
axC.set_ylabel('Posterior probability')
axC.legend(loc='lower left', fontsize=10.5)
panel_letters(fig, [(axA, 'A'), (axB, 'B'), (axC, 'C')])
save(fig, 'Figure1_design_derivation',
     f"Figure 1. Study design and derivation of the admission subphenotypes. (A) Children, indicators and analysis steps; numbers in "
     f"the subphenotype boxes are children (percentage of the {N} clustered children). (B) Bayesian information criterion (BIC; blue, "
     f"in thousands) and size of the smallest class (gold bars; children whose most likely class it is) for 1 to {K_MAX} classes. "
     f"The dashed red line marks the registered minimum class size of {MIN_CLASS_SHARE:.0%}; the circled point is the chosen "
     f"solution, the lowest BIC among solutions meeting the size rule. BIC still decreased at {K_MAX} classes, the registered maximum. "
     f"(C) Posterior probability of the assigned subphenotype (violins; white dots and numbers, means); numbers under the labels are "
     f"children per subphenotype. Subphenotypes are ordered by the registered inflammation and cell-injury score, from S1 (highest; "
     f"hyperinflammatory) to S{K_STAR} (lowest; least abnormal).")

In [ ]:
# Figure 2. Laboratory and vital-sign profiles.
W2, H2 = 10.0, 8.65
fig = plt.figure(figsize=(W2, H2))


def ax_in2(x, y, w, h):
    return fig.add_axes([x / W2, y / H2, w / W2, h / H2])


CELL_W, CELL_H = 0.33, 0.36                                               # heat-map cell, inches
HX0, HY0 = 1.0, 6.05                                                      # lower-left corner of the heat map, inches
axA = ax_in2(HX0, HY0, CELL_W * P, CELL_H * K_STAR)
cax = ax_in2(HX0 + CELL_W * P + 0.12, HY0, 0.14, CELL_H * K_STAR)
M = MAIN_MEANS                                                            # classes x indicators
vmax = float(np.ceil(np.abs(M).max() * 2) / 2)
im = axA.imshow(M, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
for i in range(K_STAR):
    for j in range(P):
        v = M[i, j]
        axA.text(j, i, '0.0' if abs(v) < 0.05 else minus(f'{v:.1f}'), ha='center', va='center', fontsize=10,
                 color='white' if abs(v) > 0.6 * vmax else 'black')
axA.set_yticks(range(K_STAR)); axA.set_yticklabels([f'{c} ({SIZES[i]})' for i, c in enumerate(CLS)])
for lab_, col_ in zip(axA.get_yticklabels(), CC):
    lab_.set_color(col_); lab_.set_fontweight('bold')
SHORT2 = {**SHORT, 'Heart rate (z-score for age)': 'Heart rate', 'Respiratory rate (z-score for age)': 'Respiratory rate'}
axA.set_xticks(range(P)); axA.set_xticklabels([SHORT2.get(LABEL[c], LABEL[c]) for c in FINAL], rotation=90, fontsize=10.5)
axA.tick_params(axis='both', length=0)
for s_ in axA.spines.values():
    s_.set_visible(False)
doms = [DOMAIN[c] for c in FINAL]
starts = [j for j in range(P) if j == 0 or doms[j] != doms[j - 1]]
DOM_SHORT = {'Inflammation and blood count': 'Inflammation, blood count', 'Charted vital signs': 'Vital signs'}
for j in starts[1:]:
    axA.axvline(j - 0.5, color='white', lw=3)
for k_, j0 in enumerate(starts):
    j1 = starts[k_ + 1] if k_ + 1 < len(starts) else P
    axA.annotate('', xy=(j0 - 0.42, -0.72), xytext=(j1 - 0.58, -0.72), annotation_clip=False,
                 arrowprops=dict(arrowstyle='-', color='#555555', lw=1.1))
    axA.text((j0 + j1 - 1) / 2, -0.85, DOM_SHORT.get(doms[j0], doms[j0]), ha='center', va='bottom', fontsize=10.5, color='#333333')
cb = fig.colorbar(im, cax=cax)
cb.set_label('Standardized mean, SD', fontweight='bold'); cb.outline.set_visible(False)
cb.set_ticks(np.arange(-np.floor(vmax), np.floor(vmax) + 0.1, 1))
cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: minus(f'{v:g}')))
KEY = [c for c in ['ast', 'inr', 'd_dimer', 'platelets', 'procalcitonin', 'hs_crp', 'neutrophils_abs', 'lymphocytes_pct'] if c in FINAL]
BW, BH, BX0, BDX, BY = 1.62, 1.72, 0.8, 2.42, [2.62, 0.48]               # box-plot panels, inches
axesB = []
for n_, col in enumerate(KEY):
    ax = ax_in2(BX0 + BDX * (n_ % 4), BY[n_ // 4], BW, BH); axesB.append(ax)
    box_panel(ax, col, rotate=True)
    ax.yaxis.label.set_size(11)
for ax in axesB[4:]:
    ax.set_xlabel('Subphenotype')
panel_letters(fig, [(axA, 'A'), (axesB[0], 'B')])
save(fig, 'Figure2_profiles',
     f"Figure 2. Laboratory and vital-sign profiles of the {NUM_WORDS.get(K_STAR, K_STAR)} subphenotypes at admission. (A) Model-estimated mean of each of the "
     f"{P} clustering indicators by subphenotype, in SD units of the {N} clustered children (0 = mean of all clustered children); "
     f"laboratory values were divided by their reference limit and log-transformed where right-skewed before standardization, and "
     f"heart and respiratory rates are z-scores for age. Numbers in parentheses are children per subphenotype. (B) Eight indicators "
     f"that separate the subphenotypes, in original units, by most likely subphenotype: boxes, interquartile range; lines, median "
     f"(value above each box); whiskers, 5th to 95th percentile. Logarithmic axes except lymphocyte percentage. FEU, "
     f"fibrinogen-equivalent units.")

In [ ]:
# Figure 3. Host-response biomarkers and outcomes (H2 and secondary validators).
W3, H3_ = 10.0, 6.75
fig = plt.figure(figsize=(W3, H3_))
PW, PH, PX, PY = 2.5, 2.2, [0.92, 4.2, 7.45], [3.95, 0.82]               # panels, inches
axes = np.array([[fig.add_axes([x / W3, y / H3_, PW / W3, PH / H3_]) for x in PX] for y in PY])
for ax, (nm, yl) in zip(axes[0], [('IL-6', 'IL-6, pg/mL'), ('IL-10', 'IL-10, pg/mL'), ('Ferritin', 'Ferritin, ng/mL')]):
    dot_panel(ax, nm, yl)
for ax, (nm, yl) in zip(axes[1], [('Death or critical discharge', 'Death or critical\ndischarge, %'),
                                  ('Invasive ventilation', 'Invasive ventilation, %'), ('Vasoactive infusion', 'Vasoactive infusion, %')]):
    bar_panel(ax, nm, yl)
for ax in axes[0]:
    ax.set_xlabel('')
panel_letters(fig, list(zip(axes.ravel(), 'ABCDEF')))
_fer = CONTRAST.set_index('validator')
save(fig, 'Figure3_biomarkers_outcomes',
     f"Figure 3. Host-response biomarkers and outcomes by subphenotype. Bias-corrected three-step (BCH) estimates with 95% "
     f"bootstrap CIs ({len(BOOT)} resamples; each refits the latent profile model). (A–C) Geometric means of the first IL-6, IL-10 "
     f"and ferritin values within 24 h before to 48 h after admission (logarithmic axes; numbers in parentheses, children measured). "
     f"(D–F) Death or critical discharge against medical advice (adjudicated), invasive mechanical ventilation and vasoactive "
     f"infusion during the index admission (numbers in parentheses, children per subphenotype). Registered tests (H2), "
     f"hyperinflammatory S1 versus least abnormal S{K_STAR}: ratio of geometric means for IL-6 {h2_text('IL-6')}; for IL-10 "
     f"{h2_text('IL-10')}; risk ratio for death or critical discharge {h2_text('Death or critical discharge')}. P values are two-sided "
     f"percentile-bootstrap P values. Ferritin, ratio of geometric means {_fer.loc['Ferritin', 'estimate']:.1f} (95% CI "
     f"{_fer.loc['Ferritin', 'ci_low']:.1f} to {_fer.loc['Ferritin', 'ci_high']:.1f}; secondary validator, no test).")

In [ ]:
# Figure 4. Neurological presentation and the ITES syndromes.
W4, H4_ = 10.0, 9.3
fig = plt.figure(figsize=(W4, H4_))


def ax_in4(x, y, w, h):
    return fig.add_axes([x / W4, y / H4_, w / W4, h / H4_])


axA = ax_in4(0.1, 4.85, 9.8, 4.0)
_ites = DC.ites_class.astype(str)
SMALL = [c for c, n in _ites.value_counts().items() if n < 15 and not c.startswith('Non-ITES')]
_ites = _ites.where(~_ites.isin(SMALL), ', '.join(sorted(SMALL)))
CX = pd.crosstab(_ites, pd.Series([CLS[c] for c in MODAL], index=CLU, name='class')).reindex(columns=CLS).fillna(0).astype(int)
bary = (CX.values * np.arange(K_STAR)).sum(1) / CX.values.sum(1)
CX = CX.iloc[np.argsort(bary, kind='stable')]
NAMES = {'Non-ITES: alternative diagnosis': 'Non-ITES, other diagnosis', 'Non-ITES: encephalitis/meningitis': 'Non-ITES, encephalitis or meningitis',
         'SAE': 'SAE', 'Possible ITES': 'Possible ITES', 'Unclassifiable': 'Unclassifiable'}
gap, tot = 0.014, CX.values.sum()
scale_l = (1 - gap * (len(CX) - 1)) / tot
scale_r = (1 - gap * (K_STAR - 1)) / tot
lpos, y = {}, 1.0
for nm, n in CX.sum(1).items():
    lpos[nm] = [y, y - n * scale_l]; y -= n * scale_l + gap
rpos, y = {}, 1.0
for j in range(K_STAR):
    n = CX.iloc[:, j].sum(); rpos[j] = [y, y - n * scale_r]; y -= n * scale_r + gap
xl, xr, wn = 0.335, 0.80, 0.022
lcur = {nm: lpos[nm][0] for nm in lpos}
rcur = {j: rpos[j][0] for j in rpos}
for nm in CX.index:
    for j in range(K_STAR):
        n = CX.loc[nm, CLS[j]]
        if not n:
            continue
        y0t, y0b = lcur[nm], lcur[nm] - n * scale_l; lcur[nm] = y0b
        y1t, y1b = rcur[j], rcur[j] - n * scale_r; rcur[j] = y1b
        xa, xb = xl + wn, xr
        xm = (xa + xb) / 2
        verts = [(xa, y0t), (xm, y0t), (xm, y1t), (xb, y1t), (xb, y1b), (xm, y1b), (xm, y0b), (xa, y0b), (xa, y0t)]
        codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.CLOSEPOLY]
        axA.add_patch(PathPatch(MPath(verts, codes), fc=CC[j], ec='none', alpha=0.55))
for nm, (t_, b_) in lpos.items():
    axA.add_patch(Rectangle((xl, b_), wn, t_ - b_, fc='#6E6E6E', ec='none'))
    axA.text(xl - 0.008, (t_ + b_) / 2, f'{NAMES.get(nm, nm)} ({int(CX.loc[nm].sum())})', ha='right', va='center', fontsize=10.5)
for j, (t_, b_) in rpos.items():
    axA.add_patch(Rectangle((xr, b_), wn, t_ - b_, fc=CC[j], ec='none'))
    axA.text(xr + wn + 0.008, (t_ + b_) / 2, f'{CLS[j]} ({int(CX.iloc[:, j].sum())})', ha='left', va='center', fontsize=11,
             fontweight='bold', color=CC[j])
axA.text(xl + wn / 2, 1.02, 'ITES class', ha='center', va='bottom', fontsize=12, fontweight='bold')
axA.text(xr + wn / 2, 1.02, 'Subphenotype', ha='center', va='bottom', fontsize=12, fontweight='bold')
axA.set_xlim(0, 1); axA.set_ylim(-0.005, 1.0); axA.axis('off')

# B. Neurological features (BCH estimates, %), as tall as panel C: same bottom, top and color-bar height.
NEURO = [('Consciousness: coma', 'Coma at\nadmission'), ('Consciousness: stupor', 'Stupor at\nadmission'),
         ('Abnormal pupillary light reflex', 'Abnormal\npupil reflex'), ('Status epilepticus', 'Status\nepilepticus'),
         ('First EEG: diffuse slowing or worse', 'EEG: slowing\nor worse'), ('First EEG: low voltage or suppressed', 'EEG: low\nvoltage')]
Y0_BC, H_BC = 0.72, 3.05                       # bottom and height of panels B and C
axB = ax_in4(1.025, Y0_BC, 0.48 * K_STAR, H_BC)                 # row labels start at 0.1 in, under the letter of panel A
caxB = ax_in4(1.025 + 0.48 * K_STAR + 0.1, Y0_BC, 0.12, H_BC)
NM = np.array([display_values(k)[0] for k, _ in NEURO])
im = axB.imshow(NM, cmap=CMAP_RED, vmin=0, vmax=100, aspect='auto')
for i in range(len(NEURO)):
    for j in range(K_STAR):
        axB.text(j, i, f'{NM[i, j]:.0f}', ha='center', va='center', fontsize=12, color='white' if NM[i, j] > 60 else 'black')
axB.set_yticks(range(len(NEURO))); axB.set_yticklabels([l for _, l in NEURO], fontsize=10.5, linespacing=1.15)
axB.xaxis.tick_top(); axB.set_xticks(range(K_STAR)); axB.set_xticklabels(CLS)
for lab_, col_ in zip(axB.get_xticklabels(), CC):
    lab_.set_color(col_); lab_.set_fontweight('bold')
axB.tick_params(axis='both', length=0)
for s_ in axB.spines.values():
    s_.set_visible(False)
cb = fig.colorbar(im, cax=caxB); cb.set_label('Children, %', fontweight='bold'); cb.outline.set_visible(False)
cb.set_ticks([0, 50, 100])

# C. Death or critical discharge by subphenotype within the ITES categories (exploratory analysis E1, not registered).
axC = ax_in4(6.4, Y0_BC, 0.45 * K_STAR, H_BC)
caxC = ax_in4(6.4 + 0.45 * K_STAR + 0.1, Y0_BC, 0.12, H_BC)
bandC = ax_in4(6.4, Y0_BC + H_BC + 0.08, 0.45 * K_STAR, 0.42)
for i, g in enumerate(ITES_GROUPS):
    for j in range(K_STAR):
        m_ = (ITES_GRP == g) & (LAB_NAMES == CLS[j])
        n_, e_ = int(m_.sum()), int(Y_POOR[m_].sum())
        if not n_:
            continue
        s_ = area(n_)
        axC.scatter(j, i, s=s_, color=CMAP_RED(e_ / n_), edgecolor=CC[j], linewidth=1.6, zorder=3)
        axC.annotate(f'{e_}/{n_}', (j, i), xytext=(0, -(math.sqrt(s_) / 2 + 2.5)), textcoords='offset points', ha='center', va='top',
                     fontsize=10)
axC.set_yticks(range(len(ITES_GROUPS))); axC.set_yticklabels(ITES_GROUPS, fontsize=10.5)
axC.set_ylim(len(ITES_GROUPS) - 0.42, -0.5)
class_axis(axC, None)
axC.tick_params(axis='y', length=0)
axC.spines['left'].set_visible(False)
smC = plt.cm.ScalarMappable(cmap=CMAP_RED, norm=plt.Normalize(0, 100))
cbC = fig.colorbar(smC, cax=caxC); cbC.set_label('Death or critical discharge, %', fontweight='bold'); cbC.outline.set_visible(False)
cbC.set_ticks([0, 50, 100])
size_key(bandC)
panel_letters(fig, [(axA, 'A'), (axB, 'B', bandC), (axC, 'C', bandC)])     # B and C letters on one line
_e1 = E1_MODEL
save(fig, 'Figure4_neurology_ites',
     f"Figure 4. Subphenotypes across the infection-triggered encephalopathy syndromes (ITES): neurological features and outcomes. "
     f"(A) Algorithmic ITES class (left) of the {N} clustered children and their most likely subphenotype (right); band widths are "
     f"proportional to the number of children (in parentheses); classes with fewer than 15 children ({', '.join(sorted(SMALL))}) are "
     f"shown together. (B) Neurological features by subphenotype: BCH-estimated percentage of children with the feature recorded "
     f"(consciousness and pupillary light reflex at admission; background of the first EEG). (C) Death or critical discharge "
     f"(events/children) by subphenotype within ITES categories; circle area, number of children; fill, event percentage (exploratory "
     f"analysis not in the registration). The hyperinflammatory subphenotype had a Mantel–Haenszel risk ratio of {RR_MH:.2f} (95% CI "
     f"{RR_MH_CI[0]:.2f} to {RR_MH_CI[1]:.2f}) versus the other subphenotypes across categories. Adding subphenotype to age, "
     f"consciousness and ITES category: likelihood-ratio chi-square {_e1['lr_stat']:.1f} on {_e1['df']} df, parametric-bootstrap "
     f"P = {_e1['p_parametric_bootstrap']:.3f} ({_e1['simulations']:,} simulations); C statistic {_e1['c_base']:.3f} to "
     f"{_e1['c_full']:.3f} (change {_e1['delta_c']:.3f}, 95% bootstrap CI {_e1['delta_c_ci'][0]:.3f} to {_e1['delta_c_ci'][1]:.3f}). "
     f"ANE, acute necrotizing encephalopathy; SAE, sepsis-associated encephalopathy; AFCE, acute fulminant cerebral edema; ASEM, acute "
     f"shock with encephalopathy and multiorgan failure; AESD, acute encephalopathy with biphasic seizures and late reduced diffusion; "
     f"MERS, mild encephalopathy with a reversible splenial lesion; FIRES, febrile infection-related epilepsy syndrome; HHE, "
     f"hemiconvulsion-hemiplegia-epilepsy syndrome. Other or possible ITES: AESD, MERS, FIRES, HHE and possible ITES.")

In [ ]:
# Figure 5. Prognostic information beyond age and consciousness (H4).
keep_ = DC.cons4.notna().values
Dk_, lab_k = DC[keep_], LAB_NAMES[keep_]
y_k = b01(Dk_['death_or_dama_critical_final'])
base_, full_, _ = h4_design(Dk_, lab_k, CLS[LEAST])
m0_, _ = logit_fit(y_k, base_); m1_, _ = logit_fit(y_k, full_)
W5, H5 = 10.0, 8.25
fig = plt.figure(figsize=(W5, H5))


def ax_in(x, y, w, h):
    return fig.add_axes([x / W5, y / H5, w / W5, h / H5])


axA = ax_in(1.45, 4.85, 6.9, 2.45)                                          # A: consciousness x subphenotype
cax = ax_in(8.5, 4.85, 0.12, 2.45)
axL = ax_in(1.45, 7.38, 6.9, 0.42)                                          # band above A: key to circle sizes
axB = ax_in(0.95, 0.78, 2.75, 3.1)                                          # B: adjusted odds ratios
axC = ax_in(6.45, 0.78, 3.1, 3.1)                                           # C: ROC curves
levels = ['alert', 'altered', 'stupor', 'coma']
for i, lev in enumerate(levels):
    for j in range(K_STAR):
        m_ = (Dk_.cons4.values == lev) & (lab_k == CLS[j])
        n_, e_ = int(m_.sum()), int(y_k[m_].sum())
        if not n_:
            continue
        s_ = area(n_)
        axA.scatter(j, i, s=s_, color=CMAP_RED(e_ / n_), edgecolor=CC[j], linewidth=1.6, zorder=3)
        axA.annotate(f'{e_}/{n_}', (j, i), xytext=(0, -(math.sqrt(s_) / 2 + 2.5)), textcoords='offset points', ha='center', va='top',
                     fontsize=10)
axA.set_yticks(range(4)); axA.set_yticklabels([l.capitalize() for l in levels]); axA.set_ylim(3.55, -0.42)
class_axis(axA, None); axA.set_ylabel('Consciousness\nat admission')
axA.tick_params(axis='y', length=0); axA.spines['left'].set_visible(False)
sm_ = plt.cm.ScalarMappable(cmap=CMAP_RED, norm=plt.Normalize(0, 100))
cb = fig.colorbar(sm_, cax=cax); cb.set_label('Death or critical\ndischarge, %', fontweight='bold')
cb.outline.set_visible(False); cb.set_ticks([0, 50, 100])
size_key(axL)
ci_ = m1_.conf_int()
rows_ = [c for c in range(K_STAR) if c != LEAST]
for i, c in enumerate(rows_):
    nm = f'class_{CLS[c]}'
    e_, n_ = int(y_k[lab_k == CLS[c]].sum()), int((lab_k == CLS[c]).sum())
    if 0 < e_ < n_:
        o, lo_, hi_ = np.exp(m1_.params[nm]), np.exp(ci_.loc[nm, 0]), np.exp(ci_.loc[nm, 1])
        axB.plot([lo_, hi_], [i, i], color=CC[c], lw=2.2, zorder=2)
        axB.scatter(o, i, s=55, color=CC[c], edgecolor='white', zorder=3)
        axB.text(1.04, i, f'{o:.1f} ({lo_:.1f}–{hi_:.1f})', transform=axB.get_yaxis_transform(), va='center', fontsize=10.5)
    else:
        axB.text(1.04, i, 'No events', transform=axB.get_yaxis_transform(), va='center', fontsize=10.5, color='#555555')
axB.axvline(1, color=GREY, ls='--', lw=1.1, label=f'Odds ratio 1\n(reference {CLS[LEAST]})')
axB.set_xscale('log'); log_axis(axB, 'x')
axB.set_yticks(range(len(rows_))); axB.set_yticklabels([CLS[c] for c in rows_]); axB.set_ylim(len(rows_) - 0.4, -0.6)
for lab_, c in zip(axB.get_yticklabels(), rows_):
    lab_.set_color(CC[c]); lab_.set_fontweight('bold')
axB.set_xlabel('Adjusted odds ratio'); axB.set_ylabel('Subphenotype')
axB.legend(loc='lower right', fontsize=10)
for mdl_, X_, col_, lab__ in [(m0_, base_, BLUE, 'Age and consciousness'), (m1_, full_, RED, '+ Subphenotype')]:
    fpr, tpr, _ = roc_curve(y_k, mdl_.predict(X_))
    axC.plot(fpr, tpr, color=col_, lw=2, label=lab__)
axC.plot([0, 1], [0, 1], color=GREY, ls='--', lw=1.1, label='Chance')
axC.set_xlim(-0.02, 1.02); axC.set_ylim(-0.02, 1.02)
axC.set_xticks(np.linspace(0, 1, 6)); axC.set_yticks(np.linspace(0, 1, 6))
axC.set_xlabel('1 − Specificity'); axC.set_ylabel('Sensitivity'); axC.legend(loc='lower right', fontsize=10)
panel_letters(fig, [(axA, 'A', axL), (axB, 'B'), (axC, 'C')])
_pb, _pbs = H4.get('p_parametric_bootstrap'), ''
if _pb is not None:
    _ns = H4_PB['simulations']
    _pbs = (f"; parametric-bootstrap P = {_pb:.3f} ({_ns:,} simulations" + ("; the smallest attainable value" if np.isclose(_pb, 1 / (_ns + 1))
                                                                         else '') + ")")
save(fig, 'Figure5_beyond_severity',
     f"Figure 5. Prognostic information of the subphenotypes beyond age and consciousness (H4). Children with a recorded "
     f"consciousness level: {H4['n']}, with {H4['events']} deaths or critical discharges. (A) Events/children by consciousness level at "
     f"admission and subphenotype; circle area, number of children; fill, event percentage. (B) Odds ratios for death or critical "
     f"discharge by subphenotype (reference {CLS[LEAST]}, least abnormal), adjusted for age (restricted cubic spline, 3 knots) and "
     f"consciousness (4 levels), with 95% CIs. A subphenotype without events has no finite odds ratio. (C) Receiver operating "
     f"characteristic curves of the model with age and consciousness (C statistic {H4['c_base']:.3f}) and with subphenotype added "
     f"({H4['c_full']:.3f}); change {H4['delta_c']:.3f} (95% bootstrap CI {H4['delta_c_ci'][0]:.3f} to {H4['delta_c_ci'][1]:.3f}; "
     f"{N_BOOT_H4} resamples, both models refitted). Likelihood-ratio test for adding subphenotype: chi-square {H4['lr_stat']:.1f} on "
     f"{H4['df']} df, P = {sci(H4['p_value'])}{_pbs}.")

In [ ]:
# Figure 6. Robustness and reproduction across periods (H3).
W6, H6 = 10.0, 7.8
fig = plt.figure(figsize=(W6, H6))


def ax_in6(x, y, w, h):
    return fig.add_axes([x / W6, y / H6, w / W6, h / H6])


SA = SENS_AGREE.copy()
R = SA[[f'r_{c}' for c in CLS]].values.astype(float)
KAP = SA.kappa_vs_main.values.astype(float)
Mx = np.column_stack([R, KAP])
CWA, CHA = 0.78, 0.3                                                      # heat-map cell of panel A, inches
axA = ax_in6(2.3, 3.8, CWA * Mx.shape[1], CHA * Mx.shape[0])
caxA = ax_in6(2.3 + CWA * Mx.shape[1] + 0.12, 3.8, 0.12, CHA * Mx.shape[0])
im = axA.imshow(np.where(np.isnan(Mx), np.nan, Mx), cmap=CMAP_BLUE, vmin=0, vmax=1, aspect='auto')
for i in range(Mx.shape[0]):
    for j in range(Mx.shape[1]):
        v = Mx[i, j]
        axA.text(j, i, '–' if np.isnan(v) else f'{v:.2f}', ha='center', va='center', fontsize=10.5,
                 color='white' if (not np.isnan(v) and v > 0.7) else 'black')
axA.axvline(K_STAR - 0.5, color='white', lw=4)
SHORT_SENS_P = {'Worst value in the window': 'Worst value', 'Window 12 h before to 24 h after admission, own admission only':
                'Window −12 to +24 h', 'Without winsorizing': 'Without winsorizing',
                'Leaving out children with a non-ITES class': 'Without non-ITES children',
                'Leaving out children transferred from another hospital': 'Without transferred children',
                'Vital signs from the admission examination': 'Vital signs from examination',
                'Onset of the acute illness 24 h or less before admission': 'Onset ≤ 24 h',
                'Onset of the acute illness more than 24 h before admission': 'Onset > 24 h',
                'Additional (not registered): white cell count kept instead of neutrophil count': 'White cells for neutrophils*',
                'Gaussian model with full covariance': 'Full covariance', 'Consensus k-means after multiple imputation': 'Consensus k-means'}
axA.set_yticks(range(len(SA)))
axA.set_yticklabels([f"{SHORT_SENS_P.get(a, a)}" + (f" ({int(k_)} classes)" if k_ != K_STAR else '') for a, k_ in zip(SA.analysis, SA.classes)],
                    fontsize=10.5)
axA.xaxis.tick_top(); axA.set_xticks(range(K_STAR + 1)); axA.set_xticklabels(CLS + ['κ'])
for lab_, col_ in zip(axA.get_xticklabels(), CC + ['black']):
    lab_.set_color(col_); lab_.set_fontweight('bold')
axA.tick_params(axis='both', length=0)
for s_ in axA.spines.values():
    s_.set_visible(False)
cb = fig.colorbar(im, cax=caxA); cb.set_label('Profile correlation r or κ', fontweight='bold'); cb.outline.set_visible(False)
cb.set_ticks([0, 0.5, 1])

CP = 0.42                                                                 # heat-map cell of the period panels, inches
TOPP = 0.8 + CP * 5                                                      # period panels hang from a common top edge
axB = ax_in6(0.62, TOPP - CP * max(2, H3['k_early']), CP * K_STAR, CP * max(2, H3['k_early']))
axC = ax_in6(3.85, TOPP - CP * max(2, H3['k_late']), CP * K_STAR, CP * max(2, H3['k_late']))
caxP = ax_in6(3.85 + CP * K_STAR + 0.12, TOPP - CP * max(2, H3['k_late']), 0.12, CP * max(2, H3['k_late']))
axD = ax_in6(8.15, 0.8, 1.7, 2.1)


def period_panel(ax, res, k, name):
    if k < 2:
        ax.text(0.5, 0.5, f'{name}: 1 class chosen', ha='center', va='center', transform=ax.transAxes); ax.axis('off')
        return None
    mu = res[k]['params'][1]
    r = np.corrcoef(np.vstack([mu, MAIN_MEANS]))[:k, k:]
    best = r.argmax(1)
    order = np.lexsort((-r.max(1), best))
    r, best = r[order], best[order]
    im_ = ax.imshow(r, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    for i in range(k):
        for j in range(K_STAR):
            ax.text(j, i, minus(f'{r[i, j]:.2f}'), ha='center', va='center', fontsize=10, color='white' if abs(r[i, j]) > 0.7 else 'black')
        ax.add_patch(Rectangle((best[i] - 0.5, i - 0.5), 1, 1, fill=False, ec='black', lw=1.8))
    ax.set_yticks(range(k)); ax.set_yticklabels([str(i + 1) for i in range(k)])
    ax.set_ylabel(f'{name} class')
    ax.set_xticks(range(K_STAR)); ax.set_xticklabels(CLS)
    for lab_, col_ in zip(ax.get_xticklabels(), CC):
        lab_.set_color(col_); lab_.set_fontweight('bold')
    ax.set_xlabel('Main subphenotype'); ax.tick_params(axis='both', length=0)
    for s_ in ax.spines.values():
        s_.set_visible(False)
    return im_


imB = period_panel(axB, H3['res_early'], H3['k_early'], '2019–2022')
imC = period_panel(axC, H3['res_late'], H3['k_late'], '2023–2026')
if imC is not None:
    cb = fig.colorbar(imC, cax=caxP); cb.set_label('Profile correlation r', fontweight='bold'); cb.outline.set_visible(False)
    cb.set_ticks([-1, 0, 1]); cb.ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: minus(f'{v:g}')))
else:
    caxP.axis('off')
axD.hist(STAB_ARI, bins=16, color=SKY, edgecolor='white', alpha=0.95)
axD.axvline(np.median(STAB_ARI), color=RED, ls='--', lw=1.5, label=f'Median\n{np.median(STAB_ARI):.2f}')
axD.set_xlabel('Adjusted Rand index'); axD.set_ylabel('Bootstrap refits')
axD.set_ylim(0, axD.get_ylim()[1] * 1.3)
axD.set_xlim(min(0.25, float(np.min(STAB_ARI)) - 0.02), 1.0); axD.set_xticks([0.4, 0.6, 0.8, 1.0])   # room for the legend
axD.legend(loc='upper right', fontsize=10, handlelength=1.2)
panel_letters(fig, [(axA, 'A'), (axB, 'B'), (axC, 'C'), (axD, 'D')])
n_same = int((STAB_K == K_STAR).sum())
k_other = {int(k): int((STAB_K == k).sum()) for k in sorted(set(STAB_K.tolist())) if k != K_STAR}
save(fig, 'Figure6_robustness_reproduction',
     f"Figure 6. Robustness of the subphenotypes and their reproduction in two periods. (A) Sensitivity analyses that refit the classes: "
     f"correlation (r) between the class mean profile of each main subphenotype and that of its matched class (Hungarian assignment on "
     f"profile correlation; for analyses with fewer classes, the main subphenotype closest to each of their classes), and Cohen's "
     f"κ between the main and the sensitivity assignment of the same children when the number of classes is equal (–, not "
     f"applicable). *Additional, not registered. (B, C) Models fitted separately in 2019–2022 ({H3['n_early']} children; "
     f"{H3['k_early']} classes) and 2023–2026 ({H3['n_late']}; {H3['k_late']} classes): correlation of each period class profile with "
     f"the main subphenotypes (boxes, best match). H3 (same number of classes and κ ≥ 0.6 between the two period models on the "
     f"2023–2026 children) was " + ('met' if H3['met'] else 'not met') +
     (f" (κ {H3['kappa']:.2f})" if np.isfinite(H3['kappa']) else ' (different numbers of classes)') +
     f". (D) Adjusted Rand index between the main assignment and the assignment of the {N} children by each of {len(STAB)} "
     f"bootstrap refits (median {np.median(STAB_ARI):.2f}; IQR {np.percentile(STAB_ARI, 25):.2f} to "
     f"{np.percentile(STAB_ARI, 75):.2f}). The refits chose {K_STAR} classes in {n_same} of {len(STAB)}"
     + (''.join(f", {k} classes in {v}" for k, v in k_other.items())) + ".")

### Supplementary figures

In [ ]:
# Figure S1. Information criteria and classification quality by number of classes.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(10, 4.3))
for col, c_, lab__ in [('bic', BLUE, 'BIC'), ('aic', RED, 'AIC'), ('sabic', GOLD, 'Sample-size-adjusted BIC')]:
    axA.plot(FITS.classes, FITS[col], '-o', color=c_, lw=2, ms=6, label=lab__)
axA.axvline(K_STAR, color=GREY, ls='--', lw=1.1, label=f'Chosen: {K_STAR} classes')
axA.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
axA.set_xlabel('Number of classes'); axA.set_ylabel('Information criterion'); axA.legend(fontsize=10)
fk = FITS[FITS.classes > 1]
axB.plot(fk.classes, fk.relative_entropy, '-o', color=TEAL, lw=2, ms=6, label='Relative entropy')
axB.plot(fk.classes, fk.min_mean_posterior, '-s', color=ORANGE, lw=2, ms=6, label='Lowest mean posterior\nprobability of a class')
for k_, t_, o_ in zip(fk.classes, fk.relative_entropy, fk.min_mean_posterior):
    up_o = o_ >= t_                                                       # the upper line's label goes above, the lower one's below
    axB.annotate(f'{o_:.2f}', (k_, o_), xytext=(0, 6 if up_o else -6), textcoords='offset points', ha='center',
                 va='bottom' if up_o else 'top', fontsize=10, color=ORANGE)
    axB.annotate(f'{t_:.2f}', (k_, t_), xytext=(0, -6 if up_o else 6), textcoords='offset points', ha='center',
                 va='top' if up_o else 'bottom', fontsize=10, color=TEAL)
axB.set_ylim(min(fk.min_mean_posterior.min(), fk.relative_entropy.min()) - 0.05, 1.01)
axB.set_xticks(fk.classes); axB.set_xlabel('Number of classes'); axB.set_ylabel('Value'); axB.legend(fontsize=10, loc='lower left')
fig.tight_layout(w_pad=3)
panel_letters(fig, [(axA, 'A'), (axB, 'B')])
save(fig, 'FigureS1_information_criteria',
     "Figure S1. Fit and classification quality by number of classes. Latent profile models with 1 to 6 classes (StepMix 3.0.0, "
     "Gaussian with diagonal covariance, missing indicators by full-information likelihood, best of 100 random starts). (A) Akaike and Bayesian information "
     "criteria and the sample-size-adjusted BIC (Sclove); the dashed line marks the chosen solution. (B) Relative entropy and the lowest "
     "mean posterior probability among the classes (children assigned to their most likely class).")

In [ ]:
# Figure S2. All clustering indicators in original units.
cols_ = [c for c in IND if c in LABVAL.columns and c in FINAL] + ['temp_c', 'heart_rate', 'resp_rate']
nr = int(np.ceil(len(cols_) / 4))
fig, axes = plt.subplots(nr, 4, figsize=(10, 2.2 * nr))
for ax, col in zip(axes.ravel(), cols_):
    box_panel(ax, col, show_labels=True, rotate=True)
    if col == 'resp_rate':
        ax.set_ylabel('Respiratory rate, /min')
    ax.yaxis.label.set_size(11)
for ax in axes.ravel()[len(cols_):]:
    ax.axis('off')
fig.tight_layout(h_pad=1.2, w_pad=0.8)
panel_letters(fig, [(ax, chr(65 + i)) for i, ax in enumerate(axes.ravel()[:len(cols_)])])
save(fig, 'FigureS2_indicators_original_units',
     f"Figure S2. Clustering indicators in original units by most likely subphenotype. First value within 24 h before to 24 h after "
     f"admission; vital signs, first charted value within 24 h after admission (heart and respiratory rates entered the model as "
     f"z-scores for age). Boxes, interquartile range; lines, median (value above each box); whiskers, 5th to 95th percentile. "
     f"Logarithmic axes for indicators modeled on the log scale.")

In [ ]:
# Figure S3. Secondary validators and outcomes.
SEC = [('TNF-alpha', 'TNF-α, pg/mL', 'dot'), ('CSF protein', 'CSF protein, g/L', 'dot'), ('CSF white cells', r'CSF white cells, $10^6$/L', 'dot'),
       ('Length of stay', 'Length of stay, days', 'dot'), ('Death', 'Death, %', 'bar'), ('Critical discharge against advice', 'Critical discharge, %', 'bar'),
       ('PICU stay', 'PICU stay, %', 'bar'), ('Abnormal pupillary light reflex', 'Abnormal pupils, %', 'bar')]
fig, axes = plt.subplots(4, 2, figsize=(10, 11.5))
for ax, (nm, yl, kind) in zip(axes.ravel(), SEC):
    (dot_panel if kind == 'dot' else bar_panel)(ax, nm, yl)
fig.tight_layout(h_pad=1.6, w_pad=3.0)
panel_letters(fig, list(zip(axes.ravel(), 'ABCDEFGH')))
save(fig, 'FigureS3_secondary_validators',
     f"Figure S3. Secondary validators and outcomes by subphenotype. BCH estimates with 95% bootstrap CIs ({len(BOOT)} resamples). "
     f"(A–D) Geometric means (CSF white cells: geometric mean of count + 1, minus 1; length of stay in days; logarithmic axes). (E–H) "
     f"Percentages. Numbers in parentheses, children measured. A BCH proportion below 0 is set to 0 (StepMix).")

In [ ]:
# Figure S4. Treatments by subphenotype (descriptive, by most likely subphenotype).
TR = [('Methylprednisolone', H_MP), ('Immunoglobulin', H_IG), ('Tocilizumab', H_TCZ)]
fig, axes = plt.subplots(1, 3, figsize=(10, 4.4))
for ax, (nm, h) in zip(axes, TR):
    any_ = np.array([100 * np.mean(~np.isnan(h[MODAL == c])) for c in range(K_STAR)])
    early = np.array([100 * np.mean(h[MODAL == c] <= 24) for c in range(K_STAR)])
    ax.bar(range(K_STAR), any_, color=CC, width=0.66, alpha=0.35, zorder=2)
    ax.bar(range(K_STAR), early, color=CC, width=0.66, alpha=0.9, zorder=3)
    top = max(any_)
    for c in range(K_STAR):
        ax.text(c, any_[c] + 0.02 * top, f'{any_[c]:.0f}', ha='center', va='bottom', fontsize=10.5)
        if early[c] >= 0.14 * top:
            ax.text(c, early[c] / 2, f'{early[c]:.0f}', ha='center', va='center', fontsize=10, color=_txt_color(CC[c]), zorder=4)
    ax.set_ylim(0, top * 1.45); class_axis(ax, None); ax.set_ylabel(f'{nm}, %')
axes[0].legend([Rectangle((0, 0), 1, 1, fc=GREY, alpha=0.9), Rectangle((0, 0), 1, 1, fc=GREY, alpha=0.35)],
               ['First order within\n24 h of admission', 'First order later'], loc='upper right', fontsize=10)
fig.tight_layout(w_pad=1.5)
panel_letters(fig, list(zip(axes, 'ABC')))
_hmp = [np.nanmedian(H_MP[MODAL == c]) for c in range(K_STAR)]
save(fig, 'FigureS4_treatments',
     "Figure S4. Immunomodulatory treatments during the index admission by most likely subphenotype. Exploratory analysis E2 (not "
     "registered; descriptive). Dark bars and the numbers inside them, children with a first order within 24 h of admission (%); light "
     "bars, a first order later in the admission; numbers above the bars, any order during the admission (%). Median hours from "
     "admission to the first methylprednisolone order among treated children: "
     + ', '.join(f'{CLS[c]} {_hmp[c]:.0f}' for c in range(K_STAR)) + ". Treatment followed severity (confounding by indication), so no "
     "treatment effect is estimated.")

In [ ]:
# Figure S5. Main estimates across the sensitivity analyses.
rows_s = [{'analysis': 'Main analysis', 'classes': K_STAR, 'h1': H1_EFFECT, 'h2': H2_TAB.rename(columns={})[['validator', 'estimate', 'ci_low', 'ci_high']],
           'dc': H4.get('delta_c', np.nan)}]
for s in SENS:
    rows_s.append({'analysis': SHORT_SENS.get(s['analysis'], s['analysis'].replace('Reviewer-unsure critical discharges counted as not critical',
                                                                                  'Reviewer-unsure discharges not critical')),
                   'classes': s.get('classes'), 'h1': s.get('h1_effect_sd', np.nan), 'h2': s.get('h2', pd.DataFrame()), 'dc': s.get('h4_delta_c', np.nan)})
fig = plt.figure(figsize=(10, 2 * (0.26 * len(rows_s) + 1.05)))
gs5 = fig.add_gridspec(2, 3, wspace=0.1, hspace=0.38, left=0.27, right=0.985, top=0.97, bottom=0.08)
axes = [fig.add_subplot(gs5[0, 0])]
axes += [fig.add_subplot(gs5[0, 1], sharey=axes[0]), fig.add_subplot(gs5[0, 2], sharey=axes[0]),
         fig.add_subplot(gs5[1, 0], sharey=axes[0]), fig.add_subplot(gs5[1, 1], sharey=axes[0])]
axK = fig.add_subplot(gs5[1, 2]); axK.axis('off')
yy = np.arange(len(rows_s))
for i, r_ in enumerate(rows_s):
    col_ = RED if i == 0 else BLUE
    axes[0].scatter(r_['h1'], i, color=col_, s=36, zorder=3)
    for ax, v in zip(axes[1:4], PRIMARY):
        h = r_['h2']
        if len(h) and (h.validator == v).any():
            g = h[h.validator == v].iloc[0]
            lo_, hi_ = g.get('ci_low', np.nan), g.get('ci_high', np.nan)
            if pd.notna(lo_):
                ax.plot([lo_, min(hi_, 1e4) if np.isfinite(hi_) else 1e4], [i, i], color=col_, lw=1.8, zorder=2)
                if not np.isfinite(hi_):
                    ax.annotate('', xy=(1.25e4, i), xytext=(1e4, i), arrowprops=dict(arrowstyle='-|>', color=col_, lw=1.4))
            ax.scatter(g.estimate, i, color=col_, s=36, zorder=3)
    axes[4].scatter(r_['dc'], i, color=col_, s=36, zorder=3)
axes[0].axvline(0.5, color=GREY, ls='--', lw=1.1)
for ax in axes[1:4]:
    ax.axvline(1, color=GREY, ls='--', lw=1.1); ax.set_xscale('log'); log_axis(ax, 'x')
axes[4].axvline(0, color=GREY, ls='--', lw=1.1)
_ylab = [f"{r_['analysis']}" + (f" ({int(r_['classes'])} classes)" if r_['classes'] not in (K_STAR, None) else '') for r_ in rows_s]
for ax in (axes[0], axes[3]):
    ax.set_yticks(yy); ax.set_yticklabels(_ylab, fontsize=10.5)
for ax in (axes[1], axes[2], axes[4]):
    ax.tick_params(axis='y', labelleft=False)
axes[0].set_ylim(len(rows_s) - 0.5, -0.5)
for ax, xl_ in zip(axes, ['H1 effect, SD', 'IL-6 ratio', 'IL-10 ratio', 'Risk ratio, death or\ncritical discharge', 'Change in C statistic']):
    ax.set_xlabel(xl_)
axes[3].set_xlim(0.5, 2.5e4); axes[3].set_xticks([1, 10, 100, 1000, 10000])
axes[3].xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}' if v < 10000 else ''))
axes[4].xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.2f}'))
axes[4].set_xticks([0, 0.04, 0.08])
axK.legend([Line2D([], [], marker='o', ls='-', color=RED, ms=6), Line2D([], [], marker='o', ls='-', color=BLUE, ms=6),
            Line2D([], [], ls='--', color=GREY, lw=1.1), Line2D([], [], ls='--', color=GREY, lw=1.1), Line2D([], [], ls='--', color=GREY, lw=1.1)],
           ['Main analysis', 'Sensitivity analyses', 'H1 threshold, 0.5 SD (A)', 'No difference (B–D)', 'No change (E)'], loc='center',
           fontsize=10.5)
panel_letters(fig, list(zip(axes, 'ABCDE')))
save(fig, 'FigureS5_sensitivity_estimates',
     f"Figure S5. Main estimates across the sensitivity analyses. (A) H1 effect: the "
     f"hyperinflammatory class's mean inflammation score above the cohort mean (SD units). (B–D) H2 contrasts, hyperinflammatory versus "
     f"least abnormal class (BCH estimates; 95% bootstrap CIs, {N_BOOT_SENS} resamples in sensitivity analyses; arrows, CI not bounded; "
     f"no CI for the full-covariance model; the consensus k-means analysis compares its clusters directly). (E) Change in C statistic "
     f"when the subphenotype is added to age and consciousness (H4). Analyses with fewer classes compare their own extreme classes.")

In [ ]:
# Figure S6. Simple classifier (exploratory): ROC curves on the 2023-2026 children.
if CLF:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5.0))
    for ax, (mdl_, cols__, nm_) in zip(axes, [(clf, list(Xtr.columns), 'L1-penalized model'), (refit, used, 'Refitted model')]):
        prob_ = mdl_.predict_proba(Xte[cols__])
        for c in range(K_STAR):
            if c in mdl_.classes_ and 0 < (yte == c).sum() < len(yte):
                fpr, tpr, _ = roc_curve((yte == c).astype(int), prob_[:, list(mdl_.classes_).index(c)])
                ax.plot(fpr, tpr, color=CC[c], lw=2, label=CLS[c])
        ax.plot([0, 1], [0, 1], color=GREY, ls='--', lw=1.1, label='Chance')
        ax.set_aspect('equal'); ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel('1 − Specificity'); ax.set_ylabel('Sensitivity'); ax.legend(loc='lower right', fontsize=10)
    fig.tight_layout(w_pad=3)
    panel_letters(fig, list(zip(axes, 'AB')))
    _l1, _rf = CLF['l1_model'], CLF['refitted_unpenalized']
    save(fig, 'FigureS6_classifier',
         f"Figure S6. Simple classifier for the subphenotypes (exploratory). One-versus-rest receiver operating characteristic curves for each subphenotype in the "
         f"{CLF['n_test']} children admitted in 2023–2026, for a multinomial model trained on the main-model labels of the "
         f"{CLF['n_train']} children admitted in 2019–2022 with at most 5 indicators ({', '.join(CLF['indicators'])}). (A) L1-penalized "
         f"model (accuracy {_l1['accuracy']:.2f}; AUC " + ', '.join(f"{k} {v:.2f}" for k, v in _l1['auc_by_class'].items()) + "). (B) "
         f"Unpenalized model refitted on the same indicators (accuracy {_rf['accuracy']:.2f}; AUC " +
         ', '.join(f"{k} {v:.2f}" for k, v in _rf['auc_by_class'].items()) + ").")

In [ ]:
# Figure legends (every number above comes from this run).
(FIG / 'figure_legends.md').write_text('# Figure legends\n\n' + '\n\n'.join(LEGENDS.values()) + '\n', encoding='utf-8')
log('11 Outputs', 'Figures written', sorted(p.name for p in FIG.glob('*.pdf')))

In [ ]:
# Submission files, from the results above (aggregate results only): Additional file 2, every table of this run on its own sheet
# with a contents sheet; Additional file 3, the supplementary Figures S1 to S6, one per page with its title and legend.
import textwrap
from matplotlib.backends.backend_pdf import PdfPages
SUB = OUT / 'submission'
SUB.mkdir(exist_ok=True)
wait_until_writable(SUB.glob('*'))
TABLE_TITLES = {
    'table1_by_class': 'Characteristics of the clustered children by subphenotype (most likely class)',
    'availability': 'Availability of the registered indicators and the 60% rule',
    'correlated_pairs': 'Registered correlated indicator pairs',
    'fit_statistics': 'Fit and classification statistics for 1 to 6 classes',
    'class_summary': 'Class sizes, posterior probabilities and inflammation and cell-injury score',
    'profiles_model_means': 'Model-estimated indicator means by subphenotype (SD units)',
    'profiles_observed_means': 'Observed indicator means by subphenotype (SD units)',
    'indicators_original_units': 'Indicators in original units by subphenotype',
    'validators_by_class': 'Validators and outcomes by subphenotype (BCH estimates with 95% bootstrap CIs)',
    'validator_contrasts': 'Hyperinflammatory versus least abnormal subphenotype: all validator contrasts',
    'h2_primary': 'Registered hypothesis H2: primary contrasts with Holm-adjusted P values',
    'ifn_gamma_by_class': 'Interferon-gamma by subphenotype',
    'ites_class_crosswalk': 'ITES class by subphenotype',
    'treatments_by_class': 'Treatments by subphenotype',
    'h4': 'Registered hypothesis H4: subphenotype added to age and consciousness',
    'stability': 'Bootstrap stability: classes chosen and adjusted Rand index in each refit',
    'sensitivity_summary': 'Sensitivity analyses: summary of H1 to H4',
    'sensitivity_class_agreement': 'Sensitivity analyses: agreement of each class with the main solution',
    'chart_review_agreement': 'Agreement of the extracted neurological variables with clinician chart review',
    'classifier': 'Simple classifier (exploratory)',
    'exploratory_E1_ites_events': 'E1 (not registered): death or critical discharge by subphenotype within ITES classes',
    'exploratory_E1_hyper_vs_other': 'E1 (not registered): hyperinflammatory versus other subphenotypes within ITES categories',
    'exploratory_E1_beyond_ites': 'E1 (not registered): subphenotype added to age, consciousness and ITES category',
    'exploratory_E2_treatment_timing': 'E2 (not registered): immunomodulatory treatment and its timing by subphenotype',
    'exploratory_E2_landmark': 'E2 (not registered): outcomes by methylprednisolone within 24 h among children in hospital at 24 h',
    'main_summary': 'Main results at a glance'}
_order = [k for k in TABLE_TITLES if k in TABLES] + [k for k in TABLES if k not in TABLE_TITLES]
_contents = pd.DataFrame({'Sheet': [f'Table S{i + 1}' for i in range(len(_order))], 'Title': [TABLE_TITLES.get(k, k) for k in _order],
                          'Same table in output_subphenotypes/tables': [f'{k}.csv' for k in _order]})
with pd.ExcelWriter(SUB / 'Additional_file_2_supplementary_tables.xlsx', engine='openpyxl') as xw:
    pd.DataFrame({'Additional file 2. Supplementary tables': ['Every table of the analysis, one per sheet. BCH, bias-corrected three-step '
                                                              'estimation; CI, confidence interval; ITES, infection-triggered '
                                                              'encephalopathy syndrome; SD, standard deviation.']}).to_excel(
        xw, sheet_name='Contents', index=False)
    _contents.to_excel(xw, sheet_name='Contents', index=False, startrow=3)
    for i, k in enumerate(_order):
        pd.DataFrame([[f'Table S{i + 1}. {TABLE_TITLES.get(k, k)}']]).to_excel(xw, sheet_name=f'Table S{i + 1}', index=False, header=False)
        TABLES[k].to_excel(xw, sheet_name=f'Table S{i + 1}', index=False, startrow=2)

PG_W, PG_H, MRG = 8.27, 11.69, 0.75                                       # A4 portrait, inches
_sup = [k for k in LEGENDS if k.startswith('FigureS')]
with PdfPages(SUB / 'Additional_file_3_supplementary_figures.pdf') as _pdf:
    for _name in _sup:
        _m = re.match(r'(Figure S\d+)\. (.*?\.)\s+(.*)$', LEGENDS[_name], re.S)
        _head = textwrap.wrap(f'{_m.group(1)}. {_m.group(2)}', 95)
        _body = textwrap.wrap(_m.group(3), 112)
        _img = plt.imread(FIG / f'{_name}.png')
        _page = plt.figure(figsize=(PG_W, PG_H))
        _y = PG_H - MRG
        _page.text(MRG / PG_W, _y / PG_H, 'Additional file 3', fontsize=9, color='#555555', va='top')
        _y -= 0.35
        _page.text(MRG / PG_W, _y / PG_H, '\n'.join(_head), fontsize=10.5, fontweight='bold', va='top', linespacing=1.3)
        _y -= 0.2 * len(_head) + 0.15
        _iw = PG_W - 2 * MRG
        _ih = _iw * _img.shape[0] / _img.shape[1]
        _room = _y - MRG - 0.16 * len(_body) - 0.25
        if _ih > _room:
            _ih = _room; _iw = _ih * _img.shape[1] / _img.shape[0]
        _ax = _page.add_axes([(PG_W - _iw) / 2 / PG_W, (_y - _ih) / PG_H, _iw / PG_W, _ih / PG_H])
        _ax.imshow(_img, interpolation='lanczos'); _ax.axis('off')
        _y -= _ih + 0.2
        _page.text(MRG / PG_W, _y / PG_H, '\n'.join(_body), fontsize=9, va='top', linespacing=1.3)
        _pdf.savefig(_page); plt.close(_page)
log('11 Outputs', 'Submission files written (folder submission)',
    ['Additional_file_2_supplementary_tables.xlsx', 'Additional_file_3_supplementary_figures.pdf'])

## Run log and identifier check

In [ ]:
# Run log, then the identifier check over every output file.
lines = [f'# Run log: subphenotype analysis of AE_freeze_v1.4', '', f"Run finished {time.strftime('%Y-%m-%d %H:%M:%S')} in {round((time.time() - T0) / 60, 1)} min.", '',
         '## Software', '', json.dumps(SOFTWARE), '', '## Results and checks', '']
lines += [f'* [{s}] {i}: {v}' if v != '' else f'* [{s}] {i}' for s, i, v in RUNLOG]
lines += ['', '## Operational details not fixed by the registration', '',
          'All were fixed before the reported run. A few were changed after the development run (a run of this notebook with reduced '
          'settings, used to test the code); those say so.', ''] + [f'* {d}' for d in DETAILS]
lines += ['', '## Changes after the development run and the independent check', '',
          'The notebook was first run with reduced settings to test the code (development run), and an independent re-implementation, '
          'written by a separate instance of a large language model without access to this code, then rebuilt the data layer and the '
          'main results from the frozen dataset. These changes followed; all were made before the reported run.', ''] + ([f'* {d}' for d in CHANGES] or ['* None.'])
lines += ['', '## Deviations from the registration', ''] + ([f'* {d}' for d in DEVIATIONS] or ['* None.'])
lines += ['', '## Exploratory analyses not in the registration', '',
          'Added after the reported run of 2026-10-14, when the registered analysis above was complete; they change no registered result.',
          ''] + ([f'* {d}' for d in ADDITIONS] or ['* None.'])
wait_until_writable([LOGD / 'run_log.md'])
(LOGD / 'run_log.md').write_text('\n'.join(lines), encoding='utf-8')

ids = set(D.index.astype(str))                                  # research admission numbers (6 digits or more)
dates = set(D.admit_time.dt.strftime('%Y-%m-%d'))
hits = []
for f in list(TAB.glob('*.csv')) + list(LOGD.glob('*.md')) + list(LOGD.glob('*.log')) + list(FIG.glob('*.md')):
    txt = f.read_text(encoding='utf-8-sig')
    found = [t for t in set(re.findall(r'\b\d{6,}\b', txt)) if t in ids]
    found_dates = [d for d in dates if d in txt]
    if found or found_dates:
        hits.append((f.name, found[:3], found_dates[:3]))
if hits:
    raise RuntimeError(f'Research numbers or admission dates found in outputs: {hits}')
log('11 Outputs', 'Identifier check', 'no research numbers or admission dates in any output table or log')
with open(LOGD / 'run_log.md', 'a', encoding='utf-8') as fh:
    fh.write('\n\n## Identifier check\n\n* No research numbers or admission dates in any output table or in this log.\n')
print(f'Done in {round((time.time() - T0) / 60, 1)} min.')